# Qwen3.8-Flash-Next — NVFP4 MTP + ReplaySSM capacity probe

Independent L4x4 deployment probe.  It keeps the submission-critical radix
cache and NVFP4 KV path, but enables NEXTN MTP plus ReplaySSM speculative
verification to measure their real KV and Mamba-state cost.

| item | setting |
|---|---|
| topology | TP4 / EP4 / attention-DP2 (attention TP2 per group) |
| KV | NVFP4, page 64, patched Triton quantize/prefill/decode on sm_89 |
| multi-turn | radix cache on, extra_buffer_lazy, sticky routed_dp_rank |
| speculative decode | NEXTN MTP: 3 steps, top-k 1, 4 draft tokens |
| ReplaySSM | speculative verifier on; decode ReplaySSM remains off for radix |
| collectives | NCCL auto P2P; SGLang custom all-reduce off |

The run fails on a missing feature or broken smoke test, but does not impose the
submission profile's 500k KV gate: its purpose is to report the capacity that
actually survives MTP + ReplaySSM.

In [ ]:
"""All runtime and harness knobs live in this cell."""

import json, os, shutil, subprocess, sys, time
from dataclasses import dataclass, replace
from pathlib import Path

NB_T0 = time.time()


@dataclass(frozen=True)
class Config:
    tp_size: int = 4
    ep_size: int = 4
    dp_size: int = 2

    mem_fraction: float = 0.94
    max_total_tokens: int | None = None
    max_running_global: int = 16
    live_per_worker: int = 8
    max_mamba_cache_global: int = 80
    cuda_graph_max_bs: int = 8
    context_length: int = 65_536
    chunked_prefill: int = 1_024
    page_size: int = 64

    # Admission-planning average, not the per-session context cap.  Long
    # sessions are dynamically rescheduled when aggregate residency grows.
    pool_context: int = 20000
    pool_safety: float = 0.80
    # The no-MTP NVFP4 control measured 708,288 tokens/DP with fewer Mamba
    # states.  Eight live requests require 32 states/DP, so keep a conservative
    # 500k gate while allowing the extra state reservation.
    min_pool_tokens_per_worker: int = 200000

    kv_cache_dtype: str = "nvfp4"
    mamba_ssm_dtype: str = "float16"
    attention_backend: str = "triton"
    enable_sglang_custom_all_reduce: bool = False

    # Decode ReplaySSM requires no_buffer/disable-radix-cache. ReplaySSM-spec
    # requires MTP; both are lower priority than radix and KV capacity.
    enable_mtp: bool = True
    enable_replayssm_spec: bool = True

    effort: str = "xhigh"
    explorers: int = 3
    ctx_cap: int = 50_000
    gen_budget: int = 75_000
    refine_budget: int = 40_000
    task_budget: int = 180_000
    lineage_budget: int = 90_000
    min_session_budget: int = 20_000
    handoff_budget: int = 12_000
    max_hops: int = 3
    max_turns: int = 200
    report_reserve: int = 10_000
    context_margin: int = 2_500
    deadline_tok_per_s: float = 30.0
    max_error_rate: float = 0.02

    temperature: float = 1.0
    top_p: float = 0.95
    top_k: int = -1

    limit_manual: int = 8
    limit_mode: str = "auto"
    total_budget_min: float = 705.0
    reserve_min: float = 25.0
    server_timeout_s: int = 3_600
    port: int = 30_020
    scratch: str = "/tmp/arcforge_qwen38fn_nvfp4"

    def __post_init__(self):
        assert (self.tp_size, self.ep_size, self.dp_size) == (4, 4, 2)
        assert self.kv_cache_dtype == "nvfp4"
        assert self.page_size == 64
        assert self.enable_mtp or not self.enable_replayssm_spec, (
            "ReplaySSM spec requires the MTP draft worker"
        )
        assert self.live_per_worker * self.dp_size == self.max_running_global
        assert self.live_per_worker * self.pool_context <= (
            self.min_pool_tokens_per_worker * self.pool_safety
        )
        assert self.cuda_graph_max_bs >= self.live_per_worker
        states_per_request = 5 if self.enable_mtp else 4
        assert self.max_mamba_cache_global >= (
            states_per_request * self.live_per_worker * self.dp_size
        )
        assert self.ctx_cap + self.context_margin + self.report_reserve <= self.context_length
        assert self.top_k == -1
        assert self.limit_mode in ("auto", "full", "manual")

    @property
    def base_url(self):
        return f"http://127.0.0.1:{self.port}/v1"

    @property
    def scratch_path(self):
        return Path(self.scratch)

    @property
    def live_global(self):
        return self.live_per_worker * self.dp_size


CFG = Config()
print(CFG)
for directory in ("src/arcforge/tir", "work/agentic_tir", "patches"):
    (CFG.scratch_path / directory).mkdir(parents=True, exist_ok=True)
(CFG.scratch_path / "src/arcforge/__init__.py").write_text(
    '"""Minimal package shell for the embedded TIR harness."""\n'
)

In [ ]:
RUNTIME_ARCHIVE_NAME = 'sglang-qwen4-73a255206f-sm120-cp312.tar.zst'

MANIFEST = {
  "target_sizes": {
    "chat_template.jinja": 8952,
    "config.json": 35616,
    "generation_config.json": 202,
    "hf_quant_config.json": 23001,
    "merges.txt": 3353259,
    "model-00001.safetensors": 5377141872,
    "model-00002.safetensors": 5413215056,
    "model-00003.safetensors": 5372128428,
    "model-00004.safetensors": 5371926048,
    "model-00005.safetensors": 5371490752,
    "model-00006.safetensors": 5371475336,
    "model-00007.safetensors": 5371823408,
    "model-00008.safetensors": 5371490824,
    "model-00009.safetensors": 6619231876,
    "model-00010.safetensors": 5379001240,
    "model-00011.safetensors": 5600029208,
    "model-00012.safetensors": 5600029208,
    "model-00013.safetensors": 5600029192,
    "model-00014.safetensors": 5600029192,
    "model-00015.safetensors": 5600029192,
    "model-00016.safetensors": 5600029192,
    "model-00017.safetensors": 5600029192,
    "model-00018.safetensors": 5600029192,
    "model-00019.safetensors": 5370342010,
    "model-00020.safetensors": 1002971104,
    "model.safetensors.index.json": 18647187,
    "preprocessor_config.json": 390,
    "prune_manifest.json": 75656,
    "tokenizer.json": 12809320,
    "tokenizer_config.json": 17928,
    "video_preprocessor_config.json": 385,
    "vocab.json": 6722759
  },
  "target_sha256": {
    "chat_template.jinja": "c3cf9e34abf4f9e36c2d72165aa9c132d3e2a725b6c2586aaa3a8af9d7a81041",
    "config.json": "655d013b7d28371b4199f2ae309402ebf4c72701ac6ee6b3f4c054dafba4f8b7",
    "generation_config.json": "e70c136c1b78ddc1fb0905bac8e733a4dc448d4f852a5dd75143fffc70be550e",
    "hf_quant_config.json": "331ad11d57c8bc374554579977198084e0d4d0933d5b3558f125f6f670aba0e8",
    "merges.txt": "a9d356d7bdf1ef4949e3e748e95b8e10ad9d4e2e838eddc38a0a7b6b94d1db8d",
    "model.safetensors.index.json": "6cb945a7d4baabe1aa06a29177fd8540ccba50aab2831e520a911acd49d8068c",
    "preprocessor_config.json": "27225450ac9c6529872ee1924fcb0962ff5634834f817040f444118116f4e516",
    "prune_manifest.json": "7c8f6d484519fc6ddc034371517fe0cb5d84a868c3857ff6ee7679f007653e33",
    "tokenizer.json": "0997f410c57a1f4e53b09e4be8f4a172d90edd9564368fb0847030937229b9f3",
    "tokenizer_config.json": "b11349aafa7cdc6a320767cf7ceb29ed82f7eda5d65e8e0819e76f0ce947bf27",
    "video_preprocessor_config.json": "7768af27c1fafa9cc9011c1dc20067e03f8915e03b63504550e11d5066986d13",
    "vocab.json": "ce99b4cb2983d118806ce0a8b777a35b093e2000a503ebde25853284c9dfa003"
  },
  "runtime_name": "sglang-qwen4-73a255206f-sm120-cp312.tar.zst",
  "runtime_sha256": "e7c03ad7300669fab9d96f9d70080671b0df198f7ab40b99e0adff9162309481"
}

PATCH_ORDER = ['0036-qwen4-ple-loader-release-maps.patch', '0037-qsa-direct-decode-sm89-tile.patch', '0038-qsa-direct-decode-fp8-descales.patch', '0039-qsa-fp4-fused-decode-sm89.patch', '0040-nvfp4-triton-quantize-sm89.patch', '0041-ple-ngram-build-on-host-sm89.patch', '0042-fp4-dequant-workspace-budget-sm89.patch', '0043-nvfp4-cudagraph-scalar-sm89.patch', '0044-qwen4-mtp-draft-topology.patch']

PATCH_SHA256 = {
  "0036-qwen4-ple-loader-release-maps.patch": "765d884efbca2b5b1f505b13444e03d384a9a951554ef6a92e855e8812febcb3",
  "0037-qsa-direct-decode-sm89-tile.patch": "29956d1e51625b1564660e766482b3042a375481e18bb4397ada492de22ddc96",
  "0038-qsa-direct-decode-fp8-descales.patch": "e93d751b522f0723e63c0878aab7e56a1a055f50c1bd32bb3e7cb2c2766a97fb",
  "0039-qsa-fp4-fused-decode-sm89.patch": "15d85b0a7c4c2a820e5c9b600e0826a3c4108a96c5b058eee04599201c7a3e5b",
  "0040-nvfp4-triton-quantize-sm89.patch": "7e881c2f2511a52135001a401d403bf8ae1623e96f3149002fd93f156c7c57bc",
  "0041-ple-ngram-build-on-host-sm89.patch": "7f5dc8a7d2e58338f4b8577e5e2f354bca0ae5892f51a4d72f162d33a2621058",
  "0042-fp4-dequant-workspace-budget-sm89.patch": "ef4a4c66e12dbebcc798c08093aa1bfc3558438f25ca54f0c2b05ddfb2548df3",
  "0043-nvfp4-cudagraph-scalar-sm89.patch": "b0d3b960724230f1a782515dfb6d0c50d4ea680aba30e7e3aa3cc21e2cb7f009",
  "0044-qwen4-mtp-draft-topology.patch": "5f39cc1ff8ba49d5d247856d4745d417c4ba4c3a9e2c8956c5244318c4dcbd41"
}

PATCH_PAYLOADS = {'0036-qwen4-ple-loader-release-maps.patch': '--- a/python/sglang/srt/models/qwen4_exp.py\n+++ b/python/sglang/srt/models/qwen4_exp.py\n@@ -2397,6 +2397,10 @@\n             if isinstance(module, Qwen3_5GatedDeltaNet):\n                 module.finalize_fused_in_proj()\n \n+        # L4: release loader-only maps captured by the recursive local helper.\n+        params_dict.clear()\n+        buffers.clear()\n+        ple_modules.clear()\n         return loaded_params\n \n     @classmethod\n\n', '0037-qsa-direct-decode-sm89-tile.patch': '--- a/python/sglang/srt/layers/attention/qsa/sparse_attn.py\n+++ b/python/sglang/srt/layers/attention/qsa/sparse_attn.py\n@@ -587,6 +587,12 @@\n         block_n, warps, stages = 64, 8, 3\n     else:\n         block_n, warps, stages = 64, 4, 3\n+    # SM89 shared memory is 99 KiB. Measured for 12 q-heads x head_dim 256:\n+    # BLOCK_N=128/stages=3 needs 140.5 KiB with BF16 K/V (does not fit) but only\n+    # 76.5 KiB with FP8 or packed FP4 (fits). Shrinking unconditionally cost 2.1x\n+    # (67 -> 140 us) on exactly the config that ships, so gate on the KV width.\n+    if torch.cuda.get_device_capability(q.device) == (8, 9) and not kv_is_fp8:\n+        block_n, stages = 64, 2\n     out = torch.empty_like(q)\n     if rows == 0:\n         return out\n\n', '0038-qsa-direct-decode-fp8-descales.patch': '--- a/python/sglang/srt/layers/attention/qsa/sparse_attn.py\n+++ b/python/sglang/srt/layers/attention/qsa/sparse_attn.py\n@@ -473,6 +473,8 @@\n     seq_lens,\n     out,\n     scale,\n+    k_scale,\n+    v_scale,\n     topk: tl.constexpr,\n     sq_m: tl.constexpr,\n     sq_h: tl.constexpr,\n@@ -492,6 +494,7 @@\n     BLOCK_M: tl.constexpr,\n     BLOCK_N: tl.constexpr,\n     HEAD_DIM: tl.constexpr,\n+    KV_IS_FP8: tl.constexpr,\n ):\n     """Sparse GQA over selected logical tokens without compacting K/V first."""\n     row = tl.program_id(0)\n@@ -534,7 +537,10 @@\n             mask=valid[:, None],\n             other=0.0,\n         ).to(q_values.dtype)\n-        scores = tl.where(valid[None, :], tl.dot(q_values, keys), -float("inf"))\n+        scores = tl.dot(q_values, keys)\n+        if KV_IS_FP8:\n+            scores *= k_scale\n+        scores = tl.where(valid[None, :], scores, -float("inf"))\n         next_max = tl.maximum(max_value, tl.max(scores, 1))\n         alpha = tl.math.exp2(max_value - next_max)\n         probabilities = tl.math.exp2(scores - next_max[:, None])\n@@ -544,6 +550,8 @@\n         normalizer = normalizer * alpha + tl.sum(probabilities, 1)\n         max_value = next_max\n     output = accumulator / normalizer[:, None]\n+    if KV_IS_FP8:\n+        output *= v_scale\n     tl.store(\n         out\n         + row * so_m\n@@ -563,8 +571,12 @@\n     indices,\n     seq_lens,\n     scale,\n+    k_scale: Optional[float] = None,\n+    v_scale: Optional[float] = None,\n ):\n     """Decode sparse GQA straight from paged K/V selected by logical indices."""\n+    # L4_FP8_DIRECT_SCALES\n+    kv_is_fp8 = _validate_sparse_gqa_dtypes(q, k, v)\n     rows, q_heads, head_dim = q.shape\n     kv_heads = k.shape[1]\n     group_size = q_heads // kv_heads\n@@ -591,6 +603,8 @@\n         seq_lens,\n         out,\n         scale,\n+        _unit_scale(k_scale),\n+        _unit_scale(v_scale),\n         indices.shape[1],\n         q.stride(0),\n         q.stride(1),\n@@ -610,6 +624,7 @@\n         BLOCK_M=block_m,\n         BLOCK_N=block_n,\n         HEAD_DIM=head_dim,\n+        KV_IS_FP8=kv_is_fp8,\n         num_warps=warps,\n         num_stages=stages,\n     )\n--- a/python/sglang/srt/layers/attention/qwen_sparse_attn_backend.py\n+++ b/python/sglang/srt/layers/attention/qwen_sparse_attn_backend.py\n@@ -1704,6 +1704,8 @@\n                 if metadata.row_req_pool_indices is not None\n                 else forward_batch.req_pool_indices\n             )\n+            # L4_FP8_DIRECT_SCALES\n+            k_scale, v_scale = self._kv_descales(layer, k_buffer.dtype)\n             output = qwen_sparse_gqa_decode_selected_triton(\n                 q.contiguous(),\n                 k_buffer,\n@@ -1713,6 +1715,8 @@\n                 topk_indices,\n                 sequence_lens,\n                 layer.scaling,\n+                k_scale=k_scale,\n+                v_scale=v_scale,\n             )\n             return output.reshape(q.shape[0], -1)\n \n\n', '0039-qsa-fp4-fused-decode-sm89.patch': 'diff --git a/python/sglang/srt/layers/attention/qsa/sparse_attn.py b/python/sglang/srt/layers/attention/qsa/sparse_attn.py\nindex 1819acd..6b83d1c 100644\n--- a/python/sglang/srt/layers/attention/qsa/sparse_attn.py\n+++ b/python/sglang/srt/layers/attention/qsa/sparse_attn.py\n@@ -798,6 +801,197 @@ __all__ = [\n     "qwen_sparse_valid_counts_triton",\n     "qwen_sparse_kv_extraction_compact_triton",\n     "qwen_sparse_gqa_decode_selected_triton",\n+    "qwen_sparse_gqa_decode_selected_fp4_triton",\n+    "qwen_sparse_nvfp4_expand_triton",\n     "sparse_gqa_fwd_interface_triton",\n     "sparse_gqa_fwd_interface_triton_ck",\n ]\n+\n+\n+@triton.jit\n+def _qsa_fp4_decode_nibble(byte, shift):\n+    """E2M1: bit3 sign, bits2-1 exponent, bit0 mantissa.\n+\n+    2^(exp-1) is an integer shift for exp in {1,2,3}; exp==0 is the subnormal\n+    pair (0, 0.5). Using exp2() here costs ~30% of the kernel because it runs on\n+    every one of the BLOCK_N*PACK*2 nibbles per tile.\n+    """\n+    nib = (byte >> (4 * shift)) & 0x0F\n+    exp = (nib >> 1) & 0x03\n+    man = (nib & 0x01).to(tl.float32)\n+    pow2 = (1 << tl.maximum(exp - 1, 0)).to(tl.float32)\n+    mag = tl.where(exp == 0, man * 0.5, pow2 * (1.0 + man * 0.5))\n+    return tl.where(((nib >> 3) & 0x01) == 1, -mag, mag)\n+\n+\n+@triton.jit\n+def _sparse_gqa_decode_selected_fp4(\n+    q_even, q_odd, k, v, k_sf, v_sf, req_to_token, req_indices, indices, seq_lens, out,\n+    topk: tl.constexpr, req_stride: tl.constexpr,\n+    GROUP_SIZE: tl.constexpr, BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr,\n+    Q_HEADS: tl.constexpr, HEADS: tl.constexpr, PACK: tl.constexpr, SF: tl.constexpr,\n+):\n+    row, kv_head = tl.program_id(0), tl.program_id(1)\n+    length = tl.load(seq_lens + row)\n+    req = tl.load(req_indices + row)\n+    offs_h = tl.arange(0, BLOCK_M)\n+    offs_p = tl.arange(0, PACK)\n+    hmask = (offs_h < GROUP_SIZE)[:, None]\n+    head_start = kv_head * GROUP_SIZE\n+    qe = tl.load(q_even + row * (Q_HEADS * PACK) + (head_start + offs_h[:, None]) * PACK\n+                 + offs_p[None, :], mask=hmask, other=0.0)\n+    qo = tl.load(q_odd + row * (Q_HEADS * PACK) + (head_start + offs_h[:, None]) * PACK\n+                 + offs_p[None, :], mask=hmask, other=0.0)\n+\n+    max_value = tl.full([BLOCK_M], -float("inf"), tl.float32)\n+    normalizer = tl.zeros([BLOCK_M], tl.float32)\n+    acc_even = tl.zeros([BLOCK_M, PACK], tl.float32)\n+    acc_odd = tl.zeros([BLOCK_M, PACK], tl.float32)\n+    offs_n = tl.arange(0, BLOCK_N)\n+    offs_s = tl.arange(0, SF)                  # 1 scale per 16 features == per 8 bytes\n+\n+    for start in range(0, topk, BLOCK_N):\n+        cur = start + offs_n\n+        pos = tl.load(indices + row * topk + cur, mask=cur < topk, other=-1)\n+        valid = (cur < topk) & (pos >= 0) & (pos < length)\n+        tok = tl.load(req_to_token + req * req_stride + tl.where(valid, pos, 0),\n+                      mask=valid, other=0).to(tl.int64)\n+\n+        # K as [PACK, BLOCK_N] so it feeds tl.dot(q[BLOCK_M, PACK], k[PACK, BLOCK_N]).\n+        k_src = tok[None, :] * (HEADS * PACK) + kv_head * PACK + offs_p[:, None]\n+        k_ss = tok[None, :] * (HEADS * SF) + kv_head * SF + offs_s[:, None]\n+        kb = tl.load(k + k_src, mask=valid[None, :], other=0)\n+        kstep = tl.load(k_sf + k_ss, mask=valid[None, :], other=0.0).to(tl.float32)\n+        kstep = tl.reshape(tl.broadcast_to(tl.reshape(kstep, (SF, 1, BLOCK_N)),\n+                                           (SF, 8, BLOCK_N)), (PACK, BLOCK_N))\n+        k_low = (_qsa_fp4_decode_nibble(kb, 0) * kstep).to(qe.dtype)\n+        k_high = (_qsa_fp4_decode_nibble(kb, 1) * kstep).to(qe.dtype)\n+\n+        scores = tl.dot(qe, k_low) + tl.dot(qo, k_high)\n+        scores = tl.where(valid[None, :], scores, -float("inf"))\n+        next_max = tl.maximum(max_value, tl.max(scores, 1))\n+        alpha = tl.math.exp2(max_value - next_max)\n+        probs = tl.math.exp2(scores - next_max[:, None])\n+\n+        v_src = tok[:, None] * (HEADS * PACK) + kv_head * PACK + offs_p[None, :]\n+        v_ss = tok[:, None] * (HEADS * SF) + kv_head * SF + offs_s[None, :]\n+        vb = tl.load(v + v_src, mask=valid[:, None], other=0)\n+        vstep = tl.load(v_sf + v_ss, mask=valid[:, None], other=0.0).to(tl.float32)\n+        vstep = tl.reshape(tl.broadcast_to(tl.reshape(vstep, (BLOCK_N, SF, 1)),\n+                                           (BLOCK_N, SF, 8)), (BLOCK_N, PACK))\n+        p = probs.to(qe.dtype)\n+        acc_even = tl.dot(p, (_qsa_fp4_decode_nibble(vb, 0) * vstep).to(qe.dtype),\n+                          acc_even * alpha[:, None])\n+        acc_odd = tl.dot(p, (_qsa_fp4_decode_nibble(vb, 1) * vstep).to(qe.dtype),\n+                         acc_odd * alpha[:, None])\n+        normalizer = normalizer * alpha + tl.sum(probs, 1)\n+        max_value = next_max\n+\n+    acc_even = acc_even / normalizer[:, None]\n+    acc_odd = acc_odd / normalizer[:, None]\n+    base = out + row * (Q_HEADS * 2 * PACK) + (head_start + offs_h[:, None]) * (2 * PACK)\n+    tl.store(base + 2 * offs_p[None, :], acc_even, mask=hmask)\n+    tl.store(base + 2 * offs_p[None, :] + 1, acc_odd, mask=hmask)\n+\n+\n+def qwen_sparse_gqa_decode_selected_fp4_triton(\n+    q, k, v, k_sf, v_sf, req_to_token, req_indices, indices, seq_lens, scale,\n+    k_global=1.0, v_global=1.0, block_n=None, stages=None, warps=None,\n+):\n+    """Sparse GQA decode straight off the packed FP4 pool.\n+\n+    The sm_100/sm_120 path hands these planes to trtllm-gen/XQA. Ada has no such\n+    kernel, and the pool\'s fallback dequantizes the entire layer buffer on every\n+    read; this does the unpack inline instead. Tile config is the measured\n+    optimum on sm_89 (BLOCK_N=128, 2 stages, 16 warps). A 16-entry LUT gather, a\n+    bf16 decode chain and a tl.join-merged V dot were each tried and each came out\n+    slower, so the arithmetic decode stays.\n+    """\n+    rows, q_heads, dim = q.shape\n+    kv_heads, pack = k.shape[1], k.shape[2]\n+    if pack * 2 != dim:\n+        raise ValueError(f"packed width {pack} does not match head_dim {dim}")\n+    group = q_heads // kv_heads\n+    scaled = (q.float() * scale * 1.4426950408 * k_global).to(torch.bfloat16)\n+    q_even = scaled[..., 0::2].contiguous()\n+    q_odd = scaled[..., 1::2].contiguous()\n+    out = torch.empty(rows, q_heads, dim, device=q.device, dtype=torch.float32)\n+    if rows == 0:\n+        return out.to(q.dtype)\n+    _sparse_gqa_decode_selected_fp4[(rows, kv_heads)](\n+        q_even, q_odd, k, v, k_sf, v_sf, req_to_token, req_indices, indices,\n+        seq_lens, out,\n+        topk=indices.shape[1], req_stride=req_to_token.stride(0),\n+        GROUP_SIZE=group, BLOCK_M=max(16, triton.next_power_of_2(group)),\n+        BLOCK_N=128 if block_n is None else block_n,\n+        Q_HEADS=q_heads, HEADS=kv_heads, PACK=pack, SF=dim // 16,\n+        num_warps=16 if warps is None else warps,\n+        num_stages=2 if stages is None else stages,\n+    )\n+    return (out * v_global).to(q.dtype)\n+\n+\n+@triton.jit\n+def _qsa_nvfp4_expand_kv(\n+    k_fp4, v_fp4, k_sf, v_sf, req_to_token, req_indices, cu_k, out_k, out_v,\n+    k_global, v_global,\n+    heads: tl.constexpr, dim: tl.constexpr, req_stride: tl.constexpr,\n+    BLOCK_T: tl.constexpr,\n+):\n+    """Gather one request\'s packed FP4 context and write it out dequantized.\n+\n+    Replaces index_select -> dequantize -> torch.cat: the packed bytes and their\n+    block scales are read once and the result lands directly at the request\'s\n+    cu_k offset, so no packed-to-dense copy of the whole layer buffer exists.\n+    """\n+    PACK: tl.constexpr = dim // 2\n+    SF: tl.constexpr = dim // 16\n+    pid_b, pid_h, pid_t = tl.program_id(0), tl.program_id(1), tl.program_id(2)\n+    start = tl.load(cu_k + pid_b).to(tl.int64)\n+    length = tl.load(cu_k + pid_b + 1).to(tl.int64) - start\n+    tokens = pid_t * BLOCK_T + tl.arange(0, BLOCK_T)\n+    live = tokens < length\n+    req = tl.load(req_indices + pid_b).to(tl.int64)\n+    slots = tl.load(req_to_token + req * req_stride + tokens, mask=live, other=0).to(tl.int64)\n+\n+    pack = tl.arange(0, PACK)\n+    packed_src = slots[:, None] * (heads * PACK) + pid_h * PACK + pack[None, :]\n+    scale_src = slots[:, None] * (heads * SF) + pid_h * SF + (pack // 8)[None, :]\n+    mask = live[:, None]\n+    kb = tl.load(k_fp4 + packed_src, mask=mask, other=0)\n+    vb = tl.load(v_fp4 + packed_src, mask=mask, other=0)\n+    k_step = tl.load(k_sf + scale_src, mask=mask, other=0.0).to(tl.float32) * tl.load(k_global)\n+    v_step = tl.load(v_sf + scale_src, mask=mask, other=0.0).to(tl.float32) * tl.load(v_global)\n+    dst = (start + tokens)[:, None] * (heads * dim) + pid_h * dim + 2 * pack[None, :]\n+\n+    tl.store(out_k + dst, (_qsa_fp4_decode_nibble(kb, 0) * k_step).to(tl.bfloat16), mask=mask)\n+    tl.store(out_k + dst + 1, (_qsa_fp4_decode_nibble(kb, 1) * k_step).to(tl.bfloat16), mask=mask)\n+    tl.store(out_v + dst, (_qsa_fp4_decode_nibble(vb, 0) * v_step).to(tl.bfloat16), mask=mask)\n+    tl.store(out_v + dst + 1, (_qsa_fp4_decode_nibble(vb, 1) * v_step).to(tl.bfloat16), mask=mask)\n+\n+\n+def qwen_sparse_nvfp4_expand_triton(\n+    k_fp4, v_fp4, k_sf, v_sf, req_to_token, req_indices, cu_k, total, k_global, v_global,\n+):\n+    """Packed FP4 pool -> contiguous BF16 context, laid out as the ck kernel wants."""\n+    heads, pack = k_fp4.shape[1], k_fp4.shape[2]\n+    dim = pack * 2\n+    dev = k_fp4.device\n+    out_k = torch.empty(total, heads, dim, dtype=torch.bfloat16, device=dev)\n+    out_v = torch.empty(total, heads, dim, dtype=torch.bfloat16, device=dev)\n+    batch = cu_k.numel() - 1\n+    if total == 0:\n+        return out_k, out_v\n+    lengths = (cu_k[1:] - cu_k[:-1])\n+    max_len = int(lengths.max().item()) if batch else 0\n+    if max_len == 0:\n+        return out_k, out_v\n+    kg = torch.as_tensor([float(k_global)], dtype=torch.float32, device=dev)\n+    vg = torch.as_tensor([float(v_global)], dtype=torch.float32, device=dev)\n+    block_t = 8\n+    _qsa_nvfp4_expand_kv[(batch, heads, triton.cdiv(max_len, block_t))](\n+        k_fp4, v_fp4, k_sf.view(torch.float8_e4m3fn), v_sf.view(torch.float8_e4m3fn),\n+        req_to_token, req_indices.to(torch.int32), cu_k.to(torch.int32), out_k, out_v, kg, vg,\n+        heads=heads, dim=dim, req_stride=req_to_token.stride(0), BLOCK_T=block_t,\n+        num_warps=4)\n+    return out_k, out_v\ndiff --git a/python/sglang/srt/layers/attention/qwen_sparse_attn_backend.py b/python/sglang/srt/layers/attention/qwen_sparse_attn_backend.py\nindex dd18cea..3f105ad 100644\n--- a/python/sglang/srt/layers/attention/qwen_sparse_attn_backend.py\n+++ b/python/sglang/srt/layers/attention/qwen_sparse_attn_backend.py\n@@ -7,6 +7,7 @@ linear-attention layers continue to use GDN.\n from __future__ import annotations\n \n import logging\n+import os\n import math\n from copy import copy\n from functools import lru_cache\n@@ -34,6 +35,8 @@ from sglang.srt.layers.attention.qsa.metadata import (\n from sglang.srt.layers.attention.qsa.sparse_attn import (\n     is_fp8_kv_dtype,\n     qwen_sparse_gqa_decode_selected_triton,\n+    qwen_sparse_gqa_decode_selected_fp4_triton,\n+    qwen_sparse_nvfp4_expand_triton,\n     qwen_sparse_fa2_cu_seqlens_triton,\n     qwen_sparse_kv_extraction_compact_triton,\n     qwen_sparse_valid_counts_triton,\n@@ -47,6 +50,8 @@ logger = logging.getLogger(__name__)\n \n _TRTLLM_SPARSE_PAGE_SIZE = 64\n _QSA_DIRECT_DECODE = envs.SGLANG_QSA_DIRECT_DECODE.get()\n+# ARC: kill switch for the fused packed-FP4 QSA decode (sm_89).\n+_ARC_QSA_FP4_FUSED = os.environ.get("ARC_QSA_FP4_FUSED", "1") == "1"\n _QSA_DIRECT_MIN_ROWS = envs.SGLANG_QSA_DIRECT_MIN_ROWS.get()\n \n \n@@ -189,6 +194,22 @@ class QSAMTPSharedSparseIndices:\n         return out\n \n \n+def _arc_fp4_pool(pool) -> bool:\n+    return getattr(getattr(pool, "full_kv_pool", pool), "k_scale_buffer", None) is not None\n+\n+\n+def _arc_fp4_global_scales(pool, layer_id: int):\n+    """NVFP4 carries a per-tensor scale per layer on top of the block scales;\n+    fp4_mx_block16 does not, so it falls through to the (1.0, 1.0) identity."""\n+    inner = getattr(pool, "full_kv_pool", pool)\n+    method = getattr(inner, "quant_method", None)\n+    getter = getattr(method, "get_bmm_scales", None)\n+    if getter is None:\n+        return 1.0, 1.0\n+    k, v = getter(layer_id)\n+    return float(k), float(v)\n+\n+\n class QwenSparseAttnBackend(AttentionBackend):\n     """QSA backend using trtllm-gen decode with a packed FA2/FA4 fallback."""\n \n@@ -1475,6 +1496,23 @@ class QwenSparseAttnBackend(AttentionBackend):\n         # The validated chunk-prefill kernel consumes tightly packed full-context\n         # K/V. Current-chunk K/V has already been committed to the cache above.\n         pool = self.token_to_kv_pool\n+        sequence_lens_t = torch.tensor(sequence_lens, dtype=torch.int32, device=q.device)\n+        cu_k = F.pad(sequence_lens_t.cumsum(0), (1, 0)).contiguous()\n+        if _ARC_QSA_FP4_FUSED and _arc_fp4_pool(pool):\n+            # index_select has no Float4_e2m1fn_x2 kernel, and dequantizing the\n+            # whole layer buffer to work around that is what this path exists to\n+            # avoid. Gather + dequantize the selected context in one pass.\n+            k_fp4, v_fp4, k_sf, v_sf = pool.get_raw_kv_buffer(layer.layer_id)\n+            kg, vg = _arc_fp4_global_scales(pool, layer.layer_id)\n+            k_ctx, v_ctx = qwen_sparse_nvfp4_expand_triton(\n+                k_fp4, v_fp4, k_sf, v_sf, self.req_to_token_pool.req_to_token,\n+                forward_batch.req_pool_indices, cu_k, int(sum(sequence_lens)), kg, vg,\n+            )\n+            output = sparse_gqa_fwd_interface_triton_ck(\n+                q.contiguous(), k_ctx, v_ctx, topk_indices, cu_seqlens_q, cu_k,\n+                sequence_lens_t, layer.scaling,\n+            )\n+            return self._pad_extend_output(output, num_output_rows)\n         k_buffer = pool.get_key_buffer(layer.layer_id)\n         v_buffer = pool.get_value_buffer(layer.layer_id)\n         scale_kwargs = {}\n@@ -1676,6 +1714,38 @@ class QwenSparseAttnBackend(AttentionBackend):\n         topk_indices: torch.Tensor,\n     ) -> torch.Tensor:\n         pool = self.token_to_kv_pool\n+        # ARC sm_89: take the packed FP4 plane before get_key_buffer(), which on a\n+        # quantized pool dequantizes the whole layer buffer on every call.\n+        if (\n+            _ARC_QSA_FP4_FUSED\n+            and _QSA_DIRECT_DECODE\n+            and q.is_cuda\n+            and q.shape[0] >= _QSA_DIRECT_MIN_ROWS\n+            # HybridLinearKVPool delegates the raw plane but does not carry the\n+            # scale buffers itself, so look through to the wrapped full pool.\n+            and _arc_fp4_pool(pool)\n+        ):\n+            metadata = self._resolve_metadata(forward_batch)\n+            k_fp4, v_fp4, k_sf, v_sf = pool.get_raw_kv_buffer(layer.layer_id)\n+            req_indices = (\n+                metadata.row_req_pool_indices\n+                if metadata.row_req_pool_indices is not None\n+                else forward_batch.req_pool_indices\n+            )\n+            output = qwen_sparse_gqa_decode_selected_fp4_triton(\n+                q.contiguous(),\n+                k_fp4,\n+                v_fp4,\n+                k_sf,\n+                v_sf,\n+                self.req_to_token_pool.req_to_token,\n+                req_indices,\n+                topk_indices.to(torch.int32).contiguous(),\n+                metadata.sequence_lengths,\n+                layer.scaling,\n+                *_arc_fp4_global_scales(pool, layer.layer_id),\n+            )\n+            return output.reshape(q.shape[0], -1)\n         k_buffer = pool.get_key_buffer(layer.layer_id)\n         v_buffer = pool.get_value_buffer(layer.layer_id)\n         if not q.is_cuda:\n\n', '0040-nvfp4-triton-quantize-sm89.patch': 'diff --git a/python/sglang/srt/layers/quantization/kvfp4_tensor.py b/python/sglang/srt/layers/quantization/kvfp4_tensor.py\nindex 553cb5f..0f550d0 100644\n--- a/python/sglang/srt/layers/quantization/kvfp4_tensor.py\n+++ b/python/sglang/srt/layers/quantization/kvfp4_tensor.py\n@@ -16,6 +16,8 @@\n from enum import Enum\n \n import torch\n+import triton\n+import triton.language as tl\n \n \n class FP4KVCacheRecipe(Enum):\n@@ -147,6 +149,56 @@ class FP4MXBlock16KVQuantizeUtil:\n         return scaled.view(b, m, n).to(dtype)\n \n \n+@triton.jit\n+def _nvfp4_quantize(x, gscale, out, sf, rows, N: tl.constexpr, BLOCK_R: tl.constexpr):\n+    """One program per (row block, 16-element scale block)."""\n+    pid_r, pid_b = tl.program_id(0), tl.program_id(1)\n+    SF: tl.constexpr = N // 16\n+    r = pid_r * BLOCK_R + tl.arange(0, BLOCK_R)\n+    live = r < rows\n+    cols = pid_b * 16 + tl.arange(0, 16)\n+    vals = tl.load(x + r[:, None] * N + cols[None, :], mask=live[:, None], other=0.0).to(tl.float32)\n+\n+    g = tl.load(gscale)\n+    amax = tl.max(tl.abs(vals), 1)\n+    bs = amax / (6.0 * g)\n+    bs = tl.minimum(tl.maximum(bs, 1e-8), 448.0).to(tl.float8e4nv)     # storable scale\n+    step = tl.maximum(bs.to(tl.float32) * g, 1e-12)\n+    q = vals / step[:, None]\n+\n+    # Nearest E2M1 level: thresholds are the midpoints of (0,.5,1,1.5,2,3,4,6).\n+    a = tl.minimum(tl.abs(q), 6.0)\n+    idx = ((a > 0.25).to(tl.int32) + (a > 0.75).to(tl.int32) + (a > 1.25).to(tl.int32)\n+           + (a > 1.75).to(tl.int32) + (a > 2.5).to(tl.int32) + (a > 3.5).to(tl.int32)\n+           + (a > 5.0).to(tl.int32))\n+    nib = idx | tl.where(q < 0, 8, 0)\n+\n+    # Pack: byte p carries element 2p in the low nibble, 2p+1 in the high nibble.\n+    even = tl.sum(tl.where(tl.arange(0, 16)[None, :] % 2 == 0, nib, 0)\n+                  .reshape(BLOCK_R, 8, 2), 2)\n+    odd = tl.sum(tl.where(tl.arange(0, 16)[None, :] % 2 == 1, nib, 0)\n+                 .reshape(BLOCK_R, 8, 2), 2)\n+    byte = (even | (odd << 4)).to(tl.uint8)\n+    pack_cols = pid_b * 8 + tl.arange(0, 8)\n+    tl.store(out + r[:, None] * (N // 2) + pack_cols[None, :], byte, mask=live[:, None])\n+    tl.store(sf + r * SF + pid_b, bs, mask=live)\n+\n+\n+def nvfp4_quantize_triton(tensor_2d, global_scale):\n+    """[rows, N] bf16/fp16 -> (packed uint8 [rows, N//2], e4m3 scales [rows, N//16])."""\n+    rows, n = tensor_2d.shape\n+    if n % 16:\n+        raise ValueError(f"NVFP4 needs a multiple of 16 features, got {n}")\n+    out = torch.empty(rows, n // 2, dtype=torch.uint8, device=tensor_2d.device)\n+    sf = torch.empty(rows, n // 16, dtype=torch.float8_e4m3fn, device=tensor_2d.device)\n+    if rows == 0:\n+        return out, sf\n+    block_r = 8\n+    _nvfp4_quantize[(triton.cdiv(rows, block_r), n // 16)](\n+        tensor_2d.contiguous(), global_scale.reshape(1), out, sf, rows,\n+        N=n, BLOCK_R=block_r, num_warps=4)\n+    return out, sf\n+\n class NVFP4KVQuantizeUtil:\n     """Utility class for NVFP4 quantization and dequantization with two-level scaling\n     (global FP32 + block FP8 E4M3).\n@@ -181,9 +233,12 @@ class NVFP4KVQuantizeUtil:\n             is_sm120_supported,\n         )\n \n-        assert (\n+        # sm_89 (Ada) has neither kernel: nvfp4_kv_quantize is SM100/SM120 PTX and\n+        # flashinfer\'s fp4_quantize refuses with "Invalid backend: 89". The Triton\n+        # quantizer above covers it.\n+        _arc_triton_quant = not (\n             is_sm100_supported() or is_sm120_supported() or is_sm90_supported()\n-        ), "NVFP4 KV cache quantize requires SM100/SM120 or SM90 fallback GPU"\n+        )\n \n         b, m, n = tensor.shape\n         tensor_2d = tensor.reshape(b * m, n)\n@@ -202,7 +257,9 @@ class NVFP4KVQuantizeUtil:\n                 "NVFP4 global scale tensor must already be on the KV tensor device."\n             )\n \n-        if is_sm100_supported() or is_sm120_supported():\n+        if _arc_triton_quant:\n+            fp4_2d, scales_2d = nvfp4_quantize_triton(tensor_2d, global_scale)\n+        elif is_sm100_supported() or is_sm120_supported():\n             from flashinfer import nvfp4_kv_quantize\n \n             # nvfp4_kv_quantize takes global_scale directly (not inverted)\ndiff --git a/python/sglang/srt/server_args.py b/python/sglang/srt/server_args.py\nindex ff61693..7c5d369 100644\n--- a/python/sglang/srt/server_args.py\n+++ b/python/sglang/srt/server_args.py\n@@ -6084,12 +6084,14 @@ class ServerArgs:\n         attention_backend = resolved_view(self).attention_backend\n \n         if is_cuda():\n-            if self.kv_cache_dtype == "nvfp4" and not (\n-                is_sm100_supported() or is_sm120_supported()\n+            if (\n+                self.kv_cache_dtype == "nvfp4"\n+                and not (is_sm100_supported() or is_sm120_supported())\n+                and os.environ.get("ARC_QSA_FP4_FUSED", "1") != "1"\n             ):\n                 raise RuntimeError(\n-                    "--kv-cache-dtype=nvfp4 requires Blackwell SM100 or SM120. "\n-                    "Use --kv-cache-dtype=fp4_mx_block16 for the block-size-16 FP4 recipe."\n+                    "--kv-cache-dtype=nvfp4 requires Blackwell SM100 or SM120, or "\n+                    "ARC_QSA_FP4_FUSED=1 for the Triton quantize+decode path (sm_89)."\n                 )\n             if (\n                 prefill_backend != decode_backend and prefill_backend != "fa4"\n\n', '0041-ple-ngram-build-on-host-sm89.patch': '--- a/python/sglang/srt/models/qwen4_exp.py\n+++ b/python/sglang/srt/models/qwen4_exp.py\n@@ -512,18 +512,29 @@\n             and get_attention_dp_size() > 1\n             and not self.use_attn_tp_ngram\n         )\n-        self.ngram_embedding = VocabParallelEmbedding(\n-            padded_vocab_size,\n-            self.head_dim_per_ngram,\n-            params_dtype=(\n-                torch.float8_e4m3fn\n-                if (quant_config is not None and quant_config.get_name() == "fp8")\n-                or getattr(config, "ple_embedding_dtype", None) == "float8_e4m3fn"\n-                else torch.bfloat16\n-            ),\n-            output_dtype=torch.bfloat16,\n-            use_attn_tp_group=self.use_attn_tp_ngram,\n+        # arc-forge local probe: sglang builds the model under a\n+        # `torch.device("cuda")` context, so this table lands on GPU and is then\n+        # immediately deleted by Qwen4ExpPinnedHostEmbedding. The transient is\n+        # padded_vocab/tp_size * head_dim bytes: 11.9 GiB at TP4, 23.8 GiB at TP2 --\n+        # more than a 24 GB card has. Build it where it is going to live.\n+        _ple_offload_target = (\n+            torch.device("cpu")\n+            if getattr(config, "ple_offload_embedding", False)\n+            else nullcontext()\n         )\n+        with _ple_offload_target:\n+            self.ngram_embedding = VocabParallelEmbedding(\n+                padded_vocab_size,\n+                self.head_dim_per_ngram,\n+                params_dtype=(\n+                    torch.float8_e4m3fn\n+                    if (quant_config is not None and quant_config.get_name() == "fp8")\n+                    or getattr(config, "ple_embedding_dtype", None) == "float8_e4m3fn"\n+                    else torch.bfloat16\n+                ),\n+                output_dtype=torch.bfloat16,\n+                use_attn_tp_group=self.use_attn_tp_ngram,\n+            )\n         self.ngram_embedding.register_buffer(\n             "weight_scale", torch.ones(1, dtype=torch.bfloat16), persistent=True\n         )\n\n', '0042-fp4-dequant-workspace-budget-sm89.patch': 'diff --git a/python/sglang/srt/layers/quantization/fp4_kv_cache_quant_method.py b/python/sglang/srt/layers/quantization/fp4_kv_cache_quant_method.py\nindex d859396..ce44372 100644\n--- a/python/sglang/srt/layers/quantization/fp4_kv_cache_quant_method.py\n+++ b/python/sglang/srt/layers/quantization/fp4_kv_cache_quant_method.py\n@@ -38,6 +38,8 @@ from dataclasses import dataclass\n from enum import Enum\n from typing import Iterable, Optional\n \n+import os\n+\n import torch\n from torch import Tensor\n \n@@ -774,14 +776,39 @@ def _native_fp4(\n     )\n \n \n+_ARC_FUSED_FP4_BACKENDS = _NVFP4_PREFILL_BACKENDS | _NVFP4_DECODE_BACKENDS | frozenset(\n+    {"triton", "torch_native", "flex_attention"}\n+)\n+\n+\n+def _arc_fused_fp4_path() -> bool:\n+    """Only on non-Blackwell: SM100/SM120 keep the upstream trtllm-gen wiring."""\n+    if os.environ.get("ARC_QSA_FP4_FUSED", "1") != "1":\n+        return False\n+    from sglang.srt.utils import is_sm100_supported, is_sm120_supported\n+\n+    return not (is_sm100_supported() or is_sm120_supported())\n+\n+\n+\n KV_CACHE_ATTENTION_ACCESS_REGISTRY: dict[str, tuple[KVCacheAttentionAccess, ...]] = {\n     UnquantizedKVCacheMethod.name: (\n         _plain(_PREFILL, _ANY_BACKEND),\n         _plain(_DECODE, _ANY_BACKEND),\n     ),\n     NVFP4KVCacheMethod.name: (\n-        _dq_workspace(_PREFILL, _NVFP4_PREFILL_BACKENDS, _NVFP4_SCALE, _FP8_E4M3),\n-        _native_fp4(_DECODE, _NVFP4_DECODE_BACKENDS, _NVFP4_SCALE, _TORCH_FP4),\n+        (\n+            # sm_89 fused path (ARC 0039): prefill gathers and dequantizes the\n+            # selected context in one Triton pass, so no pool-sized FP8 dequant\n+            # workspace is allocated -- and none should be budgeted for either.\n+            _native_fp4(_PREFILL, _ARC_FUSED_FP4_BACKENDS, _NVFP4_SCALE, _TORCH_FP4),\n+            _native_fp4(_DECODE, _ARC_FUSED_FP4_BACKENDS, _NVFP4_SCALE, _TORCH_FP4),\n+        )\n+        if _arc_fused_fp4_path()\n+        else (\n+            _dq_workspace(_PREFILL, _NVFP4_PREFILL_BACKENDS, _NVFP4_SCALE, _FP8_E4M3),\n+            _native_fp4(_DECODE, _NVFP4_DECODE_BACKENDS, _NVFP4_SCALE, _TORCH_FP4),\n+        )\n     ),\n     FP4MXBlock16KVCacheMethod.name: (\n         _plain(_PREFILL, _FP4_MX_PREFILL_BACKENDS, _FP4_MX_SCALE, _BF16),\ndiff --git a/python/sglang/srt/model_executor/pool_configurator.py b/python/sglang/srt/model_executor/pool_configurator.py\nindex b8419fc..91e3359 100644\n--- a/python/sglang/srt/model_executor/pool_configurator.py\n+++ b/python/sglang/srt/model_executor/pool_configurator.py\n@@ -84,6 +84,23 @@ if TYPE_CHECKING:\n logger = logging.getLogger(__name__)\n \n \n+\n+def _fp4_declares_dequant_workspace(kv_cache_dtype_str: Optional[str]) -> bool:\n+    from sglang.srt.layers.quantization.fp4_kv_cache_quant_method import (\n+        KV_CACHE_ATTENTION_ACCESS_REGISTRY,\n+        KVCacheAttentionAccessKind,\n+        KV_CACHE_QUANT_REGISTRY,\n+    )\n+\n+    method = KV_CACHE_QUANT_REGISTRY.get(kv_cache_dtype_str)\n+    if method is None:\n+        return True  # unknown recipe: keep the conservative budget\n+    return any(\n+        access.kind == KVCacheAttentionAccessKind.DEQUANT_WORKSPACE\n+        for access in KV_CACHE_ATTENTION_ACCESS_REGISTRY.get(method.name, ())\n+    )\n+\n+\n def _dflash_draft_cell_size(kvc: KVCacheConfigurator) -> int:\n     """Bytes/token the DFLASH draft KV pool adds to the target\'s budget, 0 if none.\n \n@@ -340,8 +357,14 @@ class DefaultPoolConfigurator(MemoryPoolConfigurator):\n                 cell_size = (cell_size // 2) + (\n                     (n * k * effective_num_layers * 2 * kv_size) // scale_block_size\n                 )\n-                # FP4 prefill uses one shared FP8 dequant workspace across layers.\n-                cell_size += n * k * 2 * kv_size\n+                # FP4 prefill may use one shared FP8 dequant workspace across\n+                # layers -- but only charge for it when the KV cache method\n+                # actually declares one. fp4_mx_block16 never allocates it, and\n+                # the fused sm_89 path drops it, yet both were paying\n+                # n*k*2 bytes/token: on a 6-layer QSA model that is 512 of 2624\n+                # bytes/token, i.e. ~20% of the pool handed back for nothing.\n+                if _fp4_declares_dequant_workspace(self.kv_cache_dtype_str):\n+                    cell_size += n * k * 2 * kv_size\n             elif self.kv_cache_dtype_str == "mxfp8":\n                 scale_block_size = 32\n                 cell_size += (\n\n', '0043-nvfp4-cudagraph-scalar-sm89.patch': 'diff --git a/python/sglang/srt/layers/quantization/kvfp4_tensor.py b/python/sglang/srt/layers/quantization/kvfp4_tensor.py\n--- a/python/sglang/srt/layers/quantization/kvfp4_tensor.py\n+++ b/python/sglang/srt/layers/quantization/kvfp4_tensor.py\n@@ -247,9 +247,9 @@ class NVFP4KVQuantizeUtil:\n         # Keep scalar/0-d support for tests and future fallback paths, but do not\n         # silently move tensor scales here.\n         if isinstance(global_scale, (int, float)):\n-            global_scale = torch.tensor(\n-                [global_scale], dtype=torch.float32, device=tensor.device\n-            )\n+            global_scale = torch.empty(\n+                1, dtype=torch.float32, device=tensor.device\n+            ).fill_(global_scale)\n         elif global_scale.dim() == 0:\n             global_scale = global_scale.unsqueeze(0)\n         elif global_scale.device != tensor.device:\n@@ -306,9 +306,9 @@ class NVFP4KVQuantizeUtil:\n         # Keep scalar/0-d support for tests and future fallback paths, but do not\n         # silently move tensor scales here.\n         if isinstance(global_scale, (int, float)):\n-            global_scale = torch.tensor(\n-                [global_scale], dtype=torch.float32, device=quant_tensor.device\n-            )\n+            global_scale = torch.empty(\n+                1, dtype=torch.float32, device=quant_tensor.device\n+            ).fill_(global_scale)\n         elif global_scale.dim() == 0:\n             global_scale = global_scale.unsqueeze(0)\n         elif global_scale.device != quant_tensor.device:\n', '0044-qwen4-mtp-draft-topology.patch': 'diff --git a/python/sglang/srt/configs/model_config.py b/python/sglang/srt/configs/model_config.py\n--- a/python/sglang/srt/configs/model_config.py\n+++ b/python/sglang/srt/configs/model_config.py\n@@ -1830,6 +1830,27 @@ class ModelConfig:\n         self.hf_config.architectures[0] = "Qwen3NextForCausalLMMTP"\n         self.hf_config.num_nextn_predict_layers = 1\n \n+        if (\n+            is_draft_model\n+            and self.hf_config.architectures[0]\n+            == "Qwen4ExpForConditionalGeneration"\n+        ):\n+            # The target and draft initially share the HF config.  Isolate the\n+            # draft before reducing its topology to the bundled one-layer MTP\n+            # head; otherwise the target is accidentally reduced as well.\n+            from copy import deepcopy\n+\n+            self.hf_config = deepcopy(self.hf_config)\n+            self.hf_text_config = get_hf_text_config(self.hf_config)\n+            self.hf_config.architectures[0] = "Qwen4ExpForCausalLMMTP"\n+            text_config = self.hf_text_config\n+            text_config.num_nextn_predict_layers = 1\n+            # layers_block_type follows layer_types, not num_hidden_layers, so\n+            # both fields must shrink or the allocator charges 48 draft layers.\n+            text_config.num_hidden_layers = 1\n+            text_config.layer_types = ["full_attention"]\n+            text_config.full_attention_interval = 1\n+\n         if is_draft_model and self.hf_config.architectures[0] == "Qwen3MoeForCausalLM":\n             self.hf_config.architectures[0] = "Qwen3MoeForCausalLMMTP"\n             self.hf_config.num_nextn_predict_layers = 1\n'}

In [ ]:
%%writefile /tmp/arcforge_qwen38fn_nvfp4/src/arcforge/tir/__init__.py
"""TIR（tool-integrated reasoning）協定的單一實作。

這裡放的四樣東西，先前分別散在 `work/deepseek_api_bench/`（sandbox）、b200 的
`model_baseline/`（parsing）與 `nvarc_tir/hf_upload/config.json`（system prompt 與
tool schema），而 `work/tir_sft/eval_tir_qwen.py` 與 `work/expert_prune/eval_tir_prune.py`
各自複製了一份 prompt 渲染。學生模型是照 **teacher 生成訓練資料時那一份**協定訓練的，
所以協定只要有兩份就遲早會漂掉——評測看起來正常、分數卻在回答另一個問題。

`tir_config.json` 是 teacher rollout 當時的凍結設定，**不要改**：改了就不再是模型受訓的格式。
"""

__all__ = ["protocol", "parsing", "sandbox", "driver"]



In [ ]:
%%writefile /tmp/arcforge_qwen38fn_nvfp4/src/arcforge/tir/protocol.py
"""學生模型受訓時的那一份 TIR 協定：system prompt、tool schema、user prompt 渲染。

三者都必須與 teacher 產生訓練資料時逐位元組相同。`tir_config.json` 是當時的凍結設定，
`ARC_PROMPT` 抄自 `render_arc_prompt(style="harness")`，與
`work/expert_prune/eval_tir_prune.py` 裡那份逐字相同（那份是 2026-08-22 那批 eval 的
凍結 provenance，不再維護）。
"""

from __future__ import annotations

import json
from pathlib import Path

CONFIG_PATH = Path(__file__).with_name("tir_config.json")

# ARC-Prize harness 的 user prompt，與 render_arc_prompt(style="harness") 逐位元組相同。
ARC_PROMPT = (
    "You are participating in a puzzle solving competition. You are an expert at "
    "solving puzzles.\n\nBelow is a list of input and output pairs with a pattern. "
    "Your goal is to identify the pattern or transformation in the training examples "
    "that maps the input to the output, then apply that pattern to the test input to "
    "give a final output.\n\nRespond in the format of the training output examples"
    "\n\n--Training Examples--\n{training_examples}--End of Training Examples--\n\n"
    "--Test Input--\n{test_input}\n--End of Test Input--\n\nYour response:{tail}"
)


def load_config(path: str | Path | None = None) -> dict:
    """讀凍結的 TIR 設定。缺鍵就 raise——協定不完整比跑不起來更危險。"""
    cfg = json.loads(Path(path or CONFIG_PATH).read_text())
    for key in ("tool", "tool_name", "system_prompt_template", "rollout"):
        if key not in cfg:
            raise RuntimeError(f"{path or CONFIG_PATH}: 缺 {key!r}，這不是 TIR 設定")
    return cfg


def system_prompt(cfg: dict, max_turns: int) -> str:
    """`max_turns` 進 prompt 是刻意的——模型被訓練成知道自己還剩幾輪。"""
    return cfg["system_prompt_template"].format(
        tool_timeout=cfg["rollout"]["tool_timeout_s"],
        tool_max_output_chars=cfg["rollout"]["tool_max_output_chars"],
        max_turns=max_turns,
    )


def render_user_prompt(train_pairs: list[dict], test_input: list[list[int]]) -> str:
    ex = ""
    for i, p in enumerate(train_pairs):
        ex += "--Example %d-- \n\n INPUT: \n\n" % i
        ex += json.dumps(p["input"]) + "\n\n"
        ex += "OUTPUT: \n\n"
        ex += json.dumps(p["output"]) + "\n\n"
    return ARC_PROMPT.format(
        training_examples=ex, tail="\n", test_input=json.dumps(test_input)
    )



In [ ]:
%%writefile /tmp/arcforge_qwen38fn_nvfp4/src/arcforge/tir/parsing.py
import json
from typing import List, Optional, Callable
import re

# Type hint for the provider's JSON extraction function (can be refined later)
ProviderJsonExtractor = Callable[[str], Optional[List[List[int]]]]

# --- Helper Parsing Functions: Can be added to as needed ---

def backscan_json_parser(log_str: str) -> Optional[List[List[int]]]:
    """
    Extract the last valid JSON substring that matches the List[List] structure
    from the given log string by scanning backwards from the end.

    Parameters:
        log_str (str): The full log output text.

    Returns:
        The parsed List[List] object if found and valid, otherwise None.
    """
    last_bracket_idx = -1
    closing_bracket = None
    for i in range(len(log_str) - 1, -1, -1):
        char = log_str[i]
        if char in (']', '}'):
            last_bracket_idx = i
            closing_bracket = char
            break

    if last_bracket_idx == -1:
        return None

    opening_bracket = '[' if closing_bracket == ']' else '{'

    bracket_counter = 1 # Start at 1 to account for the found closing bracket
    start_idx = -1

    for i in range(last_bracket_idx - 1, -1, -1):
        char = log_str[i]
        if char == closing_bracket:
            bracket_counter += 1
        elif char == opening_bracket:
            bracket_counter -= 1
            if bracket_counter == 0:
                start_idx = i
                break

    if start_idx == -1:
        return None

    json_candidate = log_str[start_idx:last_bracket_idx+1]

    try:
        parsed_json = json.loads(json_candidate)

        # Validate the structure: must be a non-empty list of lists.
        if isinstance(parsed_json, list) and parsed_json and all(isinstance(row, list) for row in parsed_json):
            return parsed_json
        else:
            return None

    except json.JSONDecodeError:
        return None

def extract_from_boxed(log_str: str) -> Optional[List[List[int]]]:
    """
    Extracts JSON from a LaTeX-style \\boxed{} command in a string.
    """
    match = re.search(r"\\boxed\{(.*?)\}", log_str, re.DOTALL)
    if match:
        content = match.group(1).strip()
        try:
            # The content inside boxed is often a list of lists
            parsed_json = json.loads(content)
            if isinstance(parsed_json, list) and all(isinstance(i, list) for i in parsed_json):
                return parsed_json
        except json.JSONDecodeError:
            # If json.loads fails, it's not the JSON we're looking for.
            pass
    return None


# --- Main Parsing Orchestrator ---

def parse_and_validate_json(response: str) -> Optional[List[List[int]]]:
    """
    Orchestrates parsing by trying the provider extractor.
    Returns the parsed List[List[int]] or raises ValueError if validation fails.
    """
    parsing_attempts = [
        extract_from_boxed,
        backscan_json_parser
    ]

    for parser in parsing_attempts:
        result = parser(response)
        if result is not None:
            # Validate the structure: must be list of lists
            if isinstance(result, list) and all(isinstance(row, list) for row in result):
                return result # Return immediately on first success and validation
            else:
                # Raise error if structure is invalid, triggering retry in main loop
                raise ValueError(f"Parser {parser.__name__} produced invalid structure: {result!r}")

    # Raise an error here if no method works, triggering retry in main loop
    raise ValueError(f"Failed to parse response after all attempts: {response!r}")

if __name__ == "__main__":
    response = """### Summary and Final Answer:\nThe pattern transforms the input, which consists of two lists, into six lists in the output. Each of the first four output lists is constructed by repeating the original or reversed input lists three times. The fifth and sixth output lists are repetitions of the first and second output lists, respectively.\n\nFor the test input `[[3, 2], [7, 8]]`:\n1. The first input list `[3, 2]` is repeated three times to form the first output list.\n2. The second input list `[7, 8]` is repeated three times to form the second output list.\n3. The reversed first input list `[2, 3]` is repeated three times to form the third output list.\n4. The reversed second input list `[8, 7]` is repeated three times to form the fourth output list.\n5. The first output list is repeated as the fifth output list.\n6. The second output list is repeated as the sixth output list.\n\nThus, the output is:\n\n```python\n[[3, 2, 3, 2, 3, 2], [7, 8, 7, 8, 7, 8], [2, 3, 2, 3, 2, 3], [8, 7, 8, 7, 8, 7], [3, 2, 3, 2, 3, 2], [7, 8, 7, 8, 7, 8]]\n```\n\nFinal answer (formatted to match the training output style):\n\n```python\n[[3, 2, 3, 2, 3, 2], [7, 8, 7, 8, 7, 8], [2, 3, 2, 3, 2, 3], [8, 7, 8, 7, 8, 7], [3, 2, 3, 2, 3, 2], [7, 8, 7, 8, 7, 8]]\n```\n\n\\boxed{[[3, 2, 3, 2, 3, 2], [7, 8, 7, 8, 7, 8], [2, 3, 2, 3, 2, 3], [8, 7, 8, 7, 8, 7], [3, 2, 3, 2, 3, 2], [7, 8, 7, 8, 7, 8]]}'
"""
    print(parse_and_validate_json(response))



In [ ]:
%%writefile /tmp/arcforge_qwen38fn_nvfp4/src/arcforge/tir/sandbox.py
"""Stateful Python executor for ARC tool-use runs.

Runs as a subprocess, one per solve attempt. Speaks newline-delimited JSON:

    parent -> child   {"init": {"train_pairs": [...], "test_input": [[...]], "extra": {...}}}
    child  -> parent  {"ready": true}
    parent -> child   {"code": "print(1+1)"}
    child  -> parent  {"output": "2\\n", "timed_out": false}
    parent -> child   EOF                         (child exits)

The protocol runs over a dedicated inherited fd (number in ARC_SANDBOX_OUT_FD),
not stdout, so a C extension writing to fd 1 cannot corrupt it. stdout goes to
/dev/null; stderr goes to a file when the parent supplies stderr_path, because a
child that dies is otherwise undiagnosable.

WHAT THIS IS NOT: a security sandbox. b200 offers this user no OS-level isolation
(no bwrap, user namespaces blocked by AppArmor, no docker socket, no sudo, and
systemd --user silently ignores its sandboxing properties). What is enforced here:

  * RLIMIT_FSIZE (0, 0)  - kernel-enforced, blocks every write to a regular file,
                           and cannot be raised back because the hard limit is 0.
  * RLIMIT_DATA          - heap ceiling.
  * SIGALRM              - per-call wall-clock timeout that leaves session state
                           intact (raises KeyboardInterrupt inside the call).
  * attribute removal    - os.remove/rmtree/system/exec*, and subprocess, socket
                           and ctypes made un-importable.
  * scratch cwd          - the child runs in a throwaway temp dir, removed on exit.
                           RLIMIT_FSIZE blocks writing content but NOT the O_CREAT
                           itself, so `np.save(...)` still leaves a 0-byte file; the
                           child inherits the launcher's cwd, so without this those
                           land in whatever directory the run was started from. A
                           47k-rollout collection put `cov0.npy` in the repo working
                           tree before this was added.

Deletion is stopped by the attribute-removal bullet only, which is a Python-level
measure that deliberately-hostile code can defeat. It is sized for non-adversarial
code that a model writes while solving a puzzle, not for untrusted input.
"""

import ast
import io
import json
import os
import resource
import signal
import sys
import tempfile
import traceback

TIMEOUT = float(os.environ.get("ARC_SANDBOX_TIMEOUT", "20"))
MAX_HEAP_MB = int(os.environ.get("ARC_SANDBOX_MAX_HEAP_MB", "8192"))
MAX_OUTPUT_CHARS = int(os.environ.get("ARC_SANDBOX_MAX_OUTPUT_CHARS", "4000"))

# os names to drop. Deleted unconditionally: if one is missing the platform is not
# what this file was written against, and that should surface at startup rather
# than leave a hole open silently.
_OS_BLOCKED = [
    "remove", "unlink", "rmdir", "removedirs", "rename", "renames", "replace",
    "truncate", "ftruncate", "system", "popen", "chmod", "chown", "link",
    "symlink", "fork", "forkpty", "kill", "killpg", "abort",
    "execv", "execve", "execl", "execle", "execlp", "execvp", "execvpe",
    "spawnv", "spawnve", "spawnl", "spawnle", "spawnlp", "spawnvp",
]
_SHUTIL_BLOCKED = [
    "rmtree", "move", "copy", "copy2", "copyfile", "copytree", "chown",
    "make_archive", "unpack_archive",
]
# shutil is deliberately absent: its dangerous names are already removed above, and
# making it un-importable only risks breaking a numpy code path for no extra cover.
_UNIMPORTABLE = ["subprocess", "socket", "ctypes", "multiprocessing"]


def harden():
    """Apply the limits. numpy is imported first: it must load before the heap
    ceiling and the module surgery, not after."""
    resource.setrlimit(resource.RLIMIT_DATA, (MAX_HEAP_MB * 1024 * 1024,) * 2)

    import numpy  # noqa: F401  (imported here so it is warm before hardening)

    import shutil

    for name in _OS_BLOCKED:
        delattr(os, name)
    for name in _SHUTIL_BLOCKED:
        delattr(shutil, name)
    for name in _UNIMPORTABLE:
        sys.modules[name] = None

    # Last, and irreversible: hard limit 0 means the process cannot raise it back.
    resource.setrlimit(resource.RLIMIT_FSIZE, (0, 0))


class Session:
    """Jupyter-ish: last expression is echoed, namespace persists across calls."""

    def __init__(self, timeout: float, max_output_chars: int):
        self._timeout = timeout
        self._max_output_chars = max_output_chars
        self.ns: dict = {"__builtins__": __builtins__, "__name__": "__main__"}

    def load_task(self, train_pairs, test_input, extra):
        """`test_input=None` leaves the name undefined: a joint multi-test solve
        exposes `test_inputs` via `extra` instead, and a stale singular name next
        to it would be an invitation to solve the wrong grid. `extra` is a dict of
        additional names (JSON values) and must not shadow the three built-ins."""
        import numpy as np

        self.ns["np"] = np
        self.ns["train_pairs"] = train_pairs
        if test_input is not None:
            self.ns["test_input"] = test_input
        clash = {"np", "train_pairs", "test_input"} & set(extra)
        if clash:
            raise ValueError(f"extra namespace shadows built-in sandbox names: {sorted(clash)}")
        self.ns.update(extra)

    def execute(self, code: str):
        stdout_buf, stderr_buf = io.StringIO(), io.StringIO()
        old_stdout, old_stderr = sys.stdout, sys.stderr
        timed_out = False

        def on_alarm(signum, frame):
            raise KeyboardInterrupt(f"execution exceeded {self._timeout:g}s")

        old_handler = signal.signal(signal.SIGALRM, on_alarm)
        signal.setitimer(signal.ITIMER_REAL, self._timeout)
        try:
            sys.stdout, sys.stderr = stdout_buf, stderr_buf
            self._run(code)
        except KeyboardInterrupt as e:
            timed_out = True
            stderr_buf.write(f"[interrupted] {e}\n")
        except SystemExit as e:
            stderr_buf.write(f"SystemExit: {e.code}\n")
        except SyntaxError as e:
            # No user frame exists yet, so the generic formatter would strip this
            # down to a bare "SyntaxError:". format_exception_only keeps the
            # offending line and the caret.
            stderr_buf.write("".join(traceback.format_exception_only(type(e), e)))
        except BaseException:
            stderr_buf.write(self._user_traceback(traceback.format_exc()))
        finally:
            signal.setitimer(signal.ITIMER_REAL, 0)
            signal.signal(signal.SIGALRM, old_handler)
            sys.stdout, sys.stderr = old_stdout, old_stderr

        out, err = stdout_buf.getvalue(), stderr_buf.getvalue()
        if err:
            text = f"{out.rstrip()}\n{err}" if out.strip() else err
        elif not out.strip():
            text = "[no output — use print() to see results]"
        else:
            text = out
        return self._truncate(text), timed_out

    def _run(self, code: str):
        """Exec, echoing the value of a trailing expression."""
        tree = ast.parse(code)
        if not tree.body:
            return
        if isinstance(tree.body[-1], ast.Expr):
            if len(tree.body) > 1:
                head = ast.Module(body=tree.body[:-1], type_ignores=[])
                exec(compile(head, "<cell>", "exec"), self.ns)
            tail = ast.Expression(body=tree.body[-1].value)
            value = eval(compile(tail, "<cell>", "eval"), self.ns)
            if value is not None:
                print(repr(value))
        else:
            exec(compile(tree, "<cell>", "exec"), self.ns)

    @staticmethod
    def _user_traceback(tb: str) -> str:
        """Keep only frames from the model's own code, plus the exception line."""
        lines = tb.strip().split("\n")
        kept, i = [], 0
        while i < len(lines):
            if lines[i].strip().startswith('File "'):
                is_user = "<cell>" in lines[i]
                if is_user:
                    kept.append(lines[i])
                i += 1
                while i < len(lines):
                    nxt = lines[i]
                    if nxt.strip().startswith('File "') or (nxt.strip() and not nxt.startswith(" ")):
                        break
                    if is_user:
                        kept.append(nxt)
                    i += 1
            else:
                kept.append(lines[i])
                i += 1
        return "\n".join(kept) + "\n"

    def _truncate(self, text: str) -> str:
        limit = self._max_output_chars
        if limit <= 0 or len(text) <= limit:
            return text
        head = limit // 3
        note = f"\n\n... [truncated: {len(text)} chars exceeded the {limit} char limit] ...\n\n"
        return text[:head] + note + text[-(limit - head):]


# ============================================================================
# Parent-side client
# ============================================================================


class SandboxError(RuntimeError):
    pass


class Sandbox:
    """Owns one executor subprocess. Use as a context manager; the process is
    killed on exit even if the solve loop raised."""

    def __init__(self, train_pairs, test_input, python=None, timeout=None,
                 max_output_chars=None, max_heap_mb=None, startup_timeout=120.0,
                 stderr_path=None, extra_ns=None):
        self._train_pairs = train_pairs
        self._test_input = test_input
        # Extra names to define in the child's namespace (JSON-serialisable values),
        # e.g. `test_inputs` for a joint multi-test solve. See Session.load_task.
        self._extra_ns = dict(extra_ns) if extra_ns else {}
        self._python = python or sys.executable
        self._startup_timeout = startup_timeout
        self._env_overrides = {
            k: str(v)
            for k, v in {
                "ARC_SANDBOX_TIMEOUT": timeout,
                "ARC_SANDBOX_MAX_OUTPUT_CHARS": max_output_chars,
                "ARC_SANDBOX_MAX_HEAP_MB": max_heap_mb,
            }.items()
            if v is not None
        }
        self._stderr_path = stderr_path
        self._stderr_file = None
        self._proc = None
        self._out = None
        self._scratch = None
        self.n_restarts = 0

    def __enter__(self):
        import subprocess as _sp  # parent side only; the child cannot import this

        read_fd, write_fd = os.pipe()
        env = dict(os.environ)
        env.update(self._env_overrides)
        env["ARC_SANDBOX_OUT_FD"] = str(write_fd)
        env["PYTHONDONTWRITEBYTECODE"] = "1"
        env["PYTHONUNBUFFERED"] = "1"
        # One BLAS thread per sandbox: dozens of these run at once, and a default
        # thread pool per process would oversubscribe the box many times over.
        for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
                    "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
            env[var] = "1"

        # stderr goes to a file rather than /dev/null: the protocol is on its own
        # fd, so nothing here can corrupt it, and a child that dies leaves a reason.
        if self._stderr_path:
            self._stderr_file = open(self._stderr_path, "w")
            stderr_target = self._stderr_file
        else:
            stderr_target = _sp.DEVNULL
        # Run the child in a throwaway directory, never in the caller's cwd.
        #
        # RLIMIT_FSIZE (0, 0) stops the kernel from writing any *content*, but an
        # O_CREAT still succeeds, so `np.save("cov0.npy", a)` leaves a 0-byte file
        # behind. The child inherits the launcher's cwd, so for a run started from
        # the repo root that litter lands in the git working tree: the 47k-rollout
        # collection on 2026-08-30 left `cov0.npy` there, and an earlier eval left
        # `diag2.npy` in the main checkout.
        #
        # A scratch cwd is the right layer for this. Blocking more of Python's file
        # API would not help -- the creation is a kernel-level open(), and the
        # docstring above is explicit that this is a blast-radius limiter, not a
        # security boundary. Bounding *where* the blast lands is exactly its job.
        self._scratch = tempfile.mkdtemp(prefix="arc-sandbox-")
        self._proc = _sp.Popen(
            [self._python, os.path.abspath(__file__)],
            stdin=_sp.PIPE,
            stdout=_sp.DEVNULL,
            stderr=stderr_target,
            env=env,
            cwd=self._scratch,
            pass_fds=(write_fd,),
            text=True,
        )
        os.close(write_fd)
        self._out = os.fdopen(read_fd, "r")

        self._send({"init": {"train_pairs": self._train_pairs, "test_input": self._test_input,
                             "extra": self._extra_ns}})
        ready = self._recv(self._startup_timeout)
        if not ready.get("ready"):
            raise SandboxError(f"sandbox refused to start: {ready}")
        return self

    def __exit__(self, *exc):
        proc, self._proc = self._proc, None
        if proc is not None:
            proc.kill()
            proc.wait()
        if self._out is not None:
            self._out.close()
            self._out = None
        if self._stderr_file is not None:
            self._stderr_file.close()
            self._stderr_file = None
        if self._scratch is not None:
            # Only ever holds 0-byte files the child managed to create, but leaving
            # one temp dir per rollout behind would be 47k of them over a full run.
            import shutil as _shutil
            _shutil.rmtree(self._scratch, ignore_errors=True)
            self._scratch = None
        return False

    def restart(self):
        """Kill the child and start a fresh one with the same task; every name the
        user defined is gone. For a caller whose `run` raised SandboxError (the
        child hung past the outer backstop -- the SIGALRM interrupt cannot stop
        code that swallows KeyboardInterrupt or sits inside a C call)."""
        self.__exit__(None, None, None)
        self.n_restarts += 1
        return self.__enter__()

    def run(self, code: str, wait: float = 300.0):
        """Returns (output_text, timed_out)."""
        self._send({"code": code})
        reply = self._recv(wait)
        return reply["output"], reply["timed_out"]

    def _send(self, obj):
        if self._proc is None:
            raise SandboxError("sandbox is not running")
        # 子行程死了（OOM、被 kill、模型碼把自己弄崩）但 handle 還在：寫 stdin 會 BrokenPipe。
        # 轉成 SandboxError，讓呼叫端走和「300s 無回應」一樣的 restart 路徑，而不是整題報廢
        # （2026-09-02 d-max-100k 的 db695cfb）。
        try:
            self._proc.stdin.write(json.dumps(obj, ensure_ascii=False) + "\n")
            self._proc.stdin.flush()
        except (BrokenPipeError, OSError) as e:
            raise SandboxError(f"sandbox process died (pipe broken: {e}; exit code {self._proc.poll()})") from e

    def _recv(self, wait: float):
        """The executor's own SIGALRM bounds each call, so a missing reply means
        the process died, not that it is slow. `wait` is the outer backstop."""
        import selectors

        sel = selectors.DefaultSelector()
        sel.register(self._out, selectors.EVENT_READ)
        try:
            if not sel.select(wait):
                raise SandboxError(f"sandbox produced no reply within {wait:g}s")
        finally:
            sel.close()
        line = self._out.readline()
        if not line:
            self._proc.poll()
            code = self._proc.wait(timeout=10)
            raise SandboxError(
                f"sandbox stopped writing (exit code {code}); stderr: {self._read_stderr()}"
            )
        return json.loads(line)

    def _read_stderr(self, limit=800):
        if self._stderr_file is not None:
            self._stderr_file.flush()
        if not self._stderr_path or not os.path.exists(self._stderr_path):
            return "(not captured)"
        with open(self._stderr_path, errors="replace") as f:
            text = f.read().strip()
        return text[-limit:] if text else "(empty)"


# ============================================================================
# Child entry point
# ============================================================================


def main():
    out_fd = int(os.environ["ARC_SANDBOX_OUT_FD"])
    out = os.fdopen(out_fd, "w")

    harden()
    session = Session(timeout=TIMEOUT, max_output_chars=MAX_OUTPUT_CHARS)

    def reply(obj):
        out.write(json.dumps(obj, ensure_ascii=False) + "\n")
        out.flush()

    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue
        req = json.loads(line)

        if "init" in req:
            init = req["init"]
            session.load_task(init["train_pairs"], init["test_input"], init["extra"])
            reply({"ready": True})
            continue

        code = req.get("code")
        if code is None:
            break
        text, timed_out = session.execute(code)
        reply({"output": text, "timed_out": timed_out})


if __name__ == "__main__":
    main()



In [ ]:
%%writefile /tmp/arcforge_qwen38fn_nvfp4/src/arcforge/tir/driver.py
"""跑 TIR rollout 的引擎：多輪對話 + python sandbox，回傳解出來的 grid。

與 `work/expert_prune/eval_tir_prune.py`（2026-08-22 那批 eval 的凍結 provenance）相比，
這裡多了三件提交才需要的東西：

1. **每條 rollout 的生成量上限**（`Budget.gen_cap`）。b200 的 334 條 transcript 顯示答對的鏈
   平均 59.5k token、答錯的 112.2k，所以砍尾巴主要砍掉的是已經卡住的 rollout。上限是把
   固定的牆鐘預算換成「跑幾條」的唯一旋鈕。
2. **全域 deadline**。比賽 rerun 到點就砍，`submission.json` 沒寫出來等於 0 分，
   所以排程必須在還來得及的時候停止發新的 rollout。
3. **sandbox 呼叫改走 thread**。原版直接在 coroutine 裡呼叫阻塞的 `sb.run()`，
   一個 20 秒的 tool call 會把整個 event loop 連同其他數十條 rollout 一起卡住。
   L4 的 CPU 比 b200 少得多，這在提交環境會是實際傷害。

答案來源分 `content` 與 `reasoning` 兩種並分開記：受過 SFT 的模型應該把 grid 放進 content，
放在 thinking block 裡代表協定壞了（量化或剪枝過頭的典型症狀），那是要看見的訊號而不是
要默默吸收的細節。
"""

from __future__ import annotations

import asyncio
import json
import time
from collections.abc import Sequence
from dataclasses import dataclass, field
from typing import Any, Callable

import httpx

from . import protocol
from .parsing import backscan_json_parser


@dataclass
class Budget:
    """一條 rollout 花得起多少。"""

    gen_cap: int = 131_072
    """整條 rollout 的生成 token 上限（累加每輪 completion_tokens）。"""
    max_turns: int = 120
    max_tokens: int = 65_536
    """單輪生成上限，再被 context 與 gen_cap 夾一次。"""
    context_length: int = 262_144
    """必須等於 serving 的 --context-length。"""
    context_margin: int = 6_000
    """留給 tool 輸出與 chat template 的餘裕。"""
    min_turn_tokens: int = 2_048
    """低於這個數就沒必要再發一輪。"""


@dataclass
class Sampling:
    temperature: float = 1.0
    top_p: float = 0.95
    top_k: int = -1
    """預設與 teacher 產生訓練資料時一致（`tir_config.json` 的 sampling 段：
    temperature 1.0 / top_p 0.95，**沒有 top_k**）。調低就偏離訓練分布。

    `top_k` 一定要明確送出，不能不送。sglang 的 `--sampling-defaults` 預設是 `model`，
    會把 checkpoint 的 `generation_config.json` 當成沒指定欄位的預設值——而這顆的
    generation_config 寫著 `top_k: 20`。不送就等於在 248k 詞表上只留前 20 個候選，
    十萬 token 的推理鏈會變得重複且收不了尾，而且**請求裡看不出來**。
    -1 表示不截斷。"""


@dataclass
class Rollout:
    task_id: str
    test_idx: int
    attempt: int
    train_pairs: list[dict]
    test_input: list[list[int]]
    expected: list[list[int]] | None = None
    """本地 dry run 才有；比賽 test set 沒有答案。"""

    @property
    def uid(self) -> str:
        return f"{self.task_id}#{self.test_idx}"


@dataclass
class Result:
    task_id: str
    test_idx: int
    attempt: int
    grid: list[list[int]] | None
    answer_src: str
    stop_reason: str
    n_turns: int
    n_tool_calls: int
    n_tool_errors: int
    gen_tokens: int
    latency_s: float
    correct: bool | None = None
    replica: str = ""
    """這條 rollout 從頭到尾用的那個 base_url（DP 下附 #dpN）。看負載有沒有歪掉。"""
    n_prompt_tokens: int = 0
    n_cached_tokens: int = 0
    """整條 rollout 累加的 prompt / 其中命中前綴快取的 token 數。

    需要 server 開 `--enable-cache-report`，否則 `prompt_tokens_details` 是 null、
    這個值恆為 0。**這是整條管線最重要的健康指標**：TIR 每輪重送完整歷史，靠
    radix cache 命中那 ~130k 前綴。命中率一崩，每輪都變成整條 re-prefill——慢十倍
    且不報錯，而 Kaggle 上看不到 server log。有了它就不必靠 log 也看得見。"""
    messages: list[dict] = field(default_factory=list, repr=False)


async def _chat(client, base_url, payload, retries, backoff=3.0):
    """4xx 直接炸——那是我們自己的請求有問題，重試只會浪費 deadline。"""
    for retry in range(retries + 1):
        try:
            r = await client.post(f"{base_url}/chat/completions", json=payload)
            r.raise_for_status()
            return r.json()
        except (httpx.TransportError, httpx.HTTPStatusError) as e:
            status = getattr(getattr(e, "response", None), "status_code", None)
            if (status and status < 500) or retry == retries:
                raise
            await asyncio.sleep(backoff * (retry + 1))
    raise AssertionError("unreachable")


async def run_rollout(
    client: httpx.AsyncClient,
    rollout: Rollout,
    *,
    base_url: str,
    model: str,
    cfg: dict,
    sandbox_cls: type,
    budget: Budget,
    sampling: Sampling,
    deadline: float | None = None,
    retries: int = 3,
    keep_messages: bool = False,
    should_abort: Callable[[], bool] | None = None,
    dp_rank: int | None = None,
) -> Result:
    """跑一條 rollout。`deadline` 是 `time.monotonic()` 座標。

    `should_abort` 與 deadline 走同一條路：呼叫端用它表達「整個 run 已經沒救了」
    （例如 server 死掉、每條 rollout 都是 ConnectError），讓在飛的 rollout 收手
    而不是把剩下的預算全部燒在重試上。
    """
    sys_prompt = protocol.system_prompt(cfg, budget.max_turns)
    messages: list[dict] = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": protocol.render_user_prompt(
            rollout.train_pairs, rollout.test_input)},
    ]
    n_turns = n_calls = n_errs = 0
    gen_used = 0
    prompt_tok = cached_tok = 0
    last_total = 0
    stop = "turn_cap"
    grid: list[list[int]] | None = None
    answer_src = ""
    t0 = time.monotonic()

    with sandbox_cls(
        rollout.train_pairs,
        rollout.test_input,
        timeout=cfg["rollout"]["tool_timeout_s"],
        max_output_chars=cfg["rollout"]["tool_max_output_chars"],
    ) as sb:
        for _turn in range(budget.max_turns):
            if deadline is not None and time.monotonic() >= deadline:
                stop = "deadline"
                break
            if should_abort is not None and should_abort():
                stop = "aborted"
                break

            # 三個上限取最小：單輪上限、context 餘額、整條 rollout 的生成餘額。
            # last_total 是上一輪的 total_tokens，即這輪 prompt 在附上 tool 結果前的長度。
            ctx_budget = budget.context_length - last_total - budget.context_margin
            gen_budget = budget.gen_cap - gen_used
            mt = min(budget.max_tokens, ctx_budget, gen_budget)
            if mt < budget.min_turn_tokens:
                stop = "gen_cap" if gen_budget <= ctx_budget else "context_exhausted"
                break

            n_turns += 1
            payload = {
                "model": model,
                "messages": messages,
                "tools": [cfg["tool"]],
                "temperature": sampling.temperature,
                "top_p": sampling.top_p,
                "top_k": sampling.top_k,
                "max_tokens": mt,
            }
            if dp_rank is not None:
                # 這條 rollout 的每一輪都送同一個 rank，radix cache 才留得住前綴。
                # rank 不合法時 sglang 回 ValueError（4xx），_chat 不會重試——那是
                # 我們自己的請求有問題，重試只會浪費 deadline。
                payload["routed_dp_rank"] = dp_rank
            data = await _chat(client, base_url, payload, retries)

            usage = data.get("usage") or {}
            last_total = int(usage.get("total_tokens") or last_total)
            gen_used += int(usage.get("completion_tokens") or 0)
            prompt_tok += int(usage.get("prompt_tokens") or 0)
            details = usage.get("prompt_tokens_details") or {}
            cached_tok += int(details.get("cached_tokens") or 0)

            msg = data["choices"][0]["message"]
            finish = data["choices"][0].get("finish_reason")
            tool_calls = msg.get("tool_calls") or []

            if tool_calls:
                # reasoning_content 一定要回灌。訓練資料保留了每一輪的 <think>（TIR 只有一次
                # user query，chat template 會為 last_query_index 之後的每則 assistant 訊息
                # 渲染 thinking；tool 結果是 role "tool"，不算 query）。丟掉的話先前每一輪都
                # 會被渲染成空的 <think>\n\n</think>，模型就被 condition 在一段它訓練時沒見過
                # 的推理歷史上。
                asst: dict[str, Any] = {
                    "role": "assistant",
                    "content": msg.get("content") or "",
                    "tool_calls": tool_calls,
                }
                if msg.get("reasoning_content"):
                    asst["reasoning_content"] = msg["reasoning_content"]
                messages.append(asst)

                for call in tool_calls:
                    n_calls += 1
                    fn = call["function"]
                    if fn["name"] != cfg["tool_name"]:
                        n_errs += 1
                        out = f"Error: no tool named {fn['name']!r}."
                    else:
                        args = fn["arguments"]
                        code = (json.loads(args) if isinstance(args, str) else args)["code"]
                        # 阻塞的 subprocess 往返移出 event loop，否則一個慢的 tool call
                        # 會連坐所有同時在飛的 rollout。
                        out, timed = await asyncio.to_thread(sb.run, code)
                        n_errs += int(timed)
                    messages.append({"role": "tool",
                                     "tool_call_id": call.get("id", ""),
                                     "content": out})
                if finish == "length":
                    # 這一輪的 max_tokens 是三個上限取最小，所以 finish_reason="length"
                    # 可能只是「整條 rollout 的生成餘額用完了」。照字面記成 length 會讓
                    # Kaggle 上的 stop_reason 直接說謊：看起來像模型話太多被單輪截斷，
                    # 實際上是 gen_cap 設太緊。這兩者要調的旋鈕不同。
                    stop = "gen_cap" if gen_used >= budget.gen_cap else "length"
                    break
                continue

            # 沒有 tool call = 最終答案。
            messages.append({"role": "assistant", "content": msg.get("content") or ""})
            grid = backscan_json_parser(msg.get("content") or "")
            answer_src = "content" if grid is not None else ""
            if grid is None:
                grid = backscan_json_parser(msg.get("reasoning_content") or "")
                answer_src = "reasoning" if grid is not None else ""
            if grid is not None:
                stop = "answered" if answer_src == "content" else "answered_in_reasoning"
            elif finish == "length":
                stop = "gen_cap" if gen_used >= budget.gen_cap else "length"
            else:
                stop = "malformed"
            break

    # 被 cap／deadline／turn_cap 中斷的，最後一則 assistant 訊息裡可能已經有答案了。
    if grid is None and messages:
        for m in reversed(messages):
            if m["role"] != "assistant":
                continue
            for key, src in (("content", "content"), ("reasoning_content", "reasoning")):
                found = backscan_json_parser(m.get(key) or "")
                if found is not None:
                    grid, answer_src = found, f"salvage_{src}"
                    break
            break

    correct = None if rollout.expected is None else (grid == rollout.expected)
    return Result(
        task_id=rollout.task_id, test_idx=rollout.test_idx, attempt=rollout.attempt,
        grid=grid, answer_src=answer_src, stop_reason=stop, n_turns=n_turns,
        n_tool_calls=n_calls, n_tool_errors=n_errs, gen_tokens=gen_used,
        latency_s=time.monotonic() - t0, correct=correct,
        n_prompt_tokens=prompt_tok, n_cached_tokens=cached_tok,
        messages=messages if keep_messages else [],
    )


async def run_all(
    rollouts: list[Rollout],
    *,
    base_url: str | Sequence[str],
    model: str,
    cfg: dict,
    sandbox_cls: type,
    budget: Budget,
    sampling: Sampling,
    concurrency: int,
    dp_size: int = 1,
    deadline: float | None = None,
    on_result: Callable[[Result], None] | None = None,
    http_timeout: float = 3600.0,
    retries: int = 3,
    keep_messages: bool = False,
    should_abort: Callable[[], bool] | None = None,
) -> list[Result]:
    """照順序發 rollout，每完成一條就回呼一次。到 deadline 就取消還在飛的。

    `on_result` 是提交管線把答案落地的地方——不要等全部跑完才寫檔，跑到一半被砍是
    預期中的結局而不是意外。

    Data parallel：一條 rollout 從頭到尾必須落在同一個 DP worker
    ----------------------------------------------------------
    這不是最佳化，是正確性需求。TIR 每一輪都把完整歷史重送，靠那個 worker 的 radix
    cache 命中 ~130k 的前綴、只 prefill 新的 tool 輸出。**sglang 的 DP 預設是
    round-robin，沒有 session affinity**（`data_parallel_controller.py` 的
    `round_robin_scheduler`），所以第二輪很可能落到另一個 worker——那邊的 cache 裡
    什麼都沒有，整條 context 重新 prefill。而且不報錯，只是慢十倍。

    釘法有兩種，這裡都支援，用「路由目標」統一表示：

    * `dp_size > 1`：單一 server 開 `--dp-size`，每個 request 帶
      `routed_dp_rank`。`DataParallelController.maybe_external_dp_rank_routing`
      在 round-robin **之前**短路（0.5.17 實測），而且 rank 不合法會 `ValueError`
      而不是默默 fallback。這是首選——一個 process、一個 port。
    * `base_url` 給一串：各自獨立的 server，各自一個 URL。

    指派用「當下 in-flight 最少的目標」，在取得全域併發名額的當下決定一次，之後不再
    更動。**刻意不用 hash(session_id) % dp_size**：那是給無狀態/分散式 client 的做法，
    而我們是單一 driver、知道每個目標當下的 in-flight 數，least-loaded 嚴格更均勻
    （hash 會讓幾條長 rollout 剛好撞在同一個 rank 上）。`Result.replica` 記下用的是
    哪一個，負載歪掉看得見。
    """
    urls = [base_url] if isinstance(base_url, str) else list(base_url)
    if not urls:
        raise ValueError("base_url 是空的")
    if dp_size < 1:
        raise ValueError(f"dp_size 要 >= 1，拿到 {dp_size}")
    # 路由目標 = (url, dp_rank)。dp_size == 1 時 rank 是 None，請求裡就不帶那個欄位。
    targets = [(u, r if dp_size > 1 else None) for u in urls for r in range(dp_size)]
    # httpx 預設 Limits(max_connections=100, max_keepalive_connections=20)，會在
    # concurrency > 100 時靜默卡在 100；keepalive 太小則讓連線在每一輪之間被拆掉重撥。
    limits = httpx.Limits(max_connections=concurrency + 16,
                          max_keepalive_connections=concurrency + 16)
    sem = asyncio.Semaphore(concurrency)
    inflight = [0] * len(targets)
    results: list[Result] = []

    async with httpx.AsyncClient(timeout=http_timeout, limits=limits) as client:
        async def one(r: Rollout) -> None:
            async with sem:
                if deadline is not None and time.monotonic() >= deadline:
                    return
                if should_abort is not None and should_abort():
                    return
                idx = min(range(len(targets)), key=lambda i: inflight[i])
                inflight[idx] += 1
                url, dp_rank = targets[idx]
                try:
                    res = await run_rollout(
                        client, r, base_url=url, model=model, cfg=cfg,
                        sandbox_cls=sandbox_cls, budget=budget, sampling=sampling,
                        deadline=deadline, retries=retries, keep_messages=keep_messages,
                        should_abort=should_abort, dp_rank=dp_rank)
                except asyncio.CancelledError:
                    raise
                except Exception as e:  # noqa: BLE001
                    res = Result(task_id=r.task_id, test_idx=r.test_idx, attempt=r.attempt,
                                 grid=None, answer_src="", stop_reason=f"error:{type(e).__name__}",
                                 n_turns=0, n_tool_calls=0, n_tool_errors=0,
                                 gen_tokens=0, latency_s=0.0)
                finally:
                    inflight[idx] -= 1
                res.replica = url if dp_rank is None else f"{url}#dp{dp_rank}"
            results.append(res)
            if on_result is not None:
                on_result(res)

        tasks = [asyncio.create_task(one(r)) for r in rollouts]
        timeout = None if deadline is None else max(0.0, deadline - time.monotonic())
        done, pending = await asyncio.wait(tasks, timeout=timeout)
        for t in pending:
            t.cancel()
        if pending:
            await asyncio.gather(*pending, return_exceptions=True)
    return results



In [ ]:
%%writefile /tmp/arcforge_qwen38fn_nvfp4/src/arcforge/tir/submission.py
"""ARC Prize `submission.json` 的落地，naive（`run_submission.py`）與 agentic（`agentic_tir.py`）共用。

官方格式：`{task_id: [{"attempt_1": grid, "attempt_2": grid}, ...]}`，list 與題目的 `test` 順序對齊。
每一格都必須有兩個 attempt，所以建檔時先用 test input 本身填滿（identity 偶爾是對的，形狀一定合法），
之後每收到一個答案就整份重寫（tmp + rename，被砍在寫到一半不會留下半份 JSON）。

grid 在寫進去之前要驗：非空矩形、每格 0–9 的整數、邊長 ≤ 30。舊版不驗——模型吐出參差列或浮點數會
原封不動寫進檔案，而計分器對壞格子的處置沒有文件保證。不合法的答案**留著 fallback 不動**並記在
`rejected`，呼叫端要印出來。
"""

from __future__ import annotations

import json
import os
from pathlib import Path

MAX_SIDE = 30


def validate_grid(g) -> str | None:
    """合法回 None，否則回一句原因。bool 是 int 的子類別，要明確擋掉。"""
    if not isinstance(g, list) or not g:
        return "not a non-empty list"
    if not all(isinstance(r, list) and r for r in g):
        return "every row must be a non-empty list"
    w = len(g[0])
    if any(len(r) != w for r in g):
        return f"ragged rows ({sorted({len(r) for r in g})})"
    if len(g) > MAX_SIDE or w > MAX_SIDE:
        return f"{len(g)}x{w} exceeds {MAX_SIDE}x{MAX_SIDE}"
    for r in g:
        for v in r:
            if isinstance(v, bool) or not isinstance(v, int) or not 0 <= v <= 9:
                return f"cell {v!r} is not an int in 0..9"
    return None


class SubmissionStore:
    """`{task_id: [{"attempt_1": grid, "attempt_2": grid}, ...]}`，每次更新就落地。"""

    def __init__(self, challenges: dict, out_path: Path):
        self.out = Path(out_path)
        self.data = {
            tid: [{"attempt_1": t["input"], "attempt_2": t["input"]} for t in task["test"]]
            for tid, task in challenges.items()
        }
        self.filled: set[tuple[str, int, int]] = set()
        self.rejected: list[dict] = []
        self.flush()

    def put(self, task_id: str, test_idx: int, attempt: int, grid) -> bool:
        """寫一格。回傳有沒有真的寫進去（None 或不合法的 grid 都不寫、不動 fallback）。"""
        if grid is None:
            return False
        why = validate_grid(grid)
        if why is not None:
            self.rejected.append({"task_id": task_id, "test_idx": test_idx, "attempt": attempt, "why": why})
            return False
        self.data[task_id][test_idx][f"attempt_{attempt + 1}"] = grid
        self.filled.add((task_id, test_idx, attempt))
        self.flush()
        return True

    def put_attempts(self, task_id: str, test_idx: int, grids: list) -> int:
        """一次給這一格的 attempt 清單（順序就是優先序，只取前兩個）。回傳寫進去的格數。"""
        return sum(int(self.put(task_id, test_idx, a, g)) for a, g in enumerate(grids[:2]))

    def has(self, task_id: str, test_idx: int, attempt: int) -> bool:
        return (task_id, test_idx, attempt) in self.filled

    @property
    def n_cells(self) -> int:
        return sum(len(v) for v in self.data.values())

    def flush(self) -> None:
        tmp = self.out.with_suffix(self.out.suffix + ".tmp")
        tmp.write_text(json.dumps(self.data))
        os.replace(tmp, self.out)



In [ ]:
%%writefile /tmp/arcforge_qwen38fn_nvfp4/src/arcforge/tir/tir_config.json
{
  "teacher_model": "deepseek-ai/DeepSeek-V4-Flash-0731",
  "source_puzzles": "NVARC synthetic ARC-style relational tasks (47,337 puzzles)",
  "reasoning_mode": "tool-integrated reasoning (TIR), thinking mode, effort=max",
  "sampling": {
    "temperature": 1.0,
    "top_p": 0.95,
    "max_tokens": 131072
  },
  "rollout": {
    "n_train_pairs": 4,
    "attempts": 1,
    "max_turns": 120,
    "tool_timeout_s": 20,
    "tool_max_output_chars": 4000,
    "prompt_style": "harness"
  },
  "serving": {
    "engine": "sglang",
    "tp": 4,
    "kv_cache_dtype": "fp8_e4m3",
    "speculative": "DSPARK",
    "concurrency": 136
  },
  "tool": {
    "type": "function",
    "function": {
      "name": "python",
      "description": "Run Python in a persistent sandbox session and get back whatever the code prints. The session keeps its variables between calls. The puzzle's data is already loaded in it.",
      "parameters": {
        "type": "object",
        "properties": {
          "code": {
            "type": "string",
            "description": "Python source to execute. The value of a trailing expression is echoed, like a notebook cell."
          }
        },
        "required": [
          "code"
        ]
      }
    }
  },
  "tool_name": "python",
  "system_prompt_template": "You are participating in a puzzle solving competition. You are an expert at solving puzzles.\n\nYou have a stateful Python sandbox. The puzzle's data is already loaded in it, so you never need to retype the grids:\n\n    train_pairs   list of {{\"input\": grid, \"output\": grid}} -- the same training pairs shown in the question\n    test_input    grid -- the same test input shown in the question\n    np            numpy, already imported\n\nA grid is a list of lists of integers 0-9. Variables you define persist across calls, so you can build up your analysis over several calls. The sandbox has no network and cannot write files; it is for computation only. Each call is cut off after {tool_timeout:g} seconds and its output is truncated at {tool_max_output_chars} characters, so print summaries rather than dumping large grids.\n\nUse the sandbox to check your reasoning rather than to replace it. A hypothesis is worth running when you can state it as code: write the transformation as a function, apply it to every training input, and compare against the expected training output. A mismatch tells you exactly where the hypothesis is wrong.\n\nYou have at most {max_turns} tool-calling turns. When you are ready, stop calling the tool and reply with the final output grid for the test input, in the same JSON format as the training outputs. Your final reply must contain the answer grid itself -- what the sandbox printed is not read by anyone.",
  "token_storage": "append-only: each turn concatenates the teacher's own output_ids; `ids` is the exact flat sequence the teacher was conditioned on. Train on span_role==1 (assistant).",
  "span_role_legend": {
    "0": "prompt",
    "1": "assistant",
    "2": "tool_output"
  }
}


In [ ]:
%%writefile /tmp/arcforge_qwen38fn_nvfp4/work/agentic_tir/prompts.py
"""Agentic-TIR 的協定：explorer / refiner 的 system prompt、user prompt 渲染、報告解析。

與 `arcforge.tir.protocol`（學生模型受訓的凍結協定）**無關**——這條臂跑的是通用模型
（Qwen3.8-Flash-Next），prompt 是這個 unit 自己的設計，可以改。

設計要點
--------
* **一題所有 test input 一起給**：sandbox 裡是 `test_inputs`（list），沒有 `test_input`。
  規則要同時對每個 test input 都產生合理輸出，這是聯合求解的核心訊號。
* **統一的最終報告格式**（explorer 與 refiner 都用）：RULE / FINDINGS / CODE / CONFIDENCE /
  ANSWER 六節（v6 加 NEXT）。explorer 正常結束就是一份報告；預算用完被迫停手也是同一份報告，
  差別只在 ANSWER 可能是空的。refiner 的輸入就是這些報告。
* **ANSWER 的每個 attempt 可以是字面 JSON grid，也可以是一句 sandbox 表達式**
  （`transform(test_inputs[0])`、`candidates[0]["A"]`）。大 grid 用表達式省 token、
  也避免抄寫錯；harness 在 sandbox 還活著時把表達式算成 grid。
* 解析採「寬鬆抽取、嚴格驗證」：標頭容許 `#`/`##`/`###`/粗體，grid 必須是矩形、
  值 0-9，否則當沒有。
* **grid 在 prompt 裡是「一列一行、每格一個數字、不加分隔」**（`_grid_text`）：Qwen tokenizer 下
  1.05 tok/cell，compact JSON 是 2.0、官方 `[[1, 1], ...]` 是 3.0。模型不需要從 prompt 抄 grid
  （sandbox 裡有 `train_pairs` / `test_inputs`），所以可以放棄 Python literal。
* **狀態放在 sandbox，不放在腦裡**：v1 的交接報告（thinking 關、不准呼叫工具）RULE/FINDINGS 寫得好，
  但 CODE 是沒跑過的、ANSWER 是手打的 grid（132/185 份交接 CODE 連 train pair 都過不了）。
  v2 要求 `transform` 隨時在 sandbox 裡保持最新、CODE 必須是跑過的那份、ANSWER 優先用表達式；
  交接時保留 thinking、允許少量 tool call 做 checkpoint。
* **一條 explorer 可以跨多個 session（v4）**：預算用完的交接報告不是只給 refiner 看，
  同一條 explorer 會在**同一個 sandbox** 上開一個全新的 context 接著做（`render_continue_user`）：
  prompt = 題目 + 上一個 session 的報告 + harness 剛跑的 `transform` 檢查 + sandbox 裡的 `notes`。
  這是把 baseline 那種 40–100k 的連續思考切成幾個 50k 視窗，狀態靠 sandbox 與報告帶過去。
"""

from __future__ import annotations

import json
import re

REPORT_SECTIONS = ("RULE", "FINDINGS", "NEXT", "CODE", "ALTERNATIVE", "CONFIDENCE", "ANSWER")   # ALTERNATIVE 只有 alt 變體寫
VARIANTS = ("default", "alt")   # --prompt-variant；default 的每一個 prompt 字串一字不動

# ---------------------------------------------------------------------------
# System prompts
# ---------------------------------------------------------------------------

_SANDBOX_COMMON = """\
You have a stateful Python sandbox (the `python` tool). The puzzle's data is already loaded, so you never need to retype the grids:

    train_pairs   list of {{"input": grid, "output": grid}} -- the training pairs shown in the question
    test_inputs   list of grids -- ALL {n_test} test input(s) of this puzzle, in order ({test_ref})
    np            numpy, already imported
{extra_names}
A grid is a list of lists of integers 0-9 (in the question, grids are printed one row per line, one digit per cell, no separators -- the same grids are already in the sandbox, don't retype them). Variables you define persist across calls. The sandbox has no network and cannot write files; it is for computation only. Each call is cut off after {tool_timeout:g} seconds and its output is truncated at {tool_max_output_chars} characters, so print summaries rather than dumping large grids.
"""

_METHOD = """\
Keep your state in the sandbox, not in your head. As soon as you can state a hypothesis as code, write it as `def transform(grid): ...` in the sandbox and run it on every training input, comparing against the expected training output -- a mismatch tells you exactly where the hypothesis is wrong. `transform` should always hold your latest version, and its result on the training pairs (from the tool output, not from memory) is the only evidence that counts. `notes` (a Python list of short strings) is your ledger: every time you verify a fact or a hypothesis fails, `notes.append(...)` one line saying what and where (which training pair, which cells). The ledger is carried verbatim to whoever continues this work -- you in a later session, or a colleague -- while your reasoning is not, so anything that is only in your head is lost. Look at what `transform` does to every test input too: the same rule has to produce a sensible output for all of them, and a test input that your rule cannot handle usually means the rule is incomplete, not that the test is special.
"""

_BUDGET = """\
Working budget: this whole conversation (your reasoning, your replies and the tool outputs) must fit in about {budget_k}k tokens. After each tool result you are told how much is left. Plan for it: spend the budget on testing hypotheses in code, not on hand-simulating grids in your head. A single reply is also limited to about {turn_k}k tokens: think in short bursts and land each one as a tool call (run the check, update `transform` / `notes`) rather than reasoning for a long time without touching the sandbox -- if a reply is cut off you will be asked to checkpoint and continue. When the total budget is nearly gone you will be asked to stop and write the report below, whatever state you are in -- but do not wait for that: as soon as you have a rule that passes every training pair and gives outputs for every test input, write the report.
"""

_REPORT_FORMAT = """\
FINAL REPORT FORMAT -- your last reply (the one with no tool call) must be exactly these six sections, in this order, with these headers:

## RULE
One to three sentences stating the transformation, or your best partial understanding of it.

## FINDINGS
The current picture, synthesised: what is established, what has been ruled out, and what is still uncertain -- at most about 10 bullets. Do not re-list the ledger (`notes` is delivered alongside this report); do not repeat the puzzle.

## NEXT
If the puzzle is not solved: the single most promising direction you have NOT yet tried, and the first concrete check to run for it (which pair, what to look at). This is the first thing whoever continues will do, so make it specific. If you are done, write `none`.

## CODE
One ```python fenced block defining `def transform(grid)` (input grid -> output grid, list of lists): the exact code you last ran in the sandbox, even if it does not pass every training pair, followed by one line stating which training pairs it passes according to the tool output. Do not write code here that you have not run. If you have no code yet, write `none`.

## CONFIDENCE
An integer 0-100: your probability that your answer for every test input is exactly right.

## ANSWER
{answer_instructions}
"""

_EXPLORER_ANSWER = """\
One block per test input, in order:

=== TEST 1 ===
<the output grid>
=== TEST 2 ===
<the output grid>

Each <grid> is normally a single Python expression that evaluates to the grid in your sandbox, e.g. `transform(test_inputs[0])` -- the harness evaluates it for you, so it is exact and costs no tokens. A literal grid (JSON like [[0,1],[2,3]], or one row per line of digits) is accepted only when it is small and you have already produced and checked it in the sandbox; never type out a grid from memory. If you genuinely have no answer for a test input, write `none` for it.
"""

_REFINER_ANSWER = """\
For each test input give ATTEMPT 1 (your best answer) and, when a second interpretation is genuinely plausible, ATTEMPT 2 (a different grid). Scoring counts a test input as solved if either attempt is exactly right, so a distinct second attempt costs nothing -- but a duplicate is wasted, and a random one is worth nothing. Format:

=== TEST 1 ATTEMPT 1 ===
<grid>
=== TEST 1 ATTEMPT 2 ===
<grid>
=== TEST 2 ATTEMPT 1 ===
<grid>

Each <grid> is normally a single Python expression that evaluates to the grid in your sandbox, e.g. `transform(test_inputs[0])`, `transform_2(test_inputs[1])` or `candidates[0]["A"]` -- the harness evaluates it for you, so it is exact and costs no tokens. A literal grid (JSON like [[0,1],[2,3]], or one row per line of digits) is accepted only when it is small and you have already produced and checked it in the sandbox; never type out a grid from memory.
"""

_EXPLORER_HEAD = (
    "You are an expert puzzle solver working on an ARC-style grid puzzle. Your job in this "
    "session is to find the transformation rule, verify it in code against the training "
    "pairs, and apply it to every test input.\n\n"
)

_REFINER_HEAD = (
    "You are an expert puzzle solver working on an ARC-style grid puzzle. Several colleagues "
    "(explorers) have already worked on this puzzle independently, each within a limited "
    "budget, and handed you their reports and ledgers. Your job is to consolidate their work, settle "
    "the disagreements, verify in code, and produce the final answer for every test input.\n\n"
)

_REFINER_HOWTO = """\
How to use the reports: the explorers' `transform` functions are already defined in your sandbox as `transform_1`, `transform_2`, ... (only those that compiled), and the harness has already run each one on every training pair and every test input -- the check table in the question is trustworthy, you don't need to redo it. Distinct predicted test outputs are collected in `candidates` (a list, one dict per test input, keyed "A", "B", ...). Start from what agrees, examine where the candidates differ cell by cell (`np.argwhere(np.array(a) != np.array(b))`), and decide which interpretation the training pairs actually support. A high-confidence explorer can still be wrong; a rule that passes every training pair AND makes sense on every test input beats one that only fits the training pairs. If no explorer got it, use their failed hypotheses to avoid repeating them and attack the puzzle yourself.
"""

EXPLORER_SYSTEM = _EXPLORER_HEAD + _SANDBOX_COMMON + "\n" + _METHOD + "\n" + _BUDGET + "\n" + _REPORT_FORMAT

REFINER_SYSTEM = (_REFINER_HEAD + _SANDBOX_COMMON + "\n" + _METHOD + "\n" + _REFINER_HOWTO
                  + "\n" + _BUDGET + "\n" + _REPORT_FORMAT)

# --prompt-variant alt（v8 分析：錯的 19 pairs 裡 14 個有全過 train 的 CODE，其中 12 個的正解沒被任何 session 產生過；
# 「一致且全過」的 102 pairs 有 72 個第二 attempt 是空的，refiner 自造的 ATTEMPT 2 36 個只對 1 個）：
# explorer 全過 train 後花一步有上限的預算找「也過 train、但 test 輸出不同」的反假設 `transform_alt`，
# 寫進 ALTERNATIVE 與 ATTEMPT 2；refiner 在 candidates 全是 train-pass 時刻意找反例，ATTEMPT 2 留給它。
_ALT_STEP = """\
Before writing the report, when `transform` passes every training pair, spend ONE bounded step on a counter-hypothesis: at most 2 tool calls and about 3k tokens, and skip the step entirely if the last budget note shows less than 8k tokens left. A counter-hypothesis is a rule that ALSO passes every training pair but gives a different output on at least one test input -- usually a different reading of a detail the training pairs never pin down (tie-breaking, direction or orientation, what counts as an object, which colour is background, a case that only the test inputs exercise). Write it as `def transform_alt(grid)` in the sandbox and run it on the training pairs and the test inputs. Keep it only if it passes every training pair AND differs from `transform` on some test input; otherwise discard it. Do not revise `transform` during this step and do not spend a third tool call on it: report what you have.
"""

_ALTERNATIVE_SECTION = """\
## ALTERNATIVE
Only if the counter-hypothesis step produced a `transform_alt` that passes every training pair and differs from `transform` on a test input: one or two sentences saying how its reading differs and why the training pairs cannot decide between the two, then one ```python fenced block defining `def transform_alt(grid)` (the exact code you ran), then one line with its training result and which test inputs it differs on. Otherwise write `none`.

"""

_REPORT_FORMAT_ALT = (_REPORT_FORMAT.replace("exactly these six sections", "these sections")
                      .replace("## CONFIDENCE\n", _ALTERNATIVE_SECTION + "## CONFIDENCE\n"))
assert _REPORT_FORMAT_ALT.count("## ALTERNATIVE") == 1 and "six sections" not in _REPORT_FORMAT_ALT

_EXPLORER_ANSWER_ALT = """\
One block per test input, in order; ATTEMPT 1 is the output of `transform`:

=== TEST 1 ATTEMPT 1 ===
<the output grid>
=== TEST 1 ATTEMPT 2 ===
<the output grid of `transform_alt`>
=== TEST 2 ATTEMPT 1 ===
<the output grid>

Each <grid> is normally a single Python expression that evaluates to the grid in your sandbox, e.g. `transform(test_inputs[0])` -- the harness evaluates it for you, so it is exact and costs no tokens. A literal grid (JSON like [[0,1],[2,3]], or one row per line of digits) is accepted only when it is small and you have already produced and checked it in the sandbox; never type out a grid from memory. If you genuinely have no answer for a test input, write `none` for it. Give ATTEMPT 2 for a test input only when ALTERNATIVE is not `none` and `transform_alt` differs from `transform` on that test input; it must be `transform_alt(test_inputs[i])`, never a hand-edited copy of attempt 1.
"""

_REFINER_ALT_STEP = """\
Agreement is weaker evidence than it looks: passing every training pair does not identify the rule, and when every explorer read the same detail the same way, their agreement says nothing about that detail. Whenever every candidate for a test input comes from code that passes all training pairs -- and especially when there is only one candidate -- spend ONE bounded step (at most 3 tool calls, about 4k tokens) on a counter-hypothesis before answering: name the detail the training pairs never pin down (tie-breaking, orientation, what counts as an object, background, a case only the test inputs exercise), write it as `def transform_alt(grid)`, and keep it only if it passes every training pair and differs from the consensus on some test input. Do not re-derive or re-implement a rule the explorers already verified -- the check table is the harness's own run, not their claim. ATTEMPT 1 MUST stay the consensus answer (candidate A, or the candidate you would have chosen without this step) unless a training pair refutes it -- never put the counter-hypothesis in ATTEMPT 1, no matter how much you like it: the consensus has the better base rate. The counter-hypothesis goes in ATTEMPT 2 only, as `transform_alt(test_inputs[i])`.
"""

_REFINER_ANSWER_ALT = """\
For each test input give ATTEMPT 1 (your best answer) and ATTEMPT 2 (the output of the most plausible alternative reading: a rule that also passes every training pair but differs from ATTEMPT 1 on this test input -- normally `transform_alt(test_inputs[i])`, or an explorer's ALTERNATIVE / a rival candidate if that is the strongest alternative). Scoring counts a test input as solved if either attempt is exactly right, so ATTEMPT 2 is free -- but it only helps if it is a different rule's output: a copy of ATTEMPT 1, a hand-edited variant of it, or a grid that no rule produced is worth nothing. Omit ATTEMPT 2 only when you found no alternative that passes every training pair. Never swap the two: ATTEMPT 1 is the consensus reading, ATTEMPT 2 is the alternative. Format:

=== TEST 1 ATTEMPT 1 ===
<grid>
=== TEST 1 ATTEMPT 2 ===
<grid>
=== TEST 2 ATTEMPT 1 ===
<grid>

Each <grid> is normally a single Python expression that evaluates to the grid in your sandbox, e.g. `transform(test_inputs[0])`, `transform_alt(test_inputs[0])`, `transform_2(test_inputs[1])` or `candidates[0]["A"]` -- the harness evaluates it for you, so it is exact and costs no tokens. A literal grid (JSON like [[0,1],[2,3]], or one row per line of digits) is accepted only when it is small and you have already produced and checked it in the sandbox; never type out a grid from memory.
"""

EXPLORER_SYSTEM_ALT = (_EXPLORER_HEAD + _SANDBOX_COMMON + "\n" + _METHOD + "\n" + _BUDGET + "\n" + _ALT_STEP
                       + "\n" + _REPORT_FORMAT_ALT)

REFINER_SYSTEM_ALT = (_REFINER_HEAD + _SANDBOX_COMMON + "\n" + _METHOD + "\n" + _REFINER_HOWTO + "\n" + _REFINER_ALT_STEP
                      + "\n" + _BUDGET + "\n" + _REPORT_FORMAT)

HANDOFF_USER = """\
STOP -- your working budget is used up. You may make at most {n_calls} more tool call(s), only to checkpoint: make sure `transform` in the sandbox is your latest version and run it on the training pairs (and on the test inputs) so you can report real results, and bring the `notes` ledger up to date. Then write the FINAL REPORT, in exactly the five-section format from the instructions (RULE / FINDINGS / NEXT / CODE / CONFIDENCE / ANSWER), so that whoever continues from where you are (you in a fresh session with this same sandbox, or a colleague) does not have to redo it. CODE must be the code that is actually in the sandbox; in ANSWER use sandbox expressions (`transform(test_inputs[0])`) or `none` -- do not type grids by hand.
"""

NOTE_TAGS = ("RULE", "VERIFIED", "REFUTED", "TODO", "CODE")
_NOTE_TAG_RE = re.compile(r"^\s*(RULE|VERIFIED|REFUTED|TODO|CODE)\s*:\s*", re.IGNORECASE)

# --notes-structured：`notes` 仍是 list（模型端 API 不變），但每條要帶標籤；harness 渲染時按標籤分組、去重、RULE 只留最新。
# OpenHands 的 condenser 摘要（COMPLETED / PENDING / CURRENT_STATE / CODE_STATE / TESTS）、Cursor 的自我摘要、ARC Prize 的
# refinement loop 帶的都是結構化狀態；v3 自由 append 的 notes 會重複、無序。
NOTES_PROTOCOL = """

`notes` protocol: every entry you append starts with one of these tags, and says one thing:
  RULE: <your current best statement of the transformation> (append a new one whenever it changes; only the newest counts)
  VERIFIED: <a fact you confirmed by running code, with the evidence in a few words>
  REFUTED: <a hypothesis you tested and ruled out, and what killed it>
  TODO: <the next concrete thing to check, in order of promise>
  CODE: <what `transform` (or a helper) currently does / its known failure cases>
Keep entries short and never duplicate an existing one. The harness shows `notes` grouped by tag (newest RULE only) \
whenever earlier rounds are removed from view or another session takes over, so this is your durable memory."""


def render_notes(notes: list[str], structured: bool) -> str:
    """把 notes 渲染成給 prompt 的文字。structured：按標籤分組、去重、RULE 只留最新；沒標籤的放 OTHER。"""
    if not notes:
        return "(empty)"
    if not structured:
        return "\n".join(f"- {x}" for x in notes)
    groups: dict[str, list[str]] = {tag: [] for tag in NOTE_TAGS}
    groups["OTHER"] = []
    for x in notes:
        m = _NOTE_TAG_RE.match(x)
        tag, body = (m.group(1).upper(), x[m.end():].strip()) if m else ("OTHER", x.strip())
        if body and body not in groups[tag]:
            groups[tag].append(body)
    if groups["RULE"]:
        groups["RULE"] = groups["RULE"][-1:]
    out = []
    for tag in (*NOTE_TAGS, "OTHER"):
        if groups[tag]:
            out.append(f"{tag}:\n" + "\n".join(f"  - {b}" for b in groups[tag]))
    return "\n".join(out) if out else "(empty)"


# --delegate：explorer 可以把一個自足的子問題交給乾淨 context 的 helper（同一個 sandbox），只拿回短結論。
# arXiv 2606.29718 的七法對照裡 sub-agent 隔離（54.0%）明顯高於各種摘要（≤48.2%）；Anthropic 的建議是 subagent 只回 1–2k token。
DELEGATE_TOOL = {
    "type": "function",
    "function": {
        "name": "delegate",
        "description": (
            "Hand one self-contained sub-question to a helper that starts with a CLEAN context but shares your sandbox "
            "(every function and variable you defined is available to it, and whatever it defines stays for you). "
            "It works on its own budget and returns only a short conclusion, so its exploration never uses your context. "
            "Good uses: an exhaustive scan or table over all pairs, testing one alternative hypothesis end to end, "
            "debugging why `transform` fails on a specific pair, or writing and validating a helper function. "
            "State precisely what to test, what is already known (or point it at `notes`), and what to return."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "question": {"type": "string", "description": "the self-contained task for the helper, incl. what to return"},
            },
            "required": ["question"],
        },
    },
}

DELEGATE_SYSTEM_NOTE = """

Delegation: you also have a `delegate` tool (see its description). Use it when a sub-investigation would flood your \
context with output or would take many rounds -- the helper does the work in its own context and you get back only \
the conclusion. Keep the question self-contained; the helper knows the puzzle and the sandbox but not your reasoning."""

HELPER_SYSTEM = (
    """You are a helper spun up by the main solver of an ARC puzzle. The main solver has been working in a Python sandbox \
and hands you ONE self-contained sub-question. You share that sandbox: everything it defined (listed in the question) is \
available to you, and anything you define stays available to it. You do not see its reasoning -- only the question, \
the puzzle and `notes`.

"""
    + _SANDBOX_COMMON + "\n" + """\
Do exactly what the question asks, with code, and verify with code. Append tagged entries to `notes` (VERIFIED: / \
REFUTED: / CODE:) for anything you establish. Do not redefine `transform` unless the question asks you to; if you write \
a candidate, name it `transform_alt` or as the question says.

Working budget: about {budget_k}k generated tokens; after each tool result you are told how much is left. \
When you are done (or the budget is nearly gone), reply WITHOUT a tool call, in at most ~1500 tokens, with exactly:

## VERDICT
One to three sentences answering the question.
## EVIDENCE
The concrete results (numbers, which pairs, what failed) -- compact, no grids unless tiny.
## DEFINED
Names you defined in the sandbox and one line each on what they do (or `none`).
## SUGGESTION
What the main solver should do next given this (one or two sentences)."""
)

HELPER_HANDOFF = """\
STOP -- your budget is used up. No more tool calls: write your reply now in the VERDICT / EVIDENCE / DEFINED / \
SUGGESTION format with what you have."""


def helper_system(n_test: int, budget_tokens: int, tool_timeout: float, tool_max_output_chars: int) -> str:
    return HELPER_SYSTEM.format(
        n_test=n_test, test_ref=_test_ref(n_test),
        extra_names="    notes         the main solver's ledger (list of tagged strings) -- read it, append to it\n",
        tool_timeout=tool_timeout, tool_max_output_chars=tool_max_output_chars, budget_k=budget_tokens // 1000,
    )


def render_helper_user(train_pairs, test_inputs, names: str, notes: list[str], structured: bool, question: str) -> str:
    return (render_task(train_pairs, test_inputs)
            + "\n\nSandbox: functions already defined by the main solver: " + (names or "(none)")
            + "\n`notes` so far:\n" + render_notes(notes, structured)
            + "\n\nYOUR QUESTION from the main solver:\n" + question.strip()
            + "\n\nWork it through in the sandbox, then reply in the VERDICT / EVIDENCE / DEFINED / SUGGESTION format.")


COMPACT_SYSTEM_NOTE = """

Context management: when this conversation gets close to the context limit, the harness removes your oldest tool \
rounds from view (a note in the task message tells you what was removed and what the sandbox holds) and you simply \
continue in the same conversation -- you do not write a report at that point. The sandbox is never reset: every \
function and variable you defined stays available. Keep `notes` and `transform` up to date so nothing is lost when \
old rounds disappear. The working budget above is the total you may generate in this conversation."""

CHECKPOINT_USER = """\
[harness] The context is about to be compacted: everything before your last round (about {n_rounds} tool rounds) will \
disappear from view; the sandbox keeps all names, but the reasoning in those rounds will be gone. Make ONE short tool \
call now that appends to `notes` the things from those rounds you would otherwise lose -- verified facts, refuted \
hypotheses, the current best rule and where it fails, and what you were about to try (a few compact lines; do not \
re-list what is already in `notes`). Then continue exactly where you left off."""

CHECKPOINT_USER_STRUCTURED = """\
[harness] The context is about to be compacted: everything before your last round (about {n_rounds} tool rounds) will \
disappear from view; the sandbox keeps all names, but the reasoning in those rounds will be gone. Make ONE short tool \
call now that appends to `notes` what you would otherwise lose, one tagged entry per line (RULE: / VERIFIED: / REFUTED: / \
TODO: / CODE:), skipping anything already in `notes`. Then continue exactly where you left off."""

CONTINUE_USER = """\
Your reply was cut off by the per-reply limit (the total budget is not used up: about {left_k}k tokens remain). Checkpoint first -- one tool call that writes what you have established so far into the sandbox (update `transform`, append to `notes`) -- then continue from where you were.
"""


def explorer_system(n_test: int, budget_tokens: int, turn_tokens: int, tool_timeout: float, tool_max_output_chars: int,
                    variant: str = "default", notes_structured: bool = False) -> str:
    assert variant in VARIANTS, variant
    system, answer = (EXPLORER_SYSTEM, _EXPLORER_ANSWER) if variant == "default" else (EXPLORER_SYSTEM_ALT, _EXPLORER_ANSWER_ALT)
    if notes_structured:
        system += NOTES_PROTOCOL
    return system.format(
        n_test=n_test, test_ref=_test_ref(n_test),
        extra_names="    notes         empty list for your own verified facts / failed hypotheses\n",
        tool_timeout=tool_timeout, tool_max_output_chars=tool_max_output_chars,
        budget_k=budget_tokens // 1000, turn_k=turn_tokens // 1000, answer_instructions=answer,
    )


def refiner_system(n_test: int, budget_tokens: int, turn_tokens: int, tool_timeout: float, tool_max_output_chars: int,
                   transform_names: list[str], variant: str = "default", notes_structured: bool = False) -> str:
    assert variant in VARIANTS, variant
    system, answer = (REFINER_SYSTEM, _REFINER_ANSWER) if variant == "default" else (REFINER_SYSTEM_ALT, _REFINER_ANSWER_ALT)
    if notes_structured:
        system += NOTES_PROTOCOL
    extra = ""
    if transform_names:
        extra += "    " + ", ".join(transform_names) + "   the explorers' transform functions (see the question)\n"
    extra += "    candidates    list (one dict per test input) of the distinct predicted test outputs, keyed \"A\", \"B\", ...\n"
    extra += "    notes         empty list for your own verified facts / failed hypotheses\n"
    return system.format(
        n_test=n_test, test_ref=_test_ref(n_test), extra_names=extra,
        tool_timeout=tool_timeout, tool_max_output_chars=tool_max_output_chars,
        budget_k=budget_tokens // 1000, turn_k=turn_tokens // 1000, answer_instructions=answer,
    )


def _test_ref(n_test: int) -> str:
    return "test_inputs[0]" if n_test == 1 else f"test_inputs[0] .. test_inputs[{n_test - 1}]"


# ---------------------------------------------------------------------------
# User prompts
# ---------------------------------------------------------------------------

def _grid_json(g) -> str:
    return json.dumps(g, separators=(",", ":"))


def _grid_text(g) -> str:
    """一列一行、每格一個數字、不加分隔。"""
    return "\n".join("".join(str(v) for v in row) for row in g)


def render_task(train_pairs: list[dict], test_inputs: list) -> str:
    out = ["Below are the training pairs of the puzzle. Find the transformation that maps each "
           "input to its output, then apply it to every test input. Grids are printed one row per "
           "line, one digit (0-9) per cell.\n", "--Training Examples--"]
    for i, p in enumerate(train_pairs):
        gi, go = p["input"], p["output"]
        out.append(f"--Example {i + 1}--\nINPUT ({len(gi)}x{len(gi[0])}):\n{_grid_text(gi)}\n"
                   f"OUTPUT ({len(go)}x{len(go[0])}):\n{_grid_text(go)}\n")
    out.append("--End of Training Examples--\n")
    out.append(f"--Test Inputs-- ({len(test_inputs)} test input(s); solve them together, they share one rule)")
    for i, g in enumerate(test_inputs):
        out.append(f"--Test {i + 1}-- (test_inputs[{i}], {len(g)}x{len(g[0])}):\n{_grid_text(g)}\n")
    out.append("--End of Test Inputs--")
    return "\n".join(out)


def render_explorer_user(train_pairs, test_inputs) -> str:
    return render_task(train_pairs, test_inputs) + "\n\nWork through the puzzle with the sandbox, then write the final report."


def render_continue_user(train_pairs, test_inputs, session_no: int, max_sessions: int, prev: dict) -> str:
    """同一條 explorer 的下一個 session。`prev`：上一個 session 的解析後報告（parse_report 欄位）+
    'stop_reason'、'check'（harness 剛在這個 sandbox 跑 `transform` 的 check dict，見 agentic_tir._check_transform）、
    'notes'（sandbox 裡 `notes` 的內容）。"""
    parts = [render_task(train_pairs, test_inputs), "", "=" * 60,
             f"YOUR PREVIOUS SESSION ON THIS PUZZLE (session {session_no - 1} of at most {max_sessions}; "
             f"it ended: {prev['stop_reason']})", "=" * 60,
             "You already worked on this puzzle in a previous session whose working budget ran out. Its sandbox "
             "is the one you have now: `transform`, `notes`, and every helper function or variable you defined are "
             "still there (use dir() if unsure). This is what that session reported:", "",
             "RULE: " + (prev["rule"].strip() or "(none)"),
             "FINDINGS:\n" + (prev["findings"].strip() or "(none)"),
             "NEXT (what that session planned to do next): " + (prev["next"].strip() or "(none)")]
    chk = prev["check"]
    if prev["code"] and chk["defined"]:
        parts.append(f"CODE (this is `transform` in your sandbox now):\n```python\n{prev['code'].strip()}\n```")
    elif prev["code"]:
        parts.append(f"CODE (reported, but it does not load in the sandbox -- {chk['summary']}):\n"
                     f"```python\n{prev['code'].strip()}\n```")
    else:
        parts.append("CODE: none reported" + ("" if not chk["defined"] else " (but a `transform` is defined in the sandbox)"))
    if chk["defined"]:
        parts.append("Harness check of `transform`, run just now: " + chk["summary"])
    conf = prev["confidence"]
    parts.append("CONFIDENCE: " + ("not stated" if conf is None else str(conf)))
    notes = prev["notes"]
    if notes:
        parts.append("`notes` (the ledger, verbatim from the sandbox):\n" + render_notes(notes, prev.get("notes_structured", False)))
    else:
        parts.append("`notes` (the ledger): empty -- the previous session did not keep one")
    parts += ["", f"This is session {session_no} of at most {max_sessions}, with the same working budget as before. "
              "Continue from where you left off: do not redo what FINDINGS already established and do not retry "
              "hypotheses that already failed; start with NEXT unless the ledger shows it is already done. Work through the puzzle "
              "with the sandbox, then write the final report -- it must stand on its own, so carry forward the "
              "still-relevant findings from the previous session."]
    return "\n".join(parts)


def render_refiner_user(train_pairs, test_inputs, reports: list[dict], checks: list[dict],
                        candidates: list[dict], inline_cells: int, variant: str = "default") -> str:
    """`reports[k]`：explorer k 的解析後報告（見 parse_report）+ 'label'/'stop_reason'，alt 變體另有 'alt_check'
    （harness 在它的 sandbox 檢查 `transform_alt` 的結果；refiner 自己的報告是 None）。
    `checks[k]`：harness 跑 transform_k 的結果（見 agentic_tir.verify_reports）。
    `candidates[t]`：{"A": {"grid":..., "sources": [...]}, ...}。"""
    assert variant in VARIANTS, variant
    parts = [render_task(train_pairs, test_inputs), "", "=" * 60, f"EXPLORER REPORTS ({len(reports)})", "=" * 60]
    for k, (rep, chk) in enumerate(zip(reports, checks), start=1):
        conf = rep["confidence"]
        conf_s = "not stated" if conf is None else str(conf)
        parts.append(f"\n### Explorer {k} -- {rep['label']} (ended: {rep['stop_reason']}; stated confidence: {conf_s})")
        parts.append("RULE: " + (rep["rule"].strip() or "(none)"))
        parts.append("FINDINGS:\n" + (rep["findings"].strip() or "(none)"))
        parts.append("NEXT (what it would have tried next): " + (rep["next"].strip() or "(none)"))
        notes = rep["notes"]
        parts.append("NOTES (its ledger, verbatim):\n" + render_notes(notes, rep.get("notes_structured", False)))
        if rep["code"]:
            where = (f"loaded in your sandbox as `transform_{k}`" if chk["defined"] else
                     "reported; it does NOT load in your sandbox -- its helpers lived in the explorer's sandbox")
            parts.append(f"CODE ({where}):\n```python\n{rep['code'].strip()}\n```")
            parts.append("Harness check of this code: " + chk["summary"])
        elif rep["home_check"]["defined"]:
            parts.append("CODE: none reported, but a `transform` existed in its sandbox -- harness check: " + chk["summary"])
        else:
            parts.append("CODE: none")
        if variant == "alt":
            alt_chk = rep["alt_check"]
            alt_txt = rep["alt"] or "none"
            if alt_chk is not None and alt_chk["defined"]:
                parts.append(f"ALTERNATIVE (its counter-hypothesis; harness check of its `transform_alt`: {alt_chk['summary']}):\n{alt_txt}")
            else:
                parts.append("ALTERNATIVE: " + alt_txt)
        ans_bits = []
        for t, grids in enumerate(rep["answers"]):
            keys = [key for g in grids for key, c in candidates[t].items() if c["grid"] == g]
            ans_bits.append(f"test {t + 1}: " + (", ".join(f"candidate {k}" for k in keys) if keys else "none"))
        parts.append("Stated ANSWER -> " + "; ".join(ans_bits))
    parts += ["", "=" * 60, "CANDIDATE TEST OUTPUTS (distinct grids; `candidates[t][key]` in your sandbox; "
              "rows of digits as in the question)", "=" * 60]
    for t, cands in enumerate(candidates):
        parts.append(f"\nTest {t + 1} ({len(test_inputs[t])}x{len(test_inputs[t][0])} input):")
        if not cands:
            parts.append("  (no candidate -- nobody produced an output for this test input)")
        for key, c in cands.items():
            g = c["grid"]
            shape = f"{len(g)}x{len(g[0])}"
            srcs = ", ".join(c["sources"])
            line = f"  {key}: {shape}, from {srcs}"
            if len(g) * len(g[0]) <= inline_cells:
                line += "\n" + _grid_text(g)
            parts.append(line)
        keys = list(cands)
        for i in range(len(keys)):
            for j in range(i + 1, len(keys)):
                parts.append(f"  {keys[i]} vs {keys[j]}: " + diff_summary(cands[keys[i]]["grid"], cands[keys[j]]["grid"]))
    if variant == "alt" and all(_train_all_pass(c["train"]) for c in checks):
        # 全部 candidate 都來自全過 train 的 code：v8 這種「一致且全過」的 102 pairs 有 72 個第二 attempt 是空的，
        # 錯的兩題 explorer 與 refiner 都寫「no ambiguity」。明講 train 決定不了規則，逼它做反假設那一步。
        assert checks
        disagree = [str(t + 1) for t in range(len(test_inputs)) if len(candidates[t]) > 1]
        agree_s = ("the explorers agree on every test input" if not disagree
                   else "they disagree on test " + ", ".join(disagree))
        parts += ["", f"NOTE: every candidate above comes from code that passes all {len(train_pairs)} training pairs, "
                  f"and {agree_s}. The training pairs therefore do not decide the rule -- the agreement or the votes tell "
                  "you which reading the explorers defaulted to, not which one is right. Do the counter-hypothesis step "
                  "from your instructions before answering, and use ATTEMPT 2 for it."]
    parts += ["", "Consolidate, verify, and write the final report with ATTEMPT 1 (and ATTEMPT 2 where a second "
              "interpretation is plausible) for every test input."]
    return "\n".join(parts)


def _train_all_pass(train: list) -> bool:
    return bool(train) and all(r == "pass" for r in train)


def diff_summary(a, b, max_cells: int = 12) -> str:
    if (len(a), len(a[0])) != (len(b), len(b[0])):
        return f"different shapes {len(a)}x{len(a[0])} vs {len(b)}x{len(b[0])}"
    diffs = [(r, c, a[r][c], b[r][c]) for r in range(len(a)) for c in range(len(a[0])) if a[r][c] != b[r][c]]
    if not diffs:
        return "identical"
    shown = ", ".join(f"({r},{c}) {x}->{y}" for r, c, x, y in diffs[:max_cells])
    more = "" if len(diffs) <= max_cells else f", ... ({len(diffs) - max_cells} more)"
    return f"{len(diffs)} cells differ: {shown}{more}"


# ---------------------------------------------------------------------------
# Report parsing
# ---------------------------------------------------------------------------

_SECTION_RE = re.compile(
    r"^[ \t]*(?:#{1,4}[ \t]*|\*\*)?(RULE|FINDINGS|NEXT|CODE|ALTERNATIVE|CONFIDENCE|ANSWER)(?:\*\*)?[ \t]*:?[ \t]*$",
    re.IGNORECASE | re.MULTILINE,
)
_FENCE_RE = re.compile(r"```(?:python|py)?[ \t]*\n(.*?)```", re.DOTALL)
_ANSWER_HDR_RE = re.compile(
    r"^[ \t]*=+[ \t]*TEST[ \t]*(\d+)(?:[ \t]*ATTEMPT[ \t]*(\d+))?[ \t]*=+[ \t]*$",
    re.IGNORECASE | re.MULTILINE,
)


def split_sections(text: str) -> dict[str, str]:
    """五節標頭切段。同名重複時取最後一個（模型偶爾會先寫草稿再重寫）。"""
    hits = list(_SECTION_RE.finditer(text))
    out: dict[str, str] = {}
    for i, m in enumerate(hits):
        end = hits[i + 1].start() if i + 1 < len(hits) else len(text)
        out[m.group(1).upper()] = text[m.end():end]
    return out


def extract_transform_code(text: str, name: str = "transform") -> str | None:
    """最後一個定義 `def <name>(` 的 fenced block（`transform` 不會匹配到 `transform_alt`）。"""
    blocks = [b for b in _FENCE_RE.findall(text) if re.search(rf"^\s*def {name}\s*\(", b, re.MULTILINE)]
    return blocks[-1].rstrip() if blocks else None


def is_grid(g) -> bool:
    return (isinstance(g, list) and len(g) > 0 and all(isinstance(r, list) and len(r) == len(g[0]) and len(r) > 0 for r in g)
            and all(isinstance(v, int) and not isinstance(v, bool) and 0 <= v <= 9 for r in g for v in r))


def _first_balanced(text: str, start: int = 0):
    """從 start 起找第一個 `[`，回傳與之配對的 `[...]` 子字串（沒有就 None）。"""
    i = text.find("[", start)
    if i < 0:
        return None
    depth = 0
    for j in range(i, len(text)):
        ch = text[j]
        if ch == "[":
            depth += 1
        elif ch == "]":
            depth -= 1
            if depth == 0:
                return text[i:j + 1]
    return None


def parse_answer_blocks(text: str) -> dict[tuple[int, int], str]:
    """`=== TEST i [ATTEMPT j] ===` 標頭 → 該段落的原始內容（去掉空白）。key 是 0-based (test, attempt)。"""
    hits = list(_ANSWER_HDR_RE.finditer(text))
    out: dict[tuple[int, int], str] = {}
    for i, m in enumerate(hits):
        end = hits[i + 1].start() if i + 1 < len(hits) else len(text)
        t = int(m.group(1)) - 1
        a = int(m.group(2)) - 1 if m.group(2) else 0
        out[(t, a)] = text[m.end():end].strip()
    return out


def classify_answer_body(body: str) -> tuple[str, str | None]:
    """一段 ANSWER 內容 → ("grid", json) | ("expr", code) | ("none", None)。

    grid：字面 `[[...]]` 能 json.loads 且是合法 grid。expr：其餘非空、非 `none` 的單行文字
    （去掉反引號與 fenced 標記）。"""
    s = body.strip()
    if not s or s.lower().rstrip(".") in ("none", "n/a", "no answer", "(none)"):
        return "none", None
    rows = [ln.strip() for ln in s.splitlines() if ln.strip()]
    if rows and all(re.fullmatch(r"[0-9]+", r) for r in rows) and len({len(r) for r in rows}) == 1:
        return "grid", json.dumps([[int(ch) for ch in r] for r in rows], separators=(",", ":"))
    lit = _first_balanced(s)
    if lit is not None and s.lstrip().startswith("["):
        try:
            g = json.loads(lit)
        except json.JSONDecodeError:
            g = None
        if is_grid(g):
            return "grid", lit
    s = re.sub(r"^```(?:python|py)?\s*|\s*```$", "", s).strip()
    s = s.strip("`").strip()
    line = s.splitlines()[0].strip() if s else ""
    if not line or line.lower() == "none":
        return "none", None
    return "expr", line


def parse_report(text: str) -> dict:
    """把最終回覆解析成報告。缺的節給空字串/None，但一律回傳全部欄位。"""
    sec = split_sections(text)
    code = extract_transform_code(sec.get("CODE", "")) or extract_transform_code(text)
    alt = sec.get("ALTERNATIVE", "").strip()   # alt 變體的反假設；default 一律空字串
    conf = None
    m = re.search(r"\d{1,3}", sec.get("CONFIDENCE", ""))
    if m:
        conf = max(0, min(100, int(m.group(0))))
    answer_text = sec.get("ANSWER", "")
    blocks = parse_answer_blocks(answer_text) or parse_answer_blocks(text)
    return {
        "has_sections": bool(sec),
        "rule": sec.get("RULE", "").strip(),
        "findings": sec.get("FINDINGS", "").strip(),
        "next": sec.get("NEXT", "").strip(),
        "code": code,
        "alt": alt,
        "alt_code": extract_transform_code(alt, "transform_alt"),
        "confidence": conf,
        "answer_blocks": {f"{t}:{a}": classify_answer_body(b) for (t, a), b in blocks.items()},
        "answer_text": answer_text.strip(),
    }



In [ ]:
%%writefile /tmp/arcforge_qwen38fn_nvfp4/work/agentic_tir/agentic_tir.py
#!/usr/bin/env python
"""Agentic TIR：explorers（並行、各自 ≤ ctx cap）→ harness 驗證 → refiner（≤ ctx cap、每個 test 最多 2 attempts）。

    python work/agentic_tir/agentic_tir.py --task-dir ... --task-list ... --out $ARC_RUNS/agentic_tir/<tag> \
        --base-url http://127.0.0.1:30031/v1 --model qwen38fn --n-explorers 3 --ctx-cap 50000

一題的流程
----------
1. **explore**：K 條獨立 rollout，同一份 prompt（所有 test input 一起給），各自的 sandbox。
   每條的 context（prompt+completion，以 server 回的 total_tokens 計）壓在 `ctx_cap` 內：
   每輪 max_tokens = ctx_cap − report_reserve − last_total − margin。餘額不夠、或模型被截斷、
   或輪數用完 → 追加一則 user 訊息要求寫交接報告（handoff，預設關 thinking、tool_choice=none）。
   正常結束的最終回覆本身就是報告（RULE/FINDINGS/CODE/CONFIDENCE/ANSWER，見 prompts.py）。
2. **verify**：開 refine 用的 sandbox，把每份報告的 `transform` 定義成 `transform_k`，
   跑全部 train pair（pass/fail/error）與全部 test input（預測 grid）；連同 explorer 明講的
   ANSWER 一起去重成 candidates（A、B、…）。這是 refiner 看到的「事實表」。
3. **refine**：一條 rollout，prompt = 題目 + K 份報告 + 檢查表 + candidates；同一個 ctx cap；
   最終報告的 ANSWER 是每個 test 的 ATTEMPT 1（必要）/ ATTEMPT 2（可選）。refine 沒給答案的
   test 用 candidates 依票數補；`--fill-attempt2` 開著時 attempt 2 缺的也由 candidates 補
   （兩種來源都記在 attempt_src 裡，分析時分得開）。refine 用完預算沒答完 → 它的報告加進
   reports、再開一輪（≤ max_refine_rounds）。

計分口徑與 work/qwen38_eval/score_partial.py 相同：pairs/*.json 可直接餵它。
"""

from __future__ import annotations

import argparse
import asyncio
import copy
import gzip
import hashlib
import collections
import contextlib
import json
import os
import re
import sys
import time
from pathlib import Path

import httpx

sys.path.insert(0, str(Path(__file__).resolve().parent))
import prompts  # noqa: E402

from arcforge.tir.driver import Sampling, _chat  # noqa: E402
from arcforge.tir.protocol import load_config  # noqa: E402
from arcforge.tir.sandbox import Sandbox, SandboxError  # noqa: E402
from arcforge.tir.submission import SubmissionStore  # noqa: E402

GRID_MARK = "__GRID__"
ERR_MARK = "__ERR__"
DEF_ERR_MARK = "__DEF_ERR__"


# ---------------------------------------------------------------------------
# One episode (a single ≤ ctx_cap TIR conversation)
# ---------------------------------------------------------------------------

class Episode:
    """一條對話的紀錄。所有計量都來自 server 的 usage，不自己估。"""

    def __init__(self, label: str, lineage: int = 0, hop: int = 1):
        self.label = label
        self.lineage = lineage        # explorer 編號（refiner 為 0）
        self.hop = hop                # 同一條 explorer 的第幾個 session
        self.messages: list[dict] = []
        self.calls: list[dict] = []
        self.stop_reason = ""
        self.forced_handoff = False
        self.n_tool_calls = 0
        self.n_tool_errors = 0
        self.n_sandbox_restarts = 0    # 模型的 code 讓 sandbox 300s 沒回應、被砍掉重開的次數（狀態全失）
        self.n_continuations = 0
        self.n_compactions = 0             # --compact：harness 把最舊的 tool round 折掉、同一條對話續跑的次數
        self.compact_dropped_tokens = 0    # 累計被折掉的估計 token（模型看不到的部分）
        self.compact_dropped_rounds = 0
        self.compact_dropped_calls = 0
        self.n_checkpoints = 0             # --compact-checkpoint：折掉前的記帳輪數
        self.compact_history: list[str] = []   # --compact-rich：被折掉的每一輪留一行「試了什麼 → 結果」
        self.compact_transform_src = ""        # --compact-rich：被折掉的輪次裡最後一版 `def transform` 原始碼
        self.n_outputs_cleared = 0             # --compact-rich：被清掉的 tool 輸出數
        self.output_heads: dict[str, str] = {}  # 被清掉的 tool 輸出的開頭（給 _round_summary 用；不存檔）
        self.task_user = ""                # 第一則 user 訊息的原文（compaction 每次從它重組，不會疊加公告）
        self.task_id = ""                  # 只給 log 用
        self.notes_structured = False      # --notes-structured（給 _report_for_refiner 渲染用）
        self.sub_episodes: list[dict] = []  # --delegate：helper 對話（to_dict，含 messages）
        self.n_delegations = 0
        self.segments: list[list[dict]] = []  # --save-segments：每次 compaction 前的整份 messages（模型實際看到的視圖），最後一段是結束時的 messages
        self.final_content = ""
        self.final_reasoning = ""
        self.report: dict | None = None
        self.notes: list[str] = []          # session 結束時 sandbox 裡的 `notes`（帳本，逐字帶給接手者）
        self.home_check: dict | None = None  # session 結束時在自己的 sandbox 跑報告 CODE 的結果（helper 都在）
        self.alt_check: dict | None = None   # alt 變體：同上，但檢查的是反假設 `transform_alt`（refiner 沒有）
        self.code_src = ""
        self.answers: list[list] = []          # answers[t] = [grid, ...]（attempt 順序）
        self.answer_resolution: list[list[str]] = []
        self.t0 = time.monotonic()
        self.latency_s = 0.0

    def usage(self, key: str) -> int:
        return sum(c[key] for c in self.calls)

    @property
    def peak_total(self) -> int:
        return max((c["total_tokens"] for c in self.calls), default=0)

    def to_dict(self, keep_messages: bool) -> dict:
        d = {
            "label": self.label, "lineage": self.lineage, "hop": self.hop,
            "stop_reason": self.stop_reason, "forced_handoff": self.forced_handoff,
            "n_calls": len(self.calls), "n_tool_calls": self.n_tool_calls, "n_tool_errors": self.n_tool_errors,
            "n_sandbox_restarts": self.n_sandbox_restarts,
            "n_continuations": self.n_continuations,
            "n_compactions": self.n_compactions, "compact_dropped_tokens": self.compact_dropped_tokens,
            "n_checkpoints": self.n_checkpoints, "n_outputs_cleared": self.n_outputs_cleared,
            "compact_history": self.compact_history,
            "n_delegations": self.n_delegations, "sub_episodes": self.sub_episodes,
            "tok_prompt": self.usage("prompt_tokens"), "tok_completion": self.usage("completion_tokens"),
            "tok_reasoning": self.usage("reasoning_tokens"), "tok_cached": self.usage("cached_tokens"),
            "peak_total": self.peak_total, "latency_s": round(self.latency_s, 1),
            "report": self.report, "notes": self.notes, "home_check": self.home_check, "alt_check": self.alt_check,
            "code_src": self.code_src, "answers": self.answers,
            "answer_resolution": self.answer_resolution,
            "final_content": self.final_content, "final_reasoning": self.final_reasoning,
            "calls": self.calls,
        }
        if keep_messages:
            d["messages"] = [{k: v for k, v in m.items() if not k.startswith("_")} for m in self.messages]
        if self.segments:
            d["segments"] = self.segments   # write_task 會把它移到 segments/<task>.jsonl.gz
        return d


def _copy_private(asst: dict, msg: dict) -> None:
    """backend 掛在回應上的私有鍵（token backend 的 `_ids` / `_gen_len`）要跟著進 messages，segments 落地時才有 token。"""
    for k, v in msg.items():
        if k.startswith("_"):
            asst[k] = v


def _record_usage(ep: Episode, data: dict, finish: str, kind: str, max_tokens: int) -> int:
    u = data["usage"]
    details = u.get("prompt_tokens_details") or {}
    cdetails = u.get("completion_tokens_details") or {}
    rec = {
        "kind": kind, "finish": finish, "max_tokens": max_tokens,
        "prompt_tokens": int(u["prompt_tokens"]), "completion_tokens": int(u["completion_tokens"]),
        "total_tokens": int(u["total_tokens"]),
        "cached_tokens": int(details.get("cached_tokens") or 0),
        "reasoning_tokens": int(u.get("reasoning_tokens") or cdetails.get("reasoning_tokens") or 0),
        "segment": ep.n_compactions,   # 這次呼叫看到的是第幾段（--save-segments 的 segment_idx）
    }
    ep.calls.append(rec)
    return rec["total_tokens"]


def _base_payload(args, messages, tool, max_tokens: int) -> dict:
    p = {
        "model": args.model, "messages": messages, "tools": [tool, *args.extra_tools],
        "temperature": args.sampling.temperature, "top_p": args.sampling.top_p, "top_k": args.sampling.top_k,
        "max_tokens": max_tokens,
    }
    if args.reasoning_effort:
        p["reasoning_effort"] = args.reasoning_effort
    if args.dp_rank is not None:
        # sglang 的 DP 是 round-robin、沒有 session affinity：不釘 rank，每一輪都會在別的 replica 重 prefill
        # 整條 context（naive driver 是在 run_rollout 釘的，這裡之前沒有）。同一題的所有 episode 釘同一個
        # rank，system + 題目的前綴還能跨 explorer / refiner 命中 radix cache。
        p["routed_dp_rank"] = args.dp_rank
    return p


def _time_room(args) -> int:
    """deadline 之前這條 stream 還生得出多少 token（用保守的每條 decode 速率估）；沒有 deadline 就是無限。"""
    if args.deadline is None:
        return 10**9
    return int((args.deadline - time.monotonic()) * args.deadline_tok_per_s)


def _cutoff(args) -> str | None:
    """整個 run 層級的停止條件：呼叫端說沒救了（伺服器死掉）、或 deadline 到了——「到了」的定義是
    剩下的時間連一個最短回合都生不完。回傳原因字串；None 表示可以繼續。"""
    if args.should_abort is not None and args.should_abort():
        return "abort"
    if args.deadline is not None and _time_room(args) < args.min_turn_tokens:
        return "deadline"
    return None


SANDBOX_KILLED = "[sandbox killed: no reply within 300 s; restarted, all state lost]"

# `notes` 的鏡像：sandbox 的 RLIMIT_FSIZE=0 封鎖所有檔案寫入，所以鏡像由 harness 這端維護——每次 tool call 後問
# `len(notes)`，只把新增的條目逐條讀回（每條 ≤ NOTE_ENTRY_CHARS，不會撞 stdout 上限），harness 自己寫進 run 目錄；
# sandbox 重啟時用 stdin 灌回去。程式狀態（函式、變數）救不回來，但知識（notes）不該跟著行程一起死——
# compaction 之後 notes 是唯一跨越被折掉輪次的記憶。
NOTE_ENTRY_CHARS = 3000
_NOTES_LEN = "print(len(notes) if isinstance(notes, (list, tuple)) else -1)"
_NOTES_ITEM = "import json as _j; print(_j.dumps(str(notes[{i}])[:{n}]))"
_NOTES_LOAD = "import json as _j; notes = _j.loads({blob!r}); print(len(notes))"


async def sb_run(sb: Sandbox, code: str) -> tuple[str, bool]:
    """`sb.run` 加一層：子行程 300s 沒回應（SIGALRM 攔不住吞掉 KeyboardInterrupt 或卡在 C 呼叫裡的碼）就砍掉重開，
    回 (SANDBOX_KILLED, True) 當 timeout 處理。v7 的 da515329、v9 的 88e364bc 都因此整題報廢過——後者掛在
    harness 自己跑的 home check，所以每個 call site 都要走這裡。重開次數記在 sb.n_restarts；重開後把 notes 鏡像
    灌回去，筆數記在 sb.notes_restored。"""
    try:
        return await asyncio.to_thread(sb.run, code)
    except SandboxError as e:
        print(f"!! {e}; restarting sandbox, state lost", file=sys.stderr)
        await asyncio.to_thread(sb.restart)
        sb.notes_synced = 0
        blob = json.dumps(sb.notes_mirror)
        out, timed = await asyncio.to_thread(sb.run, _NOTES_LOAD.format(blob=blob))
        assert not timed and out.strip().isdigit(), f"notes restore failed: {out[-200:]!r}"
        sb.notes_restored = int(out.strip())
        sb.notes_synced = sb.notes_restored
        return SANDBOX_KILLED, True


async def sync_notes(sb: Sandbox) -> None:
    """把 sandbox 裡 notes 的新增條目讀回 harness 的鏡像（sb.notes_mirror），並寫到 run 目錄。
    模型把 notes 換成更短的 list（或非 list）就整份重讀。sandbox 掛了就跳過，下一次 sb_run 會處理重啟。"""
    try:
        out, timed = await asyncio.to_thread(sb.run, _NOTES_LEN)
        if timed or not out.strip().lstrip("-").isdigit():
            return
        n = int(out.strip())
        if n < 0:   # 不是 list：當成單一條目
            out, _ = await asyncio.to_thread(sb.run, "import json as _j; print(_j.dumps(str(notes)[:%d]))" % NOTE_ENTRY_CHARS)
            sb.notes_mirror = [json.loads(out.strip())]
            sb.notes_synced = 1
        else:
            if n < sb.notes_synced:
                sb.notes_mirror, sb.notes_synced = [], 0
            for i in range(sb.notes_synced, n):
                out, _ = await asyncio.to_thread(sb.run, _NOTES_ITEM.format(i=i, n=NOTE_ENTRY_CHARS))
                sb.notes_mirror.append(json.loads(out.strip()))
            sb.notes_synced = n
    except SandboxError:
        return
    sb.notes_path.write_text(json.dumps(sb.notes_mirror, ensure_ascii=False))


async def _tool_round(ep: Episode, sb: Sandbox, tool_calls: list, tool_name: str, hint: str,
                      delegate_ctx: tuple | None = None):
    """delegate_ctx = (client, args, n_test, task)：有的話 `delegate` 工具可用（helper 自己的 _tool_round 不給）。"""
    for call in tool_calls:
        ep.n_tool_calls += 1
        fn = call["function"]
        if fn["name"] == "delegate" and delegate_ctx is not None:
            client, args, n_test, task = delegate_ctx
            try:
                a = fn["arguments"]
                question = (json.loads(a) if isinstance(a, str) else a)["question"]
                assert isinstance(question, str) and question.strip()
            except (json.JSONDecodeError, KeyError, TypeError, AssertionError) as e:
                ep.n_tool_errors += 1
                out = f"Error: delegate needs a non-empty string 'question' ({e})."
            else:
                out = await _delegate(ep, sb, client, args, n_test, task, question)
            ep.messages.append({"role": "tool", "tool_call_id": call.get("id", ""), "content": out + hint})
            continue
        if fn["name"] != tool_name:
            ep.n_tool_errors += 1
            out = f"Error: no tool named {fn['name']!r}; the only tool is {tool_name!r}."
        else:
            args = fn["arguments"]
            try:
                code = (json.loads(args) if isinstance(args, str) else args)["code"]
                if not isinstance(code, str):   # {"code": null} 之類：v8 的 0934a4d8 讓 `out` 沒被定義而整題報廢
                    raise TypeError(f"'code' must be a string, got {type(code).__name__}")
            except (json.JSONDecodeError, KeyError, TypeError) as e:
                ep.n_tool_errors += 1
                code = None
                out = f"Error: could not parse the tool arguments as JSON with a 'code' field ({e})."
            if code is not None:
                out, timed = await sb_run(sb, code)
                if out == SANDBOX_KILLED:
                    ep.n_sandbox_restarts += 1
                    notes_msg = (f"`notes` was restored from the harness's copy ({sb.notes_restored} entries)"
                                 if sb.notes_restored >= 0 else "`notes` is empty again")
                    out = ("Error: the sandbox stopped responding (your code could not be interrupted, or the process "
                           "died) and was killed and restarted. All functions and variables you defined are gone -- "
                           f"train_pairs / test_inputs / np are back, {notes_msg}; redefine anything else you need.")
                else:
                    await sync_notes(sb)
                ep.n_tool_errors += int(timed)
        ep.messages.append({"role": "tool", "tool_call_id": call.get("id", ""), "content": out + hint})


async def run_episode(client: httpx.AsyncClient, args, *, label: str, system: str, user: str,
                      sb: Sandbox, tool: dict, tool_name: str, n_test: int, lineage: int = 0, hop: int = 1,
                      task_id: str = "", handoff_user: str | None = None, finalize: bool = True,
                      task: dict | None = None) -> Episode:
    """`task` 給了且 args.delegate 開著：主迴圈的 tool round 可以分派 `delegate`。helper 自己（task=None）不能。"""
    ep = Episode(label, lineage, hop)
    delegate_ctx = (client, args, n_test, task) if (task is not None and args.delegate) else None
    ep.task_id = task_id
    ep.notes_structured = args.notes_structured
    ep.messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    ep.task_user = user
    # 第一輪還沒有 server 回的 total_tokens，用 tokenizer 估 prompt 長度（+ template/tool schema 的固定開銷）。
    # 不估的話第一輪的 max_tokens 會把整個 cap 當成可生成量，7k 的 prompt 就把 report_reserve 吃掉
    # （smoke2 的 142ca369 explorer 2：一輪純推理 41.5k，交接只剩 988 token，報告被截成只有 RULE）。
    last_total = args.count_tokens(system + user) + args.template_overhead
    need_handoff = None       # None = 正常結束；否則是觸發 handoff 的原因
    for _turn in range(args.max_turns):
        if args.compact and ep.usage("completion_tokens") >= args.compact_gen_budget:
            need_handoff = "gen_budget"
            break
        cut = _cutoff(args)
        if cut is not None:
            need_handoff = cut
            break
        room = args.ctx_cap - args.report_reserve - last_total - args.context_margin
        if room < args.min_turn_tokens and args.compact:
            # context 滿了：不交接，把最舊的 tool round 折掉、同一條對話續跑（sandbox 狀態都在）。
            if args.compact_checkpoint:
                last_total = await _checkpoint_before_compact(ep, args, sb, last_total, client, tool, tool_name)
            new_total = await _compact(ep, args, sb, n_test)
            if new_total is not None:
                last_total = new_total
                room = args.ctx_cap - args.report_reserve - last_total - args.context_margin
        if room < args.min_turn_tokens:
            need_handoff = "context_exhausted"
            break
        # 接近 deadline 時把單輪 max_tokens 一起壓下來：一輪 20k 在 L4 上要跑好幾分鐘，deadline 之後
        # 在飛的 request 不會被砍，只能事前不讓它那麼長。
        mt = min(args.max_turn_tokens, room, _time_room(args))
        if args.compact and args.gen_budget_hard:
            # 硬預算：單輪不得把這條對話的生成量帶過 compact_gen_budget。原本只在回合開頭檢查，單輪最多再超
            # max_turn_tokens（20k）；fp8 sweep 實測 refiner 40k 花到 55k。被截斷的那輪走 CONTINUE，下一輪開頭
            # gen_budget 觸發交接（交接另有 --handoff-budget）。至少給 min_turn_tokens，超支上限就是它。
            mt = max(args.min_turn_tokens, min(mt, args.compact_gen_budget - ep.usage("completion_tokens")))
        payload = _base_payload(args, ep.messages, tool, mt)
        data = await args.chat(client, payload)
        choice = data["choices"][0]
        finish = choice.get("finish_reason")
        msg = choice["message"]
        last_total = _record_usage(ep, data, finish, "turn", payload["max_tokens"])
        tool_calls = msg.get("tool_calls") or []
        if tool_calls:
            asst = {"role": "assistant", "content": msg.get("content") or "", "tool_calls": tool_calls}
            if msg.get("reasoning_content"):
                asst["reasoning_content"] = msg["reasoning_content"]
            _copy_private(asst, msg)
            ep.messages.append(asst)
            left = (args.compact_gen_budget - ep.usage("completion_tokens") if args.compact
                    else args.ctx_cap - args.report_reserve - last_total)
            hint = f"\n\n[budget: ~{max(left, 0) / 1000:.1f}k tokens of working context left]"
            if finish == "length":
                # 截斷的 tool call 引數多半是壞的，不執行。
                for call in tool_calls:
                    ep.n_tool_calls += 1
                    ep.n_tool_errors += 1
                    ep.messages.append({"role": "tool", "tool_call_id": call.get("id", ""),
                                        "content": "Error: your reply was cut off by the per-reply limit; "
                                                   "this call was not executed." + hint})
                if not _continue_after_cut(ep, args, last_total):
                    if args.compact and args.compact_checkpoint:
                        last_total = await _checkpoint_before_compact(ep, args, sb, last_total, client, tool, tool_name)
                    new_total = await _compact(ep, args, sb, n_test) if args.compact else None
                    if new_total is None or not _continue_after_cut(ep, args, new_total):
                        need_handoff = "length"
                        break
                    last_total = new_total
                continue
            await _tool_round(ep, sb, tool_calls, tool_name, hint, delegate_ctx)
            continue
        # 沒有 tool call：最終回覆。
        asst = {"role": "assistant", "content": msg.get("content") or ""}
        if msg.get("reasoning_content"):
            asst["reasoning_content"] = msg["reasoning_content"]
        _copy_private(asst, msg)
        ep.messages.append(asst)
        if finish == "length":
            # 單輪思考撞到 max_turn_tokens（v1 有 107/185 次交接是這樣來的）：總預算還有就叫它
            # checkpoint 再繼續，不要交接。preserve_thinking 讓被截斷的思考留在 context 裡。
            if _continue_after_cut(ep, args, last_total):
                continue
            if args.compact:
                if args.compact_checkpoint:
                    last_total = await _checkpoint_before_compact(ep, args, sb, last_total, client, tool, tool_name)
                new_total = await _compact(ep, args, sb, n_test)
                if new_total is not None and _continue_after_cut(ep, args, new_total):
                    last_total = new_total
                    continue
            need_handoff = "length_final"
            break
        ep.final_content = msg.get("content") or ""
        ep.final_reasoning = msg.get("reasoning_content") or ""
        ep.stop_reason = "answered"
        break
    else:
        need_handoff = "turn_cap"

    if need_handoff in ("deadline", "abort"):
        # 不再呼叫模型。交接報告本身要一次 ≥3.5k token 的生成，deadline 之後沒有這個預算；答案改由呼叫端從
        # sandbox 現有的 `transform` 收（run_hop 的 home_check 對 code=None 走的就是這條路），report 是空的。
        ep.forced_handoff = True
        ep.stop_reason = f"handoff:{need_handoff}"
    elif need_handoff is not None:
        ep.forced_handoff = True
        ep.messages.append({"role": "user", "content": handoff_user or prompts.HANDOFF_USER.format(n_calls=args.handoff_tool_calls)})
        calls_left = 0 if handoff_user else args.handoff_tool_calls
        refused = 0          # tool_choice=none 在 tokens/dsv4 backend 沒有對應物：模型在 STOP 後還叫工具就在這裡擋
        handoff_used = 0     # 交接階段累計生成量（--handoff-budget 的分母）
        while True:
            if _cutoff(args) is not None:
                break        # 交接期間到點：報告寫到哪算到哪，不再多發一次生成
            mt = args.ctx_cap - last_total - args.handoff_margin
            if mt < args.min_handoff_tokens:
                # report_reserve 是留給這裡的，正常到不了；真到了就寧可略超 cap 也要拿到完整報告。
                mt = args.min_handoff_tokens
            if calls_left > 0:
                # checkpoint 的 tool call 要短：smoke3 裡不設上限時這一輪思考 6–12k，把報告擠到被截斷。
                mt = min(mt, args.handoff_turn_tokens)
            if args.handoff_budget:
                # 交接階段整體的硬上限。不寫報告、一直叫工具的模型會把 checkpoint 2 次 + 拒絕 3 次的每一輪都生到
                # ctx 餘額（最多 ~30k）：本機 plumbing 量到 explorer 預算 20k 卻花 30–58k，超支全在這裡。
                # Kaggle 的預算是硬的，寧可拿到被截斷的報告（答案還能從 sandbox 的 transform 收）。
                left_h = args.handoff_budget - handoff_used
                if left_h < 256:
                    print(f"!! {ep.task_id} {ep.label}: handoff budget {args.handoff_budget} used up; forcing final",
                          file=sys.stderr)
                    break
                mt = min(mt, left_h)
            payload = _base_payload(args, ep.messages, tool, mt)
            if calls_left <= 0:
                payload["tool_choice"] = "none"
            if args.handoff_thinking == "off":
                payload["chat_template_kwargs"] = {"enable_thinking": False}
            elif args.handoff_thinking == "low":
                payload["reasoning_effort"] = "low"
            data = await args.chat(client, payload)
            choice = data["choices"][0]
            msg = choice["message"]
            finish = choice.get("finish_reason")
            last_total = _record_usage(ep, data, finish, "handoff", mt)
            handoff_used += ep.calls[-1]["completion_tokens"]
            tool_calls = msg.get("tool_calls") or []
            asst = {"role": "assistant", "content": msg.get("content") or ""}
            if msg.get("reasoning_content"):
                asst["reasoning_content"] = msg["reasoning_content"]
            _copy_private(asst, msg)
            if tool_calls and finish != "length" and calls_left > 0:
                asst["tool_calls"] = tool_calls
                ep.messages.append(asst)
                calls_left -= len(tool_calls)
                hint = (f"\n\n[handoff: {max(calls_left, 0)} tool call(s) left, then write the FINAL REPORT]"
                        if calls_left > 0 else "\n\n[handoff: no tool calls left -- write the FINAL REPORT now]")
                await _tool_round(ep, sb, tool_calls, tool_name, hint)
                continue
            if tool_calls or (finish == "length" and calls_left > 0):
                # checkpoint 那輪被截斷（tool call 或純思考）：不執行，直接要報告。
                if tool_calls:
                    asst["tool_calls"] = tool_calls
                ep.messages.append(asst)
                for call in tool_calls:
                    ep.n_tool_calls += 1
                    ep.n_tool_errors += 1
                    ep.messages.append({"role": "tool", "tool_call_id": call.get("id", ""),
                                        "content": "Error: cut off; not executed."})
                ep.messages.append({"role": "user", "content": "Cut off. No more tool calls -- write the FINAL REPORT now."})
                calls_left = 0
                refused += 1
                if refused <= 2:
                    continue
                # 連續三次不寫報告還在叫工具（2026-09-03 審核：20/14k 條 episode 在 STOP 後又叫 3–23 次、context 超 cap）：
                # 就把這一輪的內容當最終回覆，不再給機會。
                print(f"!! {ep.task_id} {ep.label}: still calling tools after STOP x{refused}; forcing final", file=sys.stderr)
            else:
                ep.messages.append(asst)
            ep.final_content = msg.get("content") or ""
            ep.final_reasoning = msg.get("reasoning_content") or ""
            break
        ep.stop_reason = f"handoff:{need_handoff}"

    if finalize:
        await finalize_report(ep, sb, n_test)
    if args.save_segments:
        if args.annotate_ids is not None:
            args.annotate_ids(ep.messages, args.reasoning_effort)
        ep.segments.append(copy.deepcopy(ep.messages))
    ep.latency_s = time.monotonic() - ep.t0
    return ep


async def _delegate(ep: Episode, sb: Sandbox, client, args, n_test: int, task: dict, question: str) -> str:
    """--delegate：開一條乾淨 context 的 helper 對話（同一個 sandbox），跑完只把它的短結論當 tool 結果回給 explorer。
    helper 的 token 併進父 episode 的 calls（kind 前綴 delegate:），對話本身存進 ep.sub_episodes。"""
    if ep.n_delegations >= args.delegate_max:
        ep.n_tool_errors += 1
        return f"Error: delegation limit ({args.delegate_max} per conversation) reached; continue yourself."
    ep.n_delegations += 1
    names_out, _ = await sb_run(sb, "print(sorted(k for k, v in globals().items() if not k.startswith('_') "
                                    "and callable(v) and getattr(v, '__module__', None) == '__main__'))")
    notes = await read_notes(sb, args.tool_max_output_chars, max_items=20, max_chars=2500)
    test_inputs = [t["input"] for t in task["test"]]
    system = prompts.helper_system(n_test, args.delegate_budget, args.tool_timeout, args.tool_max_output_chars)
    if args.compact:
        system += prompts.COMPACT_SYSTEM_NOTE
    user = prompts.render_helper_user(task["train"], test_inputs, names_out.strip()[-300:], notes, args.notes_structured, question)
    sub_args = argparse.Namespace(**vars(args))
    sub_args.extra_tools = []                         # helper 不能再 delegate
    sub_args.compact_gen_budget = args.delegate_budget
    sub_args.max_turns = args.delegate_max_turns
    sub = await run_episode(client, sub_args, label=f"{ep.label} helper {ep.n_delegations}", system=system, user=user,
                            sb=sb, tool=args.tool, tool_name=args.tool_name, n_test=n_test, lineage=ep.lineage, hop=ep.hop,
                            task_id=ep.task_id, handoff_user=prompts.HELPER_HANDOFF, finalize=False)
    for c in sub.calls:
        ep.calls.append(dict(c, kind="delegate:" + c["kind"]))
    ep.n_tool_calls += sub.n_tool_calls
    ep.sub_episodes.append(sub.to_dict(keep_messages=args.save_messages))
    reply = (sub.final_content or "").strip()
    if len(reply) > args.delegate_reply_chars:
        reply = reply[:args.delegate_reply_chars] + "\n... (helper reply truncated)"
    print(f"·· {ep.task_id} {ep.label}: delegation #{ep.n_delegations} used {sub.usage('completion_tokens') / 1000:.0f}k gen, "
          f"{sub.n_tool_calls} tool calls, stop={sub.stop_reason}", file=sys.stderr)
    return (f"[helper finished: {sub.usage('completion_tokens') / 1000:.0f}k tokens, {sub.n_tool_calls} tool calls, "
            f"stop={sub.stop_reason}]\n" + (reply or "(the helper produced no reply)"))


def _msg_tokens(args, m: dict) -> int:
    """一則訊息的估計 token（內容 + reasoning + tool call 引數 + 每則 8 token 的模板開銷）。"""
    n = 8 + args.count_tokens(m.get("content") or "")
    if m.get("reasoning_content"):
        n += args.count_tokens(m["reasoning_content"])
    for call in m.get("tool_calls") or []:
        n += args.count_tokens(call["function"]["arguments"]) + 12
    return n


OUTPUT_CLEARED = ("[output cleared by the harness to save context -- the code above already ran and everything it "
                  "defined is still in the sandbox; re-run it if you need to see the output again]")


def _round_summary(r: list[dict], ep: Episode) -> str:
    """被折掉的一輪留一行：第一個 tool call 的 code 開頭 → 輸出開頭（Manus：失敗要留在 context 裡，不然會重試）。"""
    a = r[0]
    calls = a.get("tool_calls") or []
    if calls:
        args = calls[0]["function"]["arguments"]
        try:
            code = (json.loads(args) if isinstance(args, str) else args).get("code") or ""
        except (json.JSONDecodeError, AttributeError, TypeError):
            code = str(args)
        lines = [ln.strip() for ln in code.splitlines() if ln.strip() and not ln.strip().startswith("#")]
        head = " | ".join(lines[:2])[:110]
    else:
        head = "(no tool call) " + (a.get("content") or "")[:80].replace("\n", " ")
    outs = [m for m in r if m["role"] == "tool"]
    if not outs:
        return head
    out = outs[0]["content"]
    if out == OUTPUT_CLEARED:
        out = ep.output_heads.get(outs[0].get("tool_call_id", ""), "(output cleared)")
    out = out.split("\n\n[budget")[0].strip().replace("\n", " / ")
    return f"{head}  ->  {out[:110]}"


def _clear_tool_outputs(r: list[dict], ep: Episode) -> None:
    """Anthropic context-editing 的做法：把一輪的 tool 輸出換成佔位符，留下模型自己的 reasoning 與 code。
    輸出是可還原的（code 還在、sandbox 還在），reasoning 不是。"""
    for m in r:
        if m["role"] == "tool" and m["content"] != OUTPUT_CLEARED:
            ep.output_heads[m.get("tool_call_id", "")] = m["content"][:160]
            m["content"] = OUTPUT_CLEARED
            ep.n_outputs_cleared += 1


COMPACT_NOTE_HEAD = """\
[harness note -- context compacted]
Your earliest {n_rounds} tool round(s) ({n_calls} tool calls, about {dropped_k}k tokens) were removed from this \
conversation to stay within the context limit; only your most recent rounds are shown below. The sandbox was NOT \
reset: whatever those rounds successfully defined is still there. The list below is the authoritative inventory -- \
a name that is not in it does not exist (its definition failed or was never run), so if you need it, define it again \
rather than calling it. Do not redo work the sandbox already holds; re-run a function if you need its output again.
Sandbox now: functions defined: {names}. `transform` on the training pairs: {train}."""


async def _checkpoint_before_compact(ep: Episode, args, sb: Sandbox, last_total: int, client, tool, tool_name) -> int:
    """--compact-checkpoint：折掉舊輪次之前先給模型一輪（≤ compact_checkpoint_tokens）把即將消失的推理寫進 `notes`。
    這一輪連同它的 tool 輸出是最新的 round，compaction 會留下來；成本從 report_reserve 的閒置空間出。回傳新的 last_total。"""
    starts = [i for i, m in enumerate(ep.messages) if m["role"] == "assistant"]
    n_rounds = len(starts)
    tmpl = prompts.CHECKPOINT_USER_STRUCTURED if args.notes_structured else prompts.CHECKPOINT_USER
    ep.messages.append({"role": "user", "content": tmpl.format(n_rounds=max(n_rounds - 1, 1))})
    mt = min(args.compact_checkpoint_tokens, args.ctx_cap - last_total - args.context_margin)
    if mt < 1000:
        # 連 checkpoint 都塞不下（單輪把 reserve 也吃掉了）：略過，直接折。
        ep.messages.pop()
        return last_total
    payload = _base_payload(args, ep.messages, tool, mt)
    data = await args.chat(client, payload)
    choice = data["choices"][0]
    msg = choice["message"]
    finish = choice.get("finish_reason")
    last_total = _record_usage(ep, data, finish, "checkpoint", mt)
    tool_calls = msg.get("tool_calls") or []
    asst = {"role": "assistant", "content": msg.get("content") or ""}
    if msg.get("reasoning_content"):
        asst["reasoning_content"] = msg["reasoning_content"]
    _copy_private(asst, msg)
    if tool_calls:
        asst["tool_calls"] = tool_calls
    ep.messages.append(asst)
    ep.n_checkpoints += 1
    if tool_calls and finish != "length":
        await _tool_round(ep, sb, tool_calls, tool_name, "")
    elif tool_calls:
        for call in tool_calls:
            ep.n_tool_calls += 1
            ep.n_tool_errors += 1
            ep.messages.append({"role": "tool", "tool_call_id": call.get("id", ""), "content": "Error: cut off; not executed."})
    return last_total


async def _compact(ep: Episode, args, sb: Sandbox, n_test: int) -> int | None:
    """--compact：context 滿了就把最舊的 tool round 從 messages 折掉（sandbox 裡的狀態一個都不少），
    題目訊息後面插一則 harness 公告（折掉多少、sandbox 現況、notes），同一條對話繼續。
    --compact-rich 再加三件事：(1) 先把非最近幾輪的 tool 輸出換成佔位符（reasoning 與 code 留著），夠了就不丟整輪；
    (2) 公告帶被折掉輪次裡最後一版 `def transform` 原始碼；(3) 每個被折掉的輪次留一行「試了什麼 → 結果」。
    回傳折掉後的估計 context token；只剩一輪、折不動就回 None（呼叫端照舊交接）。"""
    msgs = ep.messages
    assert msgs[0]["role"] == "system" and msgs[1]["role"] == "user" and msgs[1]["content"] == ep.task_user, ep.label
    starts = [i for i, m in enumerate(msgs) if m["role"] == "assistant"]
    if len(starts) < 2:
        return None
    rounds = [msgs[a:b] for a, b in zip(starts, starts[1:] + [len(msgs)])]
    # --save-segments：折之前先把模型當下看到的整份 messages 存起來（rich 模式接著會就地清 tool 輸出，所以要在這裡 deepcopy）。
    if args.save_segments and args.annotate_ids is not None:
        args.annotate_ids(msgs, args.reasoning_effort)   # 最後一輪 tool 訊息還沒送過 server，先把 token 補齊
    snap = copy.deepcopy(msgs) if args.save_segments else None
    # 折完的 context 目標 = 工作視窗的 compact_target_frac；扣掉 system + 題目 + 公告（估 3k）就是能留給最近幾輪的量，
    # 下限 compact_keep_tokens。refiner 的題目訊息（K 份報告 + 候選表）比 explorer 大得多，固定 keep 會讓它折完就又快滿。
    base = _msg_tokens(args, msgs[0]) + _msg_tokens(args, msgs[1]) + 3000 + args.template_overhead
    window = args.ctx_cap - args.report_reserve - args.context_margin
    keep_budget = max(args.compact_keep_tokens, int(window * args.compact_target_frac) - base)
    if args.compact_rich:
        # 第一階段：最近 compact_keep_full_rounds 輪以外的 tool 輸出清掉。
        for r in rounds[:max(len(rounds) - args.compact_keep_full_rounds, 0)]:
            _clear_tool_outputs(r, ep)
    sizes = [sum(_msg_tokens(args, m) for m in r) for r in rounds]
    # 第二階段：從最新一輪往回留；至少留 1 輪。rich 模式清完輸出就夠的話一輪都不丟。
    keep, acc = 0, 0
    for size in reversed(sizes):
        if keep >= 1 and acc + size > keep_budget:
            break
        keep += 1
        acc += size
    keep = min(keep, len(rounds) - 1) if not args.compact_rich else min(keep, len(rounds))
    dropped = rounds[:len(rounds) - keep]
    if not dropped and ep.n_outputs_cleared == 0:
        return None
    n_calls = sum(len(m.get("tool_calls") or []) for r in dropped for m in r)
    ep.compact_dropped_tokens += sum(sizes[:len(dropped)])
    ep.compact_dropped_rounds += len(dropped)
    ep.compact_dropped_calls += n_calls
    ep.n_compactions += 1
    if snap is not None:
        ep.segments.append(snap)
    if args.compact_rich and dropped:
        ep.compact_history.extend(_round_summary(r, ep) for r in dropped)
        ep.compact_history = ep.compact_history[-40:]
        src = _salvage_transform_code([m for r in dropped for m in r])
        if src:
            ep.compact_transform_src = src

    names_out, _ = await sb_run(sb, "print(sorted(k for k, v in globals().items() if not k.startswith('_') "
                                    "and callable(v) and getattr(v, '__module__', None) == '__main__'))")
    chk = await _check_transform(sb, None, "transform", n_test)
    notes = await read_notes(sb, args.tool_max_output_chars, max_items=12, max_chars=1500)
    parts = [COMPACT_NOTE_HEAD.format(
        n_rounds=ep.compact_dropped_rounds, n_calls=ep.compact_dropped_calls,
        dropped_k=f"{ep.compact_dropped_tokens / 1000:.0f}", names=_trim_names(names_out.strip()),
        train=chk["summary"])]
    if args.compact_rich and ep.compact_transform_src and chk["defined"]:
        src = ep.compact_transform_src
        if len(src) > 2500:
            src = src[:1600] + "\n# ... (middle elided) ...\n" + src[-800:]
        parts.append("Current `transform` in the sandbox (the last definition from the removed rounds, verbatim):\n"
                     "```python\n" + src.rstrip() + "\n```")
    if args.compact_rich and ep.compact_history:
        parts.append("What the removed rounds tried (oldest first; one line each: code -> output):\n"
                     + "\n".join(f"- {h}" for h in ep.compact_history))
    parts.append(f"`notes` ({len(notes)} entries, newest last):\n" + prompts.render_notes(notes, args.notes_structured))
    parts.append("Continue from where you left off.")
    note = "\n".join(parts)
    # 公告是獨立的一則 user 訊息，題目訊息一字不動：system + 題目 的前綴才能一直命中 radix cache
    # （smoke 裡把公告掛在題目尾巴，每次折完命中率就掉回只剩 system prompt）。
    new_msgs = [msgs[0], msgs[1], {"role": "user", "content": note}]
    for r in rounds[len(dropped):]:
        new_msgs.extend(r)
    ep.messages = new_msgs
    est = sum(_msg_tokens(args, m) for m in new_msgs) + args.template_overhead
    print(f"·· {ep.task_id} {ep.label}: compaction #{ep.n_compactions} dropped {len(dropped)} rounds "
          f"(~{sum(sizes[:len(dropped)]) / 1000:.0f}k), outputs cleared so far {ep.n_outputs_cleared}, "
          f"context est {est / 1000:.0f}k", file=sys.stderr)
    return est


def _trim_names(names: str, limit: int = 900) -> str:
    """公告裡的 sandbox 函式清單：保留開頭、超過就明講截掉幾個（之前是 `[-300:]` 截頭，11% 的公告清單開頭是半個名字）。"""
    if not names:
        return "(none)"
    if len(names) <= limit:
        return names
    cut = names[:limit]
    n_more = names[limit:].count("', '") + 1
    return cut.rsplit("', '", 1)[0] + f"', ... (list truncated, {n_more} more)"


def _continue_after_cut(ep: Episode, args, last_total: int) -> bool:
    """單輪被 max_turn_tokens 截斷：總預算還夠、次數未用完就追加 CONTINUE 訊息讓它接著做。"""
    left = args.ctx_cap - args.report_reserve - last_total - args.context_margin
    # compact 模式的總量由 --compact-gen-budget 管，續跑次數不設限（smoke 裡一條 explorer 在 104k 時因為
    # 用完 4 次續跑就交接，白丟了 100k 的預算）。
    if (not args.compact and ep.n_continuations >= args.max_continuations) or left < args.min_turn_tokens:
        return False
    ep.n_continuations += 1
    ep.messages.append({"role": "user", "content": prompts.CONTINUE_USER.format(left_k=f"{left / 1000:.0f}")})
    return True


def _salvage_transform_code(messages: list[dict]) -> str | None:
    """報告裡沒 CODE 時，撈最後一次在 tool call 裡定義 `transform` 的程式碼。"""
    for m in reversed(messages):
        if m["role"] != "assistant":
            continue
        for call in reversed(m.get("tool_calls") or []):
            args = call["function"]["arguments"]
            try:
                code = (json.loads(args) if isinstance(args, str) else args)["code"]
            except (json.JSONDecodeError, KeyError, TypeError):
                continue
            if re.search(r"^\s*def transform\s*\(", code, re.MULTILINE):
                return code
    return None


async def eval_grid_expr(sb: Sandbox, expr: str):
    """在 sandbox 算一句表達式成 grid；失敗回 (None, reason)。"""
    code = (f"import json as _json\n_v = ({expr})\n"
            f"print({GRID_MARK!r} + _json.dumps(np.asarray(_v).astype(int).tolist(), separators=(',', ':')))")
    out, timed = await sb_run(sb, code)
    if timed:
        return None, "timeout"
    return _grid_from_output(out)


def _grid_from_output(out: str):
    i = out.find(GRID_MARK)
    if i < 0:
        return None, "error: " + out.strip()[-300:]
    line = out[i + len(GRID_MARK):].split("\n", 1)[0]
    if "[truncated" in out:
        return None, "output truncated"
    try:
        g = json.loads(line)
    except json.JSONDecodeError:
        return None, "unparseable"
    if not prompts.is_grid(g):
        return None, "not a grid"
    return g, "ok"


async def finalize_report(ep: Episode, sb: Sandbox, n_test: int):
    rep = prompts.parse_report(ep.final_content)
    src = "content"
    if not rep["has_sections"] and ep.final_reasoning:
        alt = prompts.parse_report(ep.final_reasoning)
        if alt["has_sections"]:
            rep, src = alt, "reasoning"
    rep["source"] = src
    if rep["code"] is None:
        salvaged = _salvage_transform_code(ep.messages)
        if salvaged is not None:
            rep["code"] = salvaged
            ep.code_src = "salvaged_tool_call"
    else:
        ep.code_src = f"report_{src}"
    ep.report = rep
    ep.answers = [[] for _ in range(n_test)]
    ep.answer_resolution = [[] for _ in range(n_test)]
    for key, (kind, body) in sorted(rep["answer_blocks"].items(), key=lambda kv: tuple(map(int, kv[0].split(":")))):
        t, a = map(int, key.split(":"))
        if t >= n_test or a >= 2:
            continue
        if kind == "none":
            continue
        if kind == "grid":
            g, why = json.loads(body), "ok"
        else:
            g, why = await eval_grid_expr(sb, body)
            if g is None and "NameError" in why and rep["code"]:
                # 報告裡的 CODE 沒在 sandbox 跑過（或用了別的名字）：把它定義進去再算一次。
                out, _ = await sb_run(sb, rep["code"])
                g, why2 = await eval_grid_expr(sb, body)
                why = f"{why2} after defining report code"
        ep.answer_resolution[t].append(f"{kind}:{why}")
        if g is not None and g not in ep.answers[t]:
            ep.answers[t].append(g)


# ---------------------------------------------------------------------------
# Verification of explorer code in the refine sandbox
# ---------------------------------------------------------------------------

_DEFINE_TMPL = """\
import numpy as np, itertools, math, re, copy, functools, collections
from collections import Counter, deque, defaultdict
_ns = {{"np": np, "itertools": itertools, "math": math, "re": re, "copy": copy, "functools": functools,
        "collections": collections, "Counter": Counter, "deque": deque, "defaultdict": defaultdict,
        "__name__": "__explorer__"}}
try:
    exec({code!r}, _ns)
    {name} = _ns["transform"]
    print("defined")
except BaseException as _e:
    print({def_err!r} + repr(_e)[:300])
"""

# 接續 session 用：CODE 直接 exec 進 sandbox 的 globals，報告裡的碼常依賴 sandbox 裡已定義的 helper。
_DEFINE_GLOBAL_TMPL = """\
try:
    exec({code!r}, globals())
    assert callable(transform)
    print("defined")
except BaseException as _e:
    print({def_err!r} + repr(_e)[:300])
"""

_TRAIN_CHECK_TMPL = """\
import json as _json
_res = []
for _p in train_pairs:
    try:
        _o = np.asarray({name}([r[:] for r in _p["input"]])).astype(int).tolist()
        _e = _p["output"]
        if _o == _e:
            _res.append("pass")
        elif len(_o) != len(_e) or len(_o[0]) != len(_e[0]):
            _res.append("FAIL shape %dx%d vs %dx%d" % (len(_o), len(_o[0]), len(_e), len(_e[0])))
        else:
            _n = sum(1 for i in range(len(_e)) for j in range(len(_e[0])) if _o[i][j] != _e[i][j])
            _res.append("FAIL %d cells" % _n)
    except BaseException as _x:
        _res.append("ERROR " + repr(_x)[:120])
print({mark!r} + _json.dumps(_res))
"""

_TEST_PRED_TMPL = """\
import json as _json
try:
    _o = np.asarray({name}([r[:] for r in test_inputs[{t}]])).astype(int).tolist()
    print({mark!r} + _json.dumps(_o, separators=(',', ':')))
except BaseException as _x:
    print({err!r} + repr(_x)[:200])
"""


NOTES_MARK = "__NOTES__"


async def _check_transform(sb: Sandbox, code: str | None, name: str, n_test: int, in_globals: bool = False,
                           load_only: bool = False) -> dict:
    """把 `code` 定義成 sandbox 裡的 `name`（code 為 None 就用 sandbox 裡已有的），跑全部 train pair 與 test input。
    `in_globals`：直接在 sandbox 的 globals 裡 exec（只能是 name == "transform"），helper 才找得到。"""
    chk = {"name": name, "defined": False, "train": [], "test": [None] * n_test, "summary": "no code"}
    if code:
        if in_globals:
            assert name == "transform"
            define = _DEFINE_GLOBAL_TMPL.format(code=code, def_err=DEF_ERR_MARK)
        else:
            define = _DEFINE_TMPL.format(code=code, name=name, def_err=DEF_ERR_MARK)
        out, timed = await sb_run(sb, define)
        if timed or DEF_ERR_MARK in out or "defined" not in out:
            chk["summary"] = "code failed to load: " + out.replace(DEF_ERR_MARK, "").strip()[-200:]
            return chk
    else:
        out, _ = await sb_run(sb, f"print('defined' if callable(globals().get({name!r})) else 'undefined')")
        if "defined" not in out or "undefined" in out:
            chk["summary"] = f"no `{name}` in the sandbox"
            return chk
    chk["defined"] = True
    if load_only:
        chk["summary"] = "loaded"
        return chk
    out, timed = await sb_run(sb, _TRAIN_CHECK_TMPL.format(name=name, mark=GRID_MARK))
    if timed or GRID_MARK not in out:
        chk["train"] = ["TIMEOUT" if timed else "ERROR"]
    else:
        chk["train"] = json.loads(out.split(GRID_MARK, 1)[1].split("\n", 1)[0])
    for t in range(n_test):
        out, timed = await sb_run(
            sb, _TEST_PRED_TMPL.format(name=name, t=t, mark=GRID_MARK, err=ERR_MARK))
        if timed:
            chk["test"][t] = {"grid": None, "why": "timeout"}
        elif ERR_MARK in out:
            chk["test"][t] = {"grid": None, "why": out.split(ERR_MARK, 1)[1].split("\n", 1)[0][:200]}
        else:
            g, why = _grid_from_output(out)
            chk["test"][t] = {"grid": g, "why": why}
    tr = chk["train"]
    npass = sum(1 for r in tr if r == "pass")
    bits = [f"train pairs {npass}/{len(tr)} pass" + ("" if npass == len(tr) else
            " (" + "; ".join(f"pair {i + 1}: {r}" for i, r in enumerate(tr) if r != "pass") + ")")]
    for t in range(n_test):
        r = chk["test"][t]
        if r["grid"] is None:
            bits.append(f"test {t + 1}: {r['why']}")
        else:
            bits.append(f"test {t + 1}: {len(r['grid'])}x{len(r['grid'][0])} grid")
    chk["summary"] = "; ".join(bits)
    return chk


def _checks_from_home(reports: list[dict]) -> list[dict]:
    """不開 sandbox 的檢查表：直接用 explorer 自家 sandbox 的 home_check（verify_reports 多做的只是把 CODE 載進
    refine sandbox）。給「refiner 之前先寫一份 submission」與「deadline 到了、refiner 沒跑」兩條路用。"""
    out = []
    for k, rep in enumerate(reports, start=1):
        home = rep["home_check"]
        assert home is not None, rep["label"]
        out.append({"name": f"transform_{k}", "defined": home["defined"], "load_err": None,
                    "train": home["train"], "test": home["test"], "summary": home["summary"]})
    return out


async def verify_reports(sb: Sandbox, reports: list[dict], n_test: int) -> list[dict]:
    """每份報告一個 check dict：train/test 結果取自 explorer 自家 sandbox 的 home_check（v4–v6 在 refine sandbox
    重跑 CODE，28% 因為 helper / import 不在而 NameError，refiner 被告知能用的碼壞了）；refine sandbox 只嘗試把
    CODE 載成 transform_k，載不進來就在 summary 裡講明。"""
    checks = []
    for k, rep in enumerate(reports, start=1):
        home = rep["home_check"]
        assert home is not None, rep["label"]
        name = f"transform_{k}"
        chk = {"name": name, "defined": False, "load_err": None, "train": home["train"], "test": home["test"]}
        if rep["code"]:
            here = await _check_transform(sb, rep["code"], name, n_test, in_globals=False, load_only=True)
            chk["defined"] = here["defined"]
            chk["load_err"] = None if here["defined"] else here["summary"]
        if not home["defined"]:
            chk["summary"] = home["summary"]
        else:
            chk["summary"] = "run in the explorer's own sandbox (where its helpers exist): " + home["summary"]
            if chk["defined"]:
                chk["summary"] += f" -- also loaded in your sandbox as `{name}`"
            else:
                chk["summary"] += (f" -- NOTE: it does NOT load in your sandbox ({chk['load_err']}); "
                                   "re-implement the missing pieces if you need to run it")
        checks.append(chk)
    return checks


async def read_notes(sb: Sandbox, out_limit: int, max_items: int = 40, max_chars: int = 3000) -> list[str]:
    """sandbox 裡 `notes` 的內容（模型可能把它弄成非 list 或塞進奇怪的東西，一律轉字串、截短）。
    截短在 sandbox 裡做：印出來的那一行必須小於 sandbox 的輸出上限 `out_limit`，否則會被攔腰截斷、JSON 壞掉。"""
    budget = min(max_chars, out_limit - len(NOTES_MARK) - 200)
    assert budget > 0, out_limit
    code = f"""import json as _json
_n = notes if isinstance(notes, (list, tuple)) else [notes]
_items = [str(_x)[:400] for _x in list(_n)[-{max_items}:]]
_kept, _text = [], 0
for _s in _items:
    _text += len(_s)
    if _text > {budget}:
        break
    _kept.append(_s)
_out = _json.dumps([_kept, len(_items) - len(_kept)])
while len(_out) > {budget} and _kept:
    _kept.pop()
    _out = _json.dumps([_kept, len(_items) - len(_kept)])
print({NOTES_MARK!r} + _out)"""
    out, timed = await sb_run(sb, code)
    if timed or NOTES_MARK not in out:
        return [f"(could not read notes: {out.strip()[-200:]})"]
    if "[truncated:" in out:
        raise RuntimeError(f"notes output truncated despite budget {budget}: {out[:300]!r}")
    kept, dropped = json.loads(out.split(NOTES_MARK, 1)[1].split("\n", 1)[0])
    if dropped:
        kept.append(f"... ({dropped} more entries not shown)")
    return kept


def _is_alt_source(src: str) -> bool:
    return src.endswith(" alt") or src.startswith("transform_alt_")


def build_candidates(reports: list[dict], checks: list[dict], n_test: int) -> list[dict]:
    """每個 test：去重後的候選 grid，A、B、… 依（票數、首次出現）排序。
    alt 變體的反假設（ATTEMPT 2、全過 train 的 `transform_alt`）也進候選，但不算票：它是同一條 explorer的第二讀法。"""
    cands = []
    for t in range(n_test):
        pool: list[tuple[list, str, int | None]] = []
        for k, (rep, chk) in enumerate(zip(reports, checks), start=1):
            for i, g in enumerate(rep["answers"][t]):
                pool.append((g, f"explorer {k} answer" if i == 0 else f"explorer {k} alt", rep["confidence"]))
            pred = chk["test"][t]
            if pred is not None and pred["grid"] is not None:
                pool.append((pred["grid"], f"transform_{k}", rep["confidence"]))
            alt = rep["alt_check"]
            if _all_train_pass(alt) and alt["test"][t]["grid"] is not None:
                pool.append((alt["test"][t]["grid"], f"transform_alt_{k}", rep["confidence"]))
        distinct: list[dict] = []
        for g, src, conf in pool:
            for d in distinct:
                if d["grid"] == g:
                    d["sources"].append(src)
                    d["confs"].append(conf)
                    break
            else:
                distinct.append({"grid": g, "sources": [src], "confs": [conf]})
        distinct.sort(key=lambda d: (-sum(1 for s in d["sources"] if not _is_alt_source(s)),
                                     -max((c for c in d["confs"] if c is not None), default=-1)))
        keyed = {}
        for i, d in enumerate(distinct):
            key = chr(ord("A") + i) if i < 26 else f"C{i}"
            keyed[key] = d
        cands.append(keyed)
    return cands


# ---------------------------------------------------------------------------
# Per-task orchestration
# ---------------------------------------------------------------------------

def load_task(task_dir: Path, task_id: str) -> dict:
    d = json.loads((task_dir / f"{task_id}.json").read_text())
    return {"id": task_id, "train": d["train"], "test": d["test"]}


# ---------------------------------------------------------------- --task-source kaggle（ARC Prize 提交）
def load_challenges(path: Path) -> dict:
    """比賽的單一大 JSON（`arc-agi_test_challenges.json`，test 沒有 output），也吃 `<dir>/<task_id>.json`
    的本地佈局（public eval 的 test 內嵌 output，會算分）。與 work/kaggle_submit/run_submission.py 同口徑。"""
    if path.is_dir():
        return {p.stem: json.loads(p.read_text()) for p in sorted(path.glob("*.json"))}
    return json.loads(path.read_text())


# ---------------------------------------------------------------- --task-source nvarc（蒸餾語料）
def choose_pairs(uid: str, attempt: int, n_pairs: int, n_train: int, n_test: int = 1):
    """逐字對齊 work/qwen38fn_distill/rollout_runner.py::choose_pairs（n_test=1 時回傳完全相同）：
    同一個 uid/attempt 在這裡與 naive 語料抽到同一組 train/test，兩批資料才能逐題配對比。"""
    h = hashlib.sha256(f"{uid}:{attempt}".encode()).digest()
    order = sorted(range(n_pairs), key=lambda i: hashlib.sha256(h + i.to_bytes(2, "big")).digest())
    chosen = order[: n_train + n_test]
    return sorted(chosen[:n_train]), chosen[n_train:n_train + n_test]


def load_nvarc_corpus(path: str) -> dict[str, list]:
    """`nvarc_training.jsonl.zst`：每行 {"uid", "pairs": [[input, output], ...]}。整份約 47k 題、90 MB 壓縮。"""
    import io
    import zstandard
    corpus: dict[str, list] = {}
    with open(path, "rb") as f:
        text = io.TextIOWrapper(zstandard.ZstdDecompressor().stream_reader(f), encoding="utf-8")
        for line in text:
            d = json.loads(line)
            assert d["uid"] not in corpus, f"duplicate uid {d['uid']}"
            corpus[d["uid"]] = d["pairs"]
    return corpus


def nvarc_task(uid: str, pairs: list, attempt: int, n_train: int, n_test: int) -> dict:
    tr, te = choose_pairs(uid, attempt, len(pairs), n_train, n_test)
    return {"id": uid,
            "train": [{"input": pairs[i][0], "output": pairs[i][1]} for i in tr],
            "test": [{"input": pairs[i][0], "output": pairs[i][1]} for i in te],
            "train_idx": tr, "test_idx": te}


# --hparam-menu：每題決定性抽一組超參數（各軸獨立雜湊，軸之間不相關），讓語料涵蓋不同 context 上限 / effort / K。
# 權重是收資料的分佈，不是最佳設定；抽到的值連同 uid 存在 tasks/<uid>.json 的 hparams。
HPARAM_MENU: dict[str, list[tuple]] = {
    "reasoning_effort": [("xhigh", 0.6), ("medium", 0.3), ("low", 0.1)],
    "ctx_cap": [(50_000, 0.6), (40_000, 0.2), (65_000, 0.2)],
    "n_explorers": [(2, 0.7), (3, 0.3)],           # base_explorers 固定 2：第 3 條只在前兩條不一致時開
    "compact_target_frac": [(0.6, 0.6), (0.5, 0.2), (0.7, 0.2)],
    "compact_gen_budget": [(100_000, 0.7), (150_000, 0.3)],
}


def draw_hparams(uid: str, attempt: int, menu: dict = HPARAM_MENU) -> dict:
    out = {}
    for axis, table in menu.items():
        total = sum(w for _, w in table)
        h = hashlib.sha256(f"{uid}:{attempt}|{axis}".encode()).digest()
        u = int.from_bytes(h[:8], "big") / 2 ** 64 * total
        acc = 0.0
        for value, w in table:
            acc += w
            if u < acc:
                out[axis] = value
                break
        else:
            out[axis] = table[-1][0]
    return out


def _sandbox(args, task, extra, tag: str):
    sb = Sandbox(task["train"], None, python=args.sandbox_python, timeout=args.tool_timeout,
                 max_output_chars=args.tool_max_output_chars,
                 extra_ns={"test_inputs": [t["input"] for t in task["test"]], "notes": [], **extra})
    notes_dir = Path(args.out) / "notes"
    notes_dir.mkdir(exist_ok=True)
    sb.notes_path = notes_dir / f"{task['id']}_{tag}.json"   # notes 鏡像（見 sync_notes / sb_run）
    sb.notes_mirror: list[str] = []
    sb.notes_synced = 0
    sb.notes_restored = -1
    return sb


def _with_gen_budget(args, gen_budget: int):
    """args 的淺拷貝，只蓋掉這個 session 的 compact_gen_budget（chat / count_tokens 等 closure 共用，同 main_async 的 targs）。
    run_hop / run_episode / prompts 讀預算全走 args.compact_gen_budget，所以把拷貝當 args 傳進去就夠，不改簽名。"""
    assert gen_budget > 0, gen_budget
    a = copy.copy(args)
    a.compact_gen_budget = gen_budget
    return a


async def run_hop(client, args, task, k: int, hop: int, sb: Sandbox, prev: Episode | None,
                  live: asyncio.Semaphore) -> Episode:
    """explorer k 的第 hop 個 session，sandbox 跨 session 共用。hop>1 時 prompt 帶上一個 session 的報告、
    harness 剛跑的 `transform` 檢查、與 sandbox 裡的 `notes`（v4：把長鏈思考切成幾個 50k 視窗）。"""
    n_test = len(task["test"])
    test_inputs = [t["input"] for t in task["test"]]
    system = prompts.explorer_system(n_test, args.compact_gen_budget if args.compact else args.ctx_cap,
                                     args.max_turn_tokens, args.tool_timeout, args.tool_max_output_chars,
                                     variant=args.prompt_variant, notes_structured=args.notes_structured)
    if args.compact:
        system += prompts.COMPACT_SYSTEM_NOTE
    if args.delegate:
        system += prompts.DELEGATE_SYSTEM_NOTE
    async with live:
        if hop == 1:
            assert prev is None
            user = prompts.render_explorer_user(task["train"], test_inputs)
        else:
            assert prev is not None and prev.forced_handoff
            # 報告裡的 CODE 就是接手者會讀到的 `transform`：把它定義回 sandbox，文字與物件才一致。
            assert prev.home_check is not None
            info = dict(prev.report, stop_reason=prev.stop_reason, check=prev.home_check, notes=prev.notes,
                        notes_structured=args.notes_structured)
            user = prompts.render_continue_user(task["train"], test_inputs, hop, args.max_hops, info)
        ep = await run_episode(client, args, label=f"explorer {k} session {hop}", system=system, user=user,
                               sb=sb, tool=args.tool, tool_name=args.tool_name, n_test=n_test, lineage=k, hop=hop,
                               task_id=task["id"], task=task)
        ep.notes = await read_notes(sb, args.tool_max_output_chars)
        # 報告裡的 CODE 定義回 sandbox 並跑 train/test：接手者讀到的 `transform` 與文字一致，refiner 拿到的
        # 檢查結果也是在 helper 都在的環境跑出來的。
        ep.home_check = await _check_transform(sb, ep.report["code"], "transform", n_test, in_globals=True)
        if args.prompt_variant == "alt":
            # 反假設 `transform_alt`（模型有定義的話）也在自家 sandbox 跑 train/test：refiner 看得到、全過的進 candidates。
            ep.alt_check = await _check_transform(sb, None, "transform_alt", n_test)
        return ep


def _all_train_pass(chk: dict | None) -> bool:
    return bool(chk and chk["defined"] and chk["train"] and all(r == "pass" for r in chk["train"]))


def _agreed_answer(finals: list[Episode], n_test: int) -> list | None:
    """≥2 條已答完的 explorer 對每個 test 的第一個答案完全一致：回傳那組答案（每個 test 一個 grid），否則 None。
    v2 裡答完的 explorer 114/126 是對的，兩條獨立答完又一致就不必再花 session 在其他條上
    （reap128 final 120 題：一致 71 題、對 62 題；answered-only 與不分停止原因的數字相同）。"""
    done = [ep for ep in finals if ep.stop_reason == "answered" and all(ep.answers[t] for t in range(n_test))]
    for i in range(len(done)):
        for j in range(i + 1, len(done)):
            if all(done[i].answers[t][0] == done[j].answers[t][0] for t in range(n_test)):
                return [done[i].answers[t][0] for t in range(n_test)]
    return None


def _lineages_converged(finals: list[Episode], n_test: int) -> bool:
    return _agreed_answer(finals, n_test) is not None


async def run_explorers(client, args, task, live: asyncio.Semaphore) -> tuple[list[Episode], list[Episode]]:
    """K 條 explorer，各自最多 max_hops 個 session。前 base_explorers 條先跑；它們的最終答案已有兩條一致就收工，
    否則才開剩下的條（v5：v4 裡前兩條一致的 56 題沒有一題一致而錯，第 3 條在那裡是純浪費）。
    回傳 (每條的最後一個 session, 全部 session 依序)。"""
    n_test = len(task["test"])
    K = args.n_explorers
    base = min(args.base_explorers, K)
    all_eps: list[Episode] = []
    finals: list[Episode | None] = [None] * K

    async def run_group(ks: list[int], sbs: dict[int, Sandbox], stack: contextlib.ExitStack):
        for k in ks:
            sbs[k] = stack.enter_context(_sandbox(args, task, {}, f"e{k}"))
        active = list(ks)
        for hop in range(1, args.max_hops + 1):
            if not active:
                break
            # 預算收縮（不取消 in-flight）：每個 session 的 gen 預算 = min(原值, lineage 剩餘, task 剩餘平分給這批)。
            # 撞到預算照原本 handoff:gen_budget 路徑寫交接報告，答案照舊由 home_check 收。
            launch = [(k, session_budget(k)) for k in active]
            skipped = [k + 1 for k, gb in launch if gb < args.min_session_budget]
            launch = [(k, gb) for k, gb in launch if gb >= args.min_session_budget]
            if launch:
                share = (args.task_budget - spent()) // len(launch)   # 同一個 gather 的 session 看到同一個 spent()：平分
                launch = [(k, min(gb, share)) for k, gb in launch]
                skipped += [k + 1 for k, gb in launch if gb < args.min_session_budget]
                launch = [(k, gb) for k, gb in launch if gb >= args.min_session_budget]
            if skipped:
                print(f"·· {task['id']}: hop {hop} skipped for explorer(s) {skipped} "
                      f"(session budget below {args.min_session_budget // 1000}k)", flush=True)
            if not launch:
                break
            eps = await asyncio.gather(*(run_hop(client, _with_gen_budget(args, gb), task, k + 1, hop, sbs[k], finals[k], live)
                                         for k, gb in launch))
            for (k, _), ep in zip(launch, eps):
                finals[k] = ep
                all_eps.append(ep)
            active = [k for k, _ in launch]
            # v9：深的 session 只給已有全過 train 的 transform 的條——v8 的 hop 4 從「上一 session 全過」出發 16/45 答對，
            # 從沒全過出發 2/47；後者一個 session 35k，砍掉省 ~14k/題。
            active = [k for k in active if finals[k].forced_handoff
                      and (hop + 1 < args.pass_gate_hop or _all_train_pass(finals[k].home_check))]
            if active and _lineages_converged([e for e in finals if e is not None], n_test):
                active = []
            if active and spent() >= args.task_budget:
                # v10：這題的 explorer 已經花掉預算就不再開 session——v7/v8 答錯的題 265–342k、答對的 ≈ baseline，
                # 多出來的 token 全在最後沒解開的題上。
                active = []
            if active and _cutoff(args) is not None:
                active = []

    def spent() -> int:
        return sum(e.usage("completion_tokens") for e in all_eps)

    def lineage_spent(k: int) -> int:
        return sum(e.usage("completion_tokens") for e in all_eps if e.lineage == k + 1)

    def session_budget(k: int) -> int:
        gb = args.compact_gen_budget
        if args.lineage_budget:
            gb = min(gb, args.lineage_budget - lineage_spent(k))
        return gb

    with contextlib.ExitStack() as stack:
        sbs: dict[int, Sandbox] = {}
        await run_group(list(range(base)), sbs, stack)
        assert all(finals[k] is not None for k in range(base)), "base explorers 的第一個 session 一定要開（parse_args 保證預算夠）"
        # 第 3 條（base..K）只在前面幾條「分歧」或「都沒過 train」時才開：一致而錯的很少（v4 0/56、v5 7/72），
        # 但兩條都答完、都沒過 train 又一致的答案不值得信。deadline / abort 時不開。
        done_finals = [e for e in finals if e is not None]
        if (base < K and spent() < args.task_budget and _cutoff(args) is None
                and (not _lineages_converged(done_finals, n_test)
                     or not any(_all_train_pass(e.home_check) for e in done_finals))):
            await run_group(list(range(base, K)), sbs, stack)
    finals = [e for e in finals if e is not None]
    assert len(finals) >= base
    return finals, all_eps


async def run_refine(client, args, task, reports: list[dict], round_no: int, live: asyncio.Semaphore):
    """回傳 (episode, checks, candidates)。refiner 的生成預算由 solve_task 用 _with_gen_budget 帶進來。"""
    n_test = len(task["test"])
    test_inputs = [t["input"] for t in task["test"]]
    async with live:
        with _sandbox(args, task, {"candidates": []}, "refine") as sb:
            checks = await verify_reports(sb, reports, n_test)
            cands = build_candidates(reports, checks, n_test)
            cand_ns = [{k: v["grid"] for k, v in c.items()} for c in cands]
            out, _ = await sb_run(sb, f"candidates = {json.dumps(cand_ns)}\nprint('ok')")
            if "ok" not in out:
                raise RuntimeError(f"failed to set candidates in sandbox: {out[-300:]}")
            names = [c["name"] for c in checks if c["defined"]]
            system = prompts.refiner_system(n_test, args.compact_gen_budget if args.compact else args.ctx_cap,
                                            args.max_turn_tokens, args.tool_timeout,
                                            args.tool_max_output_chars, names, variant=args.prompt_variant,
                                            notes_structured=args.notes_structured)
            user = prompts.render_refiner_user(task["train"], test_inputs, reports, checks, cands, args.inline_cells,
                                               variant=args.prompt_variant)
            ep = await run_episode(client, args, label=f"refiner round {round_no}", system=system, user=user, task_id=task["id"],
                                   sb=sb, tool=args.tool, tool_name=args.tool_name, n_test=n_test)
            ep.notes = await read_notes(sb, args.tool_max_output_chars)   # refiner 的 notes 之前沒讀回（審核：0/4,499）
    return ep, checks, cands


async def run_verify(args, task, reports: list[dict], live: asyncio.Semaphore):
    """--skip-refine-when-agree：只開 refine sandbox 跑 verify_reports + build_candidates，不跑 refiner LLM。
    回傳 (checks, candidates)，格式與 run_refine 的後兩項相同，summarize() 的 rec["checks"][0] 才有東西。"""
    n_test = len(task["test"])
    async with live:
        with _sandbox(args, task, {"candidates": []}, "refine") as sb:
            checks = await verify_reports(sb, reports, n_test)
    return checks, build_candidates(reports, checks, n_test)


def _report_for_refiner(ep: Episode) -> dict:
    rep = dict(ep.report)
    rep["label"] = ep.label
    rep["stop_reason"] = ep.stop_reason
    rep["answers"] = ep.answers
    rep["notes"] = ep.notes
    rep["home_check"] = ep.home_check
    rep["alt_check"] = ep.alt_check
    rep["notes_structured"] = ep.notes_structured
    return rep


async def solve_task(client, args, task, live: asyncio.Semaphore) -> dict:
    t0 = time.monotonic()
    n_test = len(task["test"])
    finals, explorers = await run_explorers(client, args, task, live)
    reports = [_report_for_refiner(ep) for ep in finals]
    # 先用 explorer 自家 sandbox 的檢查組一份候選、寫進 submission：refiner 還要再跑一條 ≤ cap 的對話，
    # 到點被砍時這一格已經有東西（票數最高的先）。refiner 收尾後會覆寫。
    home_checks = _checks_from_home(reports)
    home_cands = build_candidates(reports, home_checks, n_test)
    if args.submission is not None:
        for t in range(n_test):
            args.submission.put_attempts(task["id"], t, [c["grid"] for c in home_cands[t].values()])
    refines: list[Episode] = []
    all_checks, all_cands = [], []
    final_attempts: list[list] = [[] for _ in range(n_test)]
    final_src: list[list[str]] = [[] for _ in range(n_test)]
    # 兩條 explorer 都自己答完且每個 test 的第一個答案一致，就不開 refiner：teacher 上「一致而錯」v4 0/56、v5 7/72，
    # 而 refiner 是一整條 ≤ cap 的對話（sweep 實測 36–55k）。預算緊時這是最便宜的一刀；attempt 2 由候選補位。
    refine_skipped = ""
    agreed = _agreed_answer(finals, n_test) if args.skip_refine_when_agree else None
    if agreed is not None:
        refine_skipped = "agree"
        print(f"·· {task['id']}: explorers agree -> refiner skipped", file=sys.stderr)
        # 不跑 refiner LLM，但仍開 refine sandbox 做 verify/candidates：attempt 2 才有依票數排序的候選可補。
        checks, cands = await run_verify(args, task, reports, live)
        all_checks.append(checks)
        all_cands.append(cands)
    for rnd in range(1, args.max_refine_rounds + 1):
        if refine_skipped:
            break
        cut = _cutoff(args)
        if cut is not None:
            refine_skipped = cut                     # deadline / abort：不開 refiner，答案就是上面那份候選
            break
        # task 預算含 explorer 與前幾輪 refine；round 1 一定跑（下面 attempts 選擇需要至少一輪），之後不夠就不 escalate。
        # 預算收縮成剩餘量但不低於 min_session_budget（K3 實測：2×75k 後 refiner 拿 max(min(40k, 剩餘), 20k)）。
        left = args.task_budget - sum(e.usage("completion_tokens") for e in explorers + refines)
        if rnd > 1 and left < args.min_session_budget:
            break
        gb = max(min(args.refine_gen_budget or args.compact_gen_budget, left), args.min_session_budget)
        ep, checks, cands = await run_refine(client, _with_gen_budget(args, gb), task, reports, rnd, live)
        refines.append(ep)
        all_checks.append(checks)
        all_cands.append(cands)
        complete = all(len(ep.answers[t]) >= 1 for t in range(n_test))
        conf = ep.report["confidence"]
        # v2 的 refiner 信心很準（≥80：76/80 對；<50：2/17）；沒答完、被迫交接、或信心不足就再開一輪，
        # 它的報告併入 reports。v2 之前只在「沒答完」時開第二輪，表達式 ANSWER 讓這幾乎不發生（rounds/task 1.00）。
        weak = (not complete) or ep.forced_handoff or conf is None or conf < args.escalate_below
        if not weak or rnd == args.max_refine_rounds:
            break
        reports.append(_report_for_refiner(ep))
    if refines:
        # 最終 attempts：信心最高的一輪（同分取後面的）的 ATTEMPT 1 先；其他輪的 ATTEMPT 1 若不同就當第 2 個
        # ——兩輪是兩次獨立的判斷，這種分歧比同一條鏈自己寫的 ATTEMPT 2 更值得一個名額；再來才是最佳輪的 ATTEMPT 2。
        order = sorted(range(len(refines)), key=lambda i: ((refines[i].report["confidence"] or -1), i), reverse=True)
        for t in range(n_test):
            ranked = [(refines[i].answers[t][0], f"refine{i + 1}") for i in order if refines[i].answers[t]]
            best = refines[order[0]].answers[t]
            if len(best) > 1:
                ranked.append((best[1], f"refine{order[0] + 1}:a2"))
            for g, src in ranked:
                if g not in final_attempts[t] and len(final_attempts[t]) < 2:
                    final_attempts[t].append(g)
                    final_src[t].append(src)
    elif agreed is not None:
        # 一致的答案就是 attempt 1；attempt 2 由下面的候選補位（票數次高的）。
        for t in range(n_test):
            final_attempts[t].append(agreed[t])
            final_src[t].append("agreed")
    # 其餘（deadline / abort 沒開 refiner）：全靠下面的候選補位。
    # candidates 補位：refine 沒答的 test 用票數最高的；--fill-attempt2 補第二個。
    # refiner 一輪都沒跑（deadline）就只有 explorer 自家檢查出來的候選。
    cands = all_cands[-1] if all_cands else home_cands
    for t in range(n_test):
        want = 2 if (args.fill_attempt2 or not final_attempts[t]) else len(final_attempts[t])
        for key, c in cands[t].items():
            if len(final_attempts[t]) >= want:
                break
            if c["grid"] not in final_attempts[t]:
                final_attempts[t].append(c["grid"])
                final_src[t].append(f"candidate:{key}")
    # 比賽的 hidden test 沒有 output：計分欄位全部 None（不是 False——False 會被 summarize 當成答錯算進分母）。
    scored = all("output" in t for t in task["test"])
    expected = [t.get("output") for t in task["test"]]
    per_test = []
    for t in range(n_test):
        att = final_attempts[t]
        if args.submission is not None:
            args.submission.put_attempts(task["id"], t, att)
        row = {"test_idx": t, "attempts": att, "attempt_src": final_src[t], "n_candidates": len(cands[t])}
        if scored:
            row.update({
                "correct": any(g == expected[t] for g in att),
                "correct_attempt1": bool(att) and att[0] == expected[t],
                "correct_refine_only": any(g == expected[t] for g, s in zip(att, final_src[t]) if not s.startswith("candidate")),
                "correct_any_explorer": any(g == expected[t] for ep in explorers for g in ep.answers[t]),
                "correct_any_candidate": any(c["grid"] == expected[t] for c in cands[t].values()),
            })
        else:
            row.update({k: None for k in ("correct", "correct_attempt1", "correct_refine_only",
                                          "correct_any_explorer", "correct_any_candidate")})
        per_test.append(row)
    eps = explorers + refines
    rec = {
        "task_id": task["id"], "n_test": n_test, "n_train": len(task["train"]), "dp_rank": args.dp_rank,
        "refine_skipped": refine_skipped,
        "explorers": [ep.to_dict(args.save_messages) for ep in explorers],
        "refines": [ep.to_dict(args.save_messages) for ep in refines],
        "checks": all_checks or [home_checks],
        "candidates": [[{k: {"sources": v["sources"], "confs": v["confs"], "shape": [len(v["grid"]), len(v["grid"][0])]}
                         for k, v in c.items()} for c in cands_r] for cands_r in (all_cands or [home_cands])],
        "results": per_test,
        "tok_prompt": sum(e.usage("prompt_tokens") for e in eps),
        "tok_completion": sum(e.usage("completion_tokens") for e in eps),
        "tok_reasoning": sum(e.usage("reasoning_tokens") for e in eps),
        "tok_cached": sum(e.usage("cached_tokens") for e in eps),
        "peak_total": max(e.peak_total for e in eps),
        "n_turns": sum(len(e.calls) for e in eps),
        "wall_s": round(time.monotonic() - t0, 1),
    }
    return rec


def write_task(out: Path, rec: dict):
    (out / "tasks").mkdir(exist_ok=True)
    (out / "pairs").mkdir(exist_ok=True)
    # --save-segments：segments 很大（每段是模型當時的整份 messages），不放進 tasks/*.json（summarize / compare 會整批讀），
    # 另存 segments/<task>.jsonl.gz，一行一段。
    segs = []
    for kind in ("explorers", "refines"):
        for ep in rec[kind]:
            eps = ep.pop("segments", None)
            if eps:
                for i, m in enumerate(eps):
                    segs.append({"label": ep["label"], "lineage": ep["lineage"], "hop": ep["hop"],
                                 "segment_idx": i, "n_segments": len(eps), "stop_reason": ep["stop_reason"], "messages": m})
    if segs:
        (out / "segments").mkdir(exist_ok=True)
        tmp = out / "segments" / f"{rec['task_id']}.jsonl.gz.tmp"
        with gzip.open(tmp, "wt", encoding="utf-8", compresslevel=6) as f:
            for sg in segs:
                f.write(json.dumps(sg, ensure_ascii=False) + "\n")
        tmp.rename(out / "segments" / f"{rec['task_id']}.jsonl.gz")
    tmp = out / "tasks" / f"{rec['task_id']}.json.tmp"
    tmp.write_text(json.dumps(rec, ensure_ascii=False))
    tmp.rename(out / "tasks" / f"{rec['task_id']}.json")
    n = rec["n_test"]
    for r in rec["results"]:
        pair = {
            "uid": f"{rec['task_id']}#{r['test_idx']}", "task_id": rec["task_id"], "test_idx": r["test_idx"],
            **{k: r[k] for k in ("attempts", "attempt_src", "correct", "correct_attempt1", "correct_refine_only",
                                 "correct_any_explorer", "correct_any_candidate", "n_candidates")},
            # token 計量按 test 均分，讓 score_partial.py 加總後等於整題。
            **{k: rec[k] / n for k in ("tok_prompt", "tok_completion", "tok_reasoning", "tok_cached")},
            "n_turns": rec["n_turns"] / n,
        }
        (out / "pairs" / f"{rec['task_id']}_{r['test_idx']}.json").write_text(json.dumps(pair))


def _fmt_task_line(rec: dict) -> str:
    def _hop(e):
        conf = e["report"]["confidence"]
        return f"{e['stop_reason'].split(':')[0][:6]} c={conf if conf is not None else '-'} {e['tok_completion'] // 1000}k"
    by_lin: dict[int, list] = {}
    for e in rec["explorers"]:
        by_lin.setdefault(e["lineage"], []).append(e)
    ex = " ".join(f"E{k}[" + " > ".join(_hop(e) for e in eps) + "]" for k, eps in sorted(by_lin.items()))
    rf = " ".join(f"R{i + 1}[{e['stop_reason'].split(':')[0][:6]} {e['tok_completion'] // 1000}k]" for i, e in enumerate(rec["refines"]))
    if rec.get("refine_skipped"):
        rf = "R[agreed]" if rec["refine_skipped"] == "agree" else f"R[skipped:{rec['refine_skipped']}]"
    res = "".join("?" if r["correct"] is None else "✓" if r["correct"] else "✗" for r in rec["results"])
    expl = "".join("?" if r["correct_any_explorer"] is None else "✓" if r["correct_any_explorer"] else "✗"
                   for r in rec["results"])
    return (f"{rec['task_id']} n_test={rec['n_test']} {res} (explorers {expl}) {ex} {rf} "
            f"gen={rec['tok_completion'] // 1000}k peak={rec['peak_total'] // 1000}k {rec['wall_s']:.0f}s")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

async def main_async(args):
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    (out / "args.json").write_text(json.dumps({k: v for k, v in vars(args).items()
                                              if k not in ("tool", "sampling", "submission", "should_abort",
                                                           "chat", "count_tokens", "annotate_ids")},
                                             indent=1, default=str))
    if args.task_source == "nvarc":
        # 可以給多個 corpus（nvarc_training + nvarc_full 混著跑）；每題記 source = 檔名去掉 .jsonl.zst，落地在 tasks/*.json。
        corpus: dict[str, list] = {}
        source: dict[str, str] = {}
        for path in args.corpus:
            name = Path(path).name.split(".")[0]
            part = load_nvarc_corpus(path)
            dup = set(part) & set(corpus)
            assert not dup, f"{len(dup)} uids in more than one corpus, e.g. {sorted(dup)[:3]}"
            corpus.update(part)
            source.update({u: name for u in part})
            print(f"nvarc corpus {name}: {len(part)} uids", flush=True)
        task_ids = ([l.strip() for l in Path(args.task_list).read_text().splitlines() if l.strip()]
                    if args.task_list else list(corpus))
        missing = [t for t in task_ids if t not in corpus]
        assert not missing, f"{len(missing)} uids not in corpus, e.g. {missing[:3]}"
        get_task = lambda tid: dict(nvarc_task(tid, corpus[tid], args.attempt, args.n_train, args.n_test), source=source[tid])
        print(f"nvarc: {len(corpus)} uids total; attempt={args.attempt} n_train={args.n_train} n_test={args.n_test}", flush=True)
    elif args.task_source == "kaggle":
        challenges = load_challenges(Path(args.challenges))
        task_ids = ([l.strip() for l in Path(args.task_list).read_text().splitlines() if l.strip()]
                    if args.task_list else list(challenges))
        missing = [t for t in task_ids if t not in challenges]
        assert not missing, f"{len(missing)} task ids not in {args.challenges}, e.g. {missing[:3]}"
        get_task = lambda tid: {"id": tid, "train": challenges[tid]["train"], "test": challenges[tid]["test"]}
        n_truth = sum(all("output" in t for t in challenges[u]["test"]) for u in task_ids)
        print(f"kaggle: {len(challenges)} tasks in {args.challenges}; {n_truth}/{len(task_ids)} with ground truth",
              flush=True)
    else:
        task_ids = [l.strip() for l in Path(args.task_list).read_text().splitlines() if l.strip()]
        get_task = lambda tid: load_task(Path(args.task_dir), tid)
    if args.tasks:
        want = set(args.tasks.split(","))
        task_ids = [t for t in task_ids if t in want]
    if args.limit:
        task_ids = task_ids[: args.limit]
    done = {p.stem for p in (out / "tasks").glob("*.json")} if args.resume else set()
    todo = [t for t in task_ids if t not in done]
    print(f"{len(task_ids)} tasks, {len(done)} already done, {len(todo)} to run; "
          f"explorers={args.n_explorers} ctx_cap={args.ctx_cap} live={args.max_live}", flush=True)
    if args.submission_out:
        # 注意：--limit / --tasks 在這之前就截斷了題目，所以 submission.json 只會有這些題的 key。
        args.submission = SubmissionStore({tid: {"test": get_task(tid)["test"]} for tid in task_ids},
                                          Path(args.submission_out))
        for tid in sorted(done):
            rec = json.loads((out / "tasks" / f"{tid}.json").read_text())
            for r in rec["results"]:
                args.submission.put_attempts(tid, r["test_idx"], r["attempts"])
        print(f"submission -> {args.submission_out}: {args.submission.n_cells} cells, "
              f"{len(args.submission.filled)} restored from {len(done)} done tasks", flush=True)
    if args.deadline is not None:
        print(f"deadline: {args.deadline_min:.1f} min from start "
              f"(per-turn max_tokens shrinks at {args.deadline_tok_per_s:g} tok/s near the end)", flush=True)
    # 伺服器死掉時每一題都以 ConnectError 收場；沒有這個護欄就會安靜地把 submission 留在 fallback 然後正常結束。
    health = {"ok": 0, "errors": 0, "tripped": ""}
    args.should_abort = lambda: bool(health["tripped"])
    dp_load = collections.Counter()
    skipped: list[str] = []

    live = asyncio.Semaphore(args.max_live)
    limits = httpx.Limits(max_connections=args.max_live + 64, max_keepalive_connections=args.max_live + 64)
    t_start = time.monotonic()
    n_done = 0
    async with httpx.AsyncClient(timeout=args.http_timeout, limits=limits) as client:
        # 一次把 120 題全丟進 gather 時，semaphore 是 FIFO：240 條 base explorer 排在所有 refiner 前面，
        # 題目要等它們全部起跑才會開始完成（compact 模式 session 長，第一題要 3 小時才出現）。
        # --task-live 限制同時在跑的題數，完成進度就平滑、--resume 也少丟。
        task_live = asyncio.Semaphore(args.task_live) if args.task_live > 0 else contextlib.nullcontext()

        def _check_health():
            total = health["ok"] + health["errors"]
            rate = health["errors"] / max(total, 1)
            if (args.max_error_rate > 0 and not health["tripped"] and total >= args.error_grace
                    and rate > args.max_error_rate):
                health["tripped"] = (f"{health['errors']}/{total} tasks failed "
                                     f"({100 * rate:.1f}% > {100 * args.max_error_rate:.1f}%)")
                print(f"\n{'=' * 70}\nABORT: {health['tripped']}\n  伺服器多半已經死了。停止發新的題目，"
                      f"在飛的對話不再呼叫模型。\n{'=' * 70}\n", flush=True)

        async def one(tid):
            nonlocal n_done
            task = get_task(tid)
            targs, hp = copy.copy(args), None    # 淺拷貝：chat / count_tokens / submission 等共用，只蓋掉每題的東西
            if args.hparam_menu:
                hp = draw_hparams(tid, args.attempt)
                vars(targs).update(hp)
            rank = None
            try:
                async with task_live:
                    if _cutoff(args) is not None:
                        skipped.append(tid)      # deadline 到了 / 已中止：沒開始的題不開，submission 留 fallback
                        return None
                    if args.fixed_dp_rank is not None:
                        # 單 rank 收資料：即使 server 是 DP2 也全部釘死，另一個 pool 空著。
                        rank = args.fixed_dp_rank
                        dp_load[rank] += 1
                    elif args.dp_size > 1:
                        # 拿到名額才選 rank：以在飛的題數最少者為準（同分取小），explorer/refiner 都跟著這題走。
                        rank = min(range(args.dp_size), key=lambda r: (dp_load[r], r))
                        dp_load[rank] += 1
                    targs.dp_rank = rank
                    rec = await solve_task(client, targs, task, live)
            except Exception as e:  # 一題炸掉不該連坐其他 119 題，但要大聲記下來
                n_done += 1
                health["errors"] += 1
                _check_health()
                print(f"!! {tid} FAILED: {type(e).__name__}: {e}", flush=True)
                (out / "failures.jsonl").open("a").write(json.dumps({"task_id": tid, "error": repr(e)}) + "\n")
                raise
            finally:
                if rank is not None:
                    dp_load[rank] -= 1
            health["ok"] += 1
            rec["attempt"] = args.attempt
            rec["hparams"] = hp
            if "train_idx" in task:
                rec["train_idx"], rec["test_idx"] = task["train_idx"], task["test_idx"]
            rec["source"] = task.get("source", "arc")
            write_task(out, rec)
            n_done += 1
            print(f"[{n_done}/{len(todo)} {time.monotonic() - t_start:6.0f}s] " + _fmt_task_line(rec), flush=True)

        results = await asyncio.gather(*(one(t) for t in todo), return_exceptions=True)
    n_fail = sum(1 for r in results if isinstance(r, Exception))
    summarize(out)
    if skipped:
        print(f"{len(skipped)} tasks never started (deadline / abort): {skipped[:8]}{' ...' if len(skipped) > 8 else ''}",
              flush=True)
    if args.submission is not None:
        sub = args.submission
        print(f"submission: {len(sub.filled)} / {sub.n_cells * 2} attempt cells filled (rest = test input fallback) "
              f"-> {sub.out}", flush=True)
        if sub.rejected:
            print(f"  ⚠️  {len(sub.rejected)} answers were not valid grids and were NOT written: {sub.rejected[:5]}",
                  flush=True)
    if health["tripped"]:
        # 刻意不當成「跑完」：submission 裡多數格子是 fallback，交出去會被當成模型的 0 分。
        print(f"\nFAILED: {health['tripped']}", flush=True)
        sys.exit(2)
    if n_fail:
        print(f"!! {n_fail} tasks failed; see failures.jsonl", flush=True)
        if args.task_source != "kaggle":
            sys.exit(1)
        # kaggle：個別題目失敗不能讓 notebook 以非零碼收場（code competition 的 notebook 出錯 = 整份提交作廢），
        # 失敗的格子留 fallback；只有錯誤率超標（上面）才中止。


def summarize(out: Path):
    recs = [json.loads(p.read_text()) for p in sorted((out / "tasks").glob("*.json"))]
    if not recs:
        print("no tasks finished")
        return
    n = len(recs)
    pairs = [r for rec in recs for r in rec["results"]]
    # kaggle 模式（hidden test 沒有答案）的 correct 是 None：那些題不進分數的分母，一題都沒有就不印分數。
    scored = [rec for rec in recs if all(r["correct"] is not None for r in rec["results"])]

    def partial(key):
        return sum(sum(1 for r in rec["results"] if r[key]) / rec["n_test"] for rec in scored) / len(scored) * 100

    def strict(key):
        return sum(all(r[key] for r in rec["results"]) for rec in scored) / len(scored) * 100

    lines = [f"== {out.name}: {n} tasks, {len(pairs)} test pairs =="]
    if scored:
        lines += [f"partial pass@2 (final attempts)     {partial('correct'):6.2f}%   strict {strict('correct'):6.2f}%"
                  + (f"   (scored {len(scored)}/{n} tasks)" if len(scored) < n else ""),
                  f"partial pass@1 (attempt 1 only)      {partial('correct_attempt1'):6.2f}%",
                  f"partial refine-only (no candidate fill) {partial('correct_refine_only'):6.2f}%",
                  f"partial any-explorer-answer          {partial('correct_any_explorer'):6.2f}%   (oracle over explorer answers)",
                  f"partial any-candidate                {partial('correct_any_candidate'):6.2f}%   (oracle over candidates)"]
    else:
        lines.append("no ground truth (kaggle hidden test): pass@k not computed")
    lines += [f"gen tokens/task: mean {sum(r['tok_completion'] for r in recs) / n / 1000:.1f}k   "
             f"peak context max {max(r['peak_total'] for r in recs) / 1000:.1f}k   "
             f"cache hit {100 * sum(r['tok_cached'] for r in recs) / max(1, sum(r['tok_prompt'] for r in recs)):.1f}%   "
             f"wall/task mean {sum(r['wall_s'] for r in recs) / n:.0f}s"]
    if any(rec.get("dp_rank") is not None for rec in recs):
        lines.append(f"dp_rank distribution: {dict(collections.Counter(rec.get('dp_rank') for rec in recs))}")
    stops = {}
    for rec in recs:
        for e in rec["explorers"]:
            stops[e["stop_reason"]] = stops.get(e["stop_reason"], 0) + 1
    rstops = {}
    for rec in recs:
        for e in rec["refines"]:
            rstops[e["stop_reason"]] = rstops.get(e["stop_reason"], 0) + 1
    n_lin = sum(len({e["lineage"] for e in rec["explorers"]}) for rec in recs)
    lines.append(f"explorer stops: {stops}   sessions/explorer mean "
                 f"{sum(len(rec['explorers']) for rec in recs) / n_lin:.2f}")
    lines.append(f"refiner stops:  {rstops}   refine rounds/task mean {sum(len(r['refines']) for r in recs) / n:.2f}"
                 f"   skipped (explorers agree): {sum(1 for r in recs if r.get('refine_skipped') == 'agree')}")
    eps = [e for rec in recs for e in rec["explorers"] + rec["refines"]]
    if json.loads((out / "args.json").read_text()).get("compact"):
        lines.append(f"compactions: {sum(e['n_compactions'] for e in eps)} over {len(eps)} episodes   "
                     f"episodes compacted: {sum(1 for e in eps if e['n_compactions'])}   "
                     f"dropped tokens/episode: {sum(e['compact_dropped_tokens'] for e in eps) / max(len(eps), 1) / 1000:.1f}k   "
                     f"checkpoints: {sum(e['n_checkpoints'] for e in eps)}   "
                     f"outputs cleared: {sum(e['n_outputs_cleared'] for e in eps)}")
    if json.loads((out / "args.json").read_text()).get("delegate"):
        subs = [d for e in eps for d in e["sub_episodes"]]
        lines.append(f"delegations: {sum(e['n_delegations'] for e in eps)} over {len(eps)} episodes   "
                     f"episodes delegating: {sum(1 for e in eps if e['n_delegations'])}   "
                     f"helper gen/delegation: {sum(d['tok_completion'] for d in subs) / max(len(subs), 1) / 1000:.1f}k   "
                     f"helper stops: {collections.Counter(d['stop_reason'] for d in subs)}")
    lines.append(f"continuations: {sum(e['n_continuations'] for e in eps)} over {len(eps)} episodes   "
                 f"code passes all train pairs: explorers "
                 f"{sum(1 for rec in recs for c in rec['checks'][0] if c['train'] and all(x == 'pass' for x in c['train']))}"
                 f"/{sum(len(rec['checks'][0]) for rec in recs)}   "
                 f"sandbox restarts: {sum(e.get('n_sandbox_restarts', 0) for e in eps)}")
    if json.loads((out / "args.json").read_text())["prompt_variant"] == "alt":
        lines.append(_alt_summary(recs))
    text = "\n".join(lines)
    print(text, flush=True)
    (out / "summary.txt").write_text(text + "\n")


def _alt_summary(recs: list[dict]) -> str:
    """alt 變體的專屬計量：有寫 ALTERNATIVE 的 explorer session 數、其 `transform_alt` 全過 train 的數、
    refiner ATTEMPT 2 的來源（transform_alt 表達式 / 抄既有 candidate / 自造 grid；v8 自造的 36 個只對 1 個）。"""
    ex = [e for rec in recs for e in rec["explorers"]]
    n_alt = sum(1 for e in ex if e["report"]["alt_code"])   # 有 code 的才算：模型常寫 "none (因為…)" 交代為何沒有
    n_alt_pass = sum(1 for e in ex if _all_train_pass(e["alt_check"]))
    src = {"transform_alt": 0, "copy": 0, "novel": 0}
    for rec in recs:
        finals = {e["lineage"]: e for e in rec["explorers"]}   # session 依 hop 順序存，後面的蓋掉前面的
        for r in rec["results"]:
            t = r["test_idx"]
            grids = [g for e in finals.values() for g in e["answers"][t]]
            for e in finals.values():
                if e["home_check"]["defined"] and e["home_check"]["test"][t]["grid"] is not None:
                    grids.append(e["home_check"]["test"][t]["grid"])
                if _all_train_pass(e["alt_check"]) and e["alt_check"]["test"][t]["grid"] is not None:
                    grids.append(e["alt_check"]["test"][t]["grid"])
            for i, s in enumerate(r["attempt_src"]):
                m = re.fullmatch(r"refine(\d+):a2", s)
                if not m:
                    continue
                kind, body = rec["refines"][int(m.group(1)) - 1]["report"]["answer_blocks"][f"{t}:1"]
                src["transform_alt" if kind == "expr" and "transform_alt" in body else
                    "copy" if r["attempts"][i] in grids else "novel"] += 1
    return (f"alt variant: explorer sessions with ALTERNATIVE {n_alt}/{len(ex)}, transform_alt passes all train pairs "
            f"{n_alt_pass}; refiner attempt-2 source {src}")


def parse_args():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--task-source", choices=["arc", "nvarc", "kaggle"], default="arc",
                    help="arc：--task-dir/<id>.json；nvarc：--corpus 的 jsonl.zst，用 choose_pairs 切 train/test")
    ap.add_argument("--task-dir", default="")
    ap.add_argument("--challenges", default="",
                    help="--task-source kaggle：比賽的 arc-agi_test_challenges.json，或 <task_id>.json 的目錄"
                         "（本地 dry run；test 帶 output 就算分）")
    ap.add_argument("--submission-out", default="",
                    help="邊跑邊寫 ARC Prize 的 submission.json：建檔先用 test input 填滿，explorer 收完先寫候選、"
                         "refiner 收完覆寫，每次整份重寫")
    ap.add_argument("--deadline-min", type=float, default=0.0,
                    help="從現在起算的分鐘數。到點不再呼叫模型（含交接報告），explorer 的答案改從 sandbox 現有的 "
                         "transform 收、refiner 不開；沒開始的題不開。0 = 不設限")
    ap.add_argument("--deadline-tok-per-s", type=float, default=30.0,
                    help="接近 deadline 時單輪 max_tokens 的上限 = 剩餘秒數 × 這個值（每條 stream 的保守 decode 速率；"
                         "Kaggle L4 每 replica 4 條時每條 60–90 tok/s，12 條時估 ~30）")
    ap.add_argument("--dp-size", type=int, default=1,
                    help="server 的 --dp-size；>1 時每題的所有 episode 釘同一個 routed_dp_rank（sglang 的 DP 沒有 "
                         "session affinity，不釘就每輪重 prefill）")
    ap.add_argument("--dp-rank", type=int, default=None, dest="fixed_dp_rank",
                    help="釘全部題到這個 routed_dp_rank。不設且 --dp-size>1 時才按在飛最少分配。"
                         "DP attention 就算只用一個 rank 也要帶，否則 round-robin 跨 rank、前綴快取全空")
    ap.add_argument("--refine-gen-budget", type=int, default=0,
                    help="compact 模式下 refiner 的生成預算；0 = 同 --compact-gen-budget")
    ap.add_argument("--max-error-rate", type=float, default=0.0,
                    help=">0 時：整題失敗的比例超過它就停止發新題、以非零碼（2）結束。伺服器死掉的護欄")
    ap.add_argument("--error-grace", type=int, default=10, help="至少完成這麼多題才開始檢查錯誤率")
    ap.add_argument("--task-list", default="", help="一行一個 task id / uid；nvarc 不給就是整份語料")
    ap.add_argument("--corpus", nargs="*", default=[], help="--task-source nvarc 的 jsonl.zst，可多個（uid 不能重複）；每題記 source")
    ap.add_argument("--attempt", type=int, default=0, help="nvarc 的 pair 抽樣種子（同 rollout_runner 的 attempt）")
    ap.add_argument("--n-train", type=int, default=4)
    ap.add_argument("--n-test", type=int, default=1)
    ap.add_argument("--hparam-menu", action="store_true",
                    help="每題按 HPARAM_MENU 決定性抽 effort / ctx_cap / K / target_frac / gen_budget，記在 tasks/*.json")
    ap.add_argument("--save-segments", action="store_true",
                    help="每次 compaction 前把整份 messages 存進 segments/<task>.jsonl.gz（蒸餾語料）")
    ap.add_argument("--tasks", default="", help="逗號分隔的 task id 子集")
    ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--out", required=True)
    ap.add_argument("--resume", action="store_true", help="跳過 out/tasks 裡已有的題")
    ap.add_argument("--base-url", required=True)
    ap.add_argument("--model", required=True)
    ap.add_argument("--reasoning-effort", default="xhigh")
    ap.add_argument("--backend", choices=["openai", "dsv4", "tokens"], default="openai",
                    help="dsv4：用模型自帶 encoding 組 prompt 打 /generate（見 dsv4_backend.py）")
    ap.add_argument("--encoding-path", default=None, help="dsv4 backend 用的 encoding_dsv4.py 路徑")
    ap.add_argument("--prompt-variant", choices=list(prompts.VARIANTS), default="default",
                    help="alt：explorer 全過 train 後多一步找反假設（ALTERNATIVE / ATTEMPT 2）、refiner 在 candidates 全是 "
                         "train-pass 時刻意找反例；default 的 prompt 一字不動（見 prompts.py）")
    ap.add_argument("--n-explorers", type=int, default=3)
    ap.add_argument("--base-explorers", type=int, default=2,
                    help="先跑這幾條 explorer；它們的答案已有兩條一致就不開剩下的（設成 n-explorers 即全部並行）")
    ap.add_argument("--task-budget", type=int, default=10**9,
                    help="這題的 gen token 總量（explorer + refine）：到了就不再開新 session / 新 lineage，而且每個新 session 的"
                         "預算收縮成剩餘量（同一批並行的 session 平分）。v7/v8 模擬：省 49k 但砍掉 11 題的唯一正解，不採用；"
                         "reap128 final 120 題：沒解出的 41 題平均 865k、解出的 293k；Kaggle 用 180k")
    ap.add_argument("--lineage-budget", type=int, default=0,
                    help="每條 explorer 跨 session 的生成 token 累計上限（0 = 不限）：每個 session 的 compact_gen_budget = "
                         "min(原值, 剩餘)，不砍正在跑的 session（可超一輪 max_turn_tokens + 交接）")
    ap.add_argument("--min-session-budget", type=int, default=20_000,
                    help="lineage / task 剩餘預算不到這麼多就不開新 session；refine 第 1 輪至少給這麼多")
    ap.add_argument("--pass-gate-hop", type=int, default=99,
                    help="從第幾個 session 起只有 home check 全過 train 的條才能繼續（v9 用 4）")
    ap.add_argument("--max-hops", type=int, default=3,
                    help="每條 explorer 最多幾個 session（交接後在同一個 sandbox 開新 context 接著做）")
    ap.add_argument("--max-refine-rounds", type=int, default=1)
    ap.add_argument("--escalate-below", type=int, default=80,
                    help="refiner 信心低於這個（或沒答完 / 被迫交接）就再開一輪 refine")
    ap.add_argument("--ctx-cap", type=int, default=50_000, help="每條對話的 context 上限（prompt+completion）")
    ap.add_argument("--report-reserve", type=int, default=10_000, help="留給交接（checkpoint tool call + 報告）的 context")
    ap.add_argument("--context-margin", type=int, default=2_500, help="留給 tool 輸出與 template 的餘裕")
    ap.add_argument("--handoff-margin", type=int, default=300)
    ap.add_argument("--min-handoff-tokens", type=int, default=3_500, help="交接報告至少給這麼多 max_tokens（可略超 cap）")
    ap.add_argument("--tokenizer", required=True, help="HF tokenizer 路徑，估第一輪 prompt 長度")
    ap.add_argument("--template-overhead", type=int, default=600, help="chat template + tool schema 的固定 token 開銷估計")
    ap.add_argument("--min-turn-tokens", type=int, default=4_000,
                    help="餘額低於這個就直接交接，不再開一輪只會被截斷的小回合")
    ap.add_argument("--max-turn-tokens", type=int, default=20_000,
                    help="單一回覆的 max_tokens；撞到就要求 checkpoint 後繼續（不是交接）")
    ap.add_argument("--max-continuations", type=int, default=4, help="每條對話最多接續幾次")
    ap.add_argument("--compact", action="store_true",
                    help="context 滿了不交接：把最舊的 tool round 折掉、同一條對話續跑（sandbox 狀態不動），"
                         "直到 --compact-gen-budget 或 --max-turns 才寫報告。配 --max-hops 1 用")
    ap.add_argument("--compact-keep-tokens", type=int, default=8_000, help="折掉之後至少保留最近幾輪的估計 token（下限）")
    ap.add_argument("--delegate", action="store_true",
                    help="explorer 多一個 `delegate` 工具：把子問題交給乾淨 context 的 helper（同 sandbox），只拿回短結論")
    ap.add_argument("--delegate-budget", type=int, default=60_000, help="helper 每次的生成 token 預算")
    ap.add_argument("--delegate-max", type=int, default=6, help="每條對話最多 delegate 幾次")
    ap.add_argument("--delegate-max-turns", type=int, default=80)
    ap.add_argument("--delegate-reply-chars", type=int, default=6000, help="helper 回覆塞回 explorer context 的字元上限")
    ap.add_argument("--notes-structured", action="store_true",
                    help="notes 每條帶 RULE:/VERIFIED:/REFUTED:/TODO:/CODE: 標籤；harness 渲染時分組、去重、RULE 只留最新")
    ap.add_argument("--compact-rich", action="store_true",
                    help="compact 模式：先清舊輪的 tool 輸出（留 reasoning/code）再丟整輪；公告帶 `transform` 原始碼與被折掉輪次的一行摘要")
    ap.add_argument("--compact-keep-full-rounds", type=int, default=2, help="--compact-rich：最近幾輪的 tool 輸出保持完整")
    ap.add_argument("--compact-checkpoint", action="store_true",
                    help="compact 模式：折掉舊輪次之前先給模型一輪把即將消失的推理 append 進 `notes`")
    ap.add_argument("--compact-checkpoint-tokens", type=int, default=3000, help="checkpoint 那一輪的 max_tokens")
    ap.add_argument("--compact-target-frac", type=float, default=0.6,
                    help="折完的 context 目標佔工作視窗（ctx_cap − report_reserve − margin）的比例")
    ap.add_argument("--compact-gen-budget", type=int, default=200_000, help="compact 模式下每條對話的生成 token 總預算")
    ap.add_argument("--max-turns", type=int, default=60)
    ap.add_argument("--handoff-thinking", choices=["off", "low", "same"], default="same")
    ap.add_argument("--handoff-tool-calls", type=int, default=2, help="交接時最多再給幾次 tool call 做 checkpoint")
    ap.add_argument("--handoff-turn-tokens", type=int, default=4_000, help="交接期 checkpoint tool call 那一輪的 max_tokens")
    ap.add_argument("--gen-budget-hard", action="store_true",
                    help="compact 模式：單輪 max_tokens 不得超過剩餘生成預算（至少 min_turn_tokens），超支上限從 20k 變 4k")
    ap.add_argument("--skip-refine-when-agree", action="store_true",
                    help="兩條 explorer 都答完且答案一致就不開 refiner，attempt 2 由候選補位（省一整條對話）")
    ap.add_argument("--handoff-budget", type=int, default=0,
                    help="交接階段（checkpoint tool call + 報告 + 拒絕重試）合計的生成上限；0 = 不設限（原行為）。"
                         "Kaggle 用 10k：模型不寫報告時每輪可生到 ctx 餘額，20k 的預算會花成 58k")
    ap.add_argument("--no-fill-attempt2", dest="fill_attempt2", action="store_false",
                    help="refine 只給 1 個 attempt 時，不用 candidates 補第 2 個")
    ap.add_argument("--inline-cells", type=int, default=400, help="candidate grid 小於這麼多格才直接印進 refine prompt")
    ap.add_argument("--max-live", type=int, default=64, help="同時在飛的對話數（含 sandbox）")
    ap.add_argument("--task-live", type=int, default=0, help="同時在跑的題數上限（0 = 不限；建議 max_live/2）")
    ap.add_argument("--http-timeout", type=float, default=3600.0)
    ap.add_argument("--retries", type=int, default=3)
    ap.add_argument("--sandbox-python", default=sys.executable)
    ap.add_argument("--no-save-messages", dest="save_messages", action="store_false")
    args = ap.parse_args()
    if args.task_source == "nvarc":
        if not args.corpus:
            raise SystemExit("--task-source nvarc 需要 --corpus")
    elif args.task_source == "kaggle":
        if not args.challenges:
            raise SystemExit("--task-source kaggle 需要 --challenges")
    elif not (args.task_dir and args.task_list):
        raise SystemExit("--task-source arc 需要 --task-dir 與 --task-list")
    if args.dp_size < 1:
        raise SystemExit(f"--dp-size 要 >= 1，拿到 {args.dp_size}")
    if args.fixed_dp_rank is not None:
        if args.fixed_dp_rank < 0:
            raise SystemExit(f"--dp-rank 要 >= 0，拿到 {args.fixed_dp_rank}")
        if args.dp_size > 1 and args.fixed_dp_rank >= args.dp_size:
            raise SystemExit(
                f"--dp-rank {args.fixed_dp_rank} 超出 --dp-size {args.dp_size}"
            )
    if (args.lineage_budget or args.refine_gen_budget or args.gen_budget_hard) and not args.compact:
        raise SystemExit("--lineage-budget / --refine-gen-budget / --gen-budget-hard 是 compact 模式的旋鈕，要配 --compact")
    if args.lineage_budget and args.lineage_budget < args.min_session_budget:
        raise SystemExit("--lineage-budget 小於 --min-session-budget，第一個 session 就開不了")
    if args.task_budget < args.base_explorers * args.min_session_budget:
        raise SystemExit("--task-budget 不夠 base explorers 各開一個 --min-session-budget 的 session")
    args.dp_rank = None           # 每題在 main_async 裡指定
    args.should_abort = None      # main_async 接上錯誤率護欄
    args.submission = None        # --submission-out 時是 SubmissionStore
    if args.hparam_menu and not args.compact:
        raise SystemExit("--hparam-menu 抽的軸是 compact 模式的旋鈕，要配 --compact")
    cfg = load_config()
    args.tool = cfg["tool"]
    args.tool_name = cfg["tool_name"]
    args.extra_tools = [prompts.DELEGATE_TOOL] if args.delegate else []
    args.tool_timeout = cfg["rollout"]["tool_timeout_s"]
    args.tool_max_output_chars = cfg["rollout"]["tool_max_output_chars"]
    args.sampling = Sampling()
    args.annotate_ids = None      # tokens backend 會換成 TokenBackend.annotate
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(args.tokenizer, trust_remote_code=True)
    args.count_tokens = lambda text: len(tok(text)["input_ids"])
    if args.backend == "dsv4":
        from dsv4_backend import DSV4_EFFORTS, DSv4Backend
        if args.encoding_path is None:
            raise SystemExit("--backend dsv4 需要 --encoding-path")
        if args.reasoning_effort not in DSV4_EFFORTS:
            raise SystemExit(f"dsv4 的 --reasoning-effort 只有 {DSV4_EFFORTS}")
        if args.handoff_thinking == "off":
            raise SystemExit("dsv4 backend 沒有 enable_thinking=False；--handoff-thinking 用 same 或 low")
        backend = DSv4Backend(args.base_url, args.encoding_path, tok, args.retries)
        args.chat = backend.chat
    elif args.backend == "tokens":
        from token_backend import EFFORTS, TokenBackend
        if args.reasoning_effort not in EFFORTS:
            raise SystemExit(f"tokens backend 的 --reasoning-effort 只有 {EFFORTS}")
        if args.handoff_thinking != "same":
            raise SystemExit("tokens backend 的 --handoff-thinking 只支援 same（prompt 前綴要固定）")
        backend = TokenBackend(args.base_url, tok, [args.tool, *args.extra_tools], args.retries)
        args.chat = backend.chat
        args.annotate_ids = backend.annotate
    else:
        async def _openai_chat(client, payload):
            return await _chat(client, args.base_url, payload, args.retries)
        args.chat = _openai_chat
    if args.ctx_cap - args.report_reserve - args.context_margin < 10_000:
        raise SystemExit("ctx_cap 扣掉 reserve/margin 後不到 10k，沒得解")
    # deadline 從這裡起算（tokenizer / backend 都載好了），呼叫端給的分鐘數才全是跑題的時間。
    args.deadline = time.monotonic() + args.deadline_min * 60 if args.deadline_min > 0 else None
    return args


if __name__ == "__main__":
    asyncio.run(main_async(parse_args()))



In [ ]:
"""Resolve and hash every offline asset before touching a GPU runtime."""

import hashlib
import torch


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as fh:
        for chunk in iter(lambda: fh.read(16 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


input_root = Path("/kaggle/input")
names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
caps = [torch.cuda.get_device_capability(i) for i in range(torch.cuda.device_count())]
print("GPU", names, caps)
assert len(names) == 4 and all("L4" in name for name in names), names
assert all(tuple(cap) == (8, 9) for cap in caps), caps
topology = subprocess.run(
    ["nvidia-smi", "topo", "-m"], capture_output=True, text=True, check=True
).stdout
print(topology)
p2p = [
    [i == j or torch.cuda.can_device_access_peer(i, j) for j in range(len(names))]
    for i in range(len(names))
]
print("CUDA_P2P", p2p)
assert all(all(row) for row in p2p), p2p
assert os.environ.get("NCCL_P2P_DISABLE") not in {"1", "true", "True"}

target_config_sha = MANIFEST["target_sha256"]["config.json"]
target_candidates = [
    p.parent for p in input_root.rglob("config.json") if sha256_file(p) == target_config_sha
]
archives = list(input_root.rglob(RUNTIME_ARCHIVE_NAME))
zstds = [p for p in input_root.rglob("zstd") if p.is_file()]
assert len(target_candidates) == len(archives) == len(zstds) == 1, (
    target_candidates, archives, zstds
)
TARGET, ARCHIVE, ZSTD_SOURCE = target_candidates[0], archives[0], zstds[0]

for rel, want_size in MANIFEST["target_sizes"].items():
    path = TARGET / rel
    assert path.is_file(), f"missing target file: {rel}"
    assert path.stat().st_size == want_size, (
        rel, path.stat().st_size, want_size
    )
for rel, want in MANIFEST["target_sha256"].items():
    path = TARGET / rel
    got = sha256_file(path)
    assert got == want, f"target/{rel}: got {got}, want {want}"
assert sha256_file(ARCHIVE) == MANIFEST["runtime_sha256"], "runtime archive SHA mismatch"
for name, payload in PATCH_PAYLOADS.items():
    got = hashlib.sha256(payload.encode()).hexdigest()
    assert got == PATCH_SHA256[name], (name, got, PATCH_SHA256[name])
print(
    f"assets verified: target {len(MANIFEST['target_sizes'])} files "
    f"({len(MANIFEST['target_sha256'])} hashed), runtime, {len(PATCH_ORDER)} patches"
)

challenge_files = (
    sorted(input_root.rglob("*test_challenges*.json"))
    or sorted(input_root.rglob("*challenges*.json"))
)
assert challenge_files, "competition challenges JSON not found"
CHALLENGES = challenge_files[0]
tasks = json.loads(CHALLENGES.read_text())
print("tasks", len(tasks), "test inputs", sum(len(t["test"]) for t in tasks.values()))

PUBLIC_PLACEHOLDER_IDS = {
    "00dbd492", "00d62c1b", "009d5c81", "00576224",
    "03560426", "017c7c7b", "025d127b", "007bbfb7",
}
rerun_flag = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "")
looks_public = PUBLIC_PLACEHOLDER_IDS <= set(tasks)
if CFG.limit_mode == "full":
    LIMIT, reason = 0, "forced full"
elif CFG.limit_mode == "manual":
    LIMIT, reason = CFG.limit_manual, "forced manual"
elif rerun_flag or not looks_public:
    LIMIT, reason = 0, "competition rerun/hidden tasks"
else:
    LIMIT, reason = CFG.limit_manual, "manual public validation"
print("task limit", LIMIT or "all", reason)

In [ ]:
"""Extract the frozen runtime and install the complete canonical patch chain."""

scratch = CFG.scratch_path
package_root = scratch / "package"
if not (package_root / "python/sglang").is_dir():
    zstd_bin = scratch / "zstd"
    shutil.copy2(ZSTD_SOURCE, zstd_bin)
    zstd_bin.chmod(0o755)
    extract_env = {
        **os.environ,
        "PATH": f"{scratch}:{os.environ.get('PATH', '')}",
        "LD_LIBRARY_PATH": f"{ZSTD_SOURCE.parent}:{os.environ.get('LD_LIBRARY_PATH', '')}",
    }
    t0 = time.perf_counter()
    subprocess.run(
        ["tar", "--zstd", "-xf", str(ARCHIVE), "-C", str(scratch)],
        check=True,
        env=extract_env,
    )
    print(f"runtime extracted in {time.perf_counter() - t0:.1f}s")

SGL_SRC = package_root / "python"
SITE = package_root / "lib/python3.12/site-packages"
if not (SITE / "torch/__init__.py").is_file() and (
    SITE / "site-packages/torch/__init__.py"
).is_file():
    SITE = SITE / "site-packages"
assert (SGL_SRC / "sglang").is_dir() and (SITE / "torch/__init__.py").is_file(), (
    SGL_SRC, SITE
)

patch_root = scratch / "patches"
patch_root.mkdir(exist_ok=True)
for name, payload in PATCH_PAYLOADS.items():
    path = patch_root / name
    path.write_text(payload)
    assert sha256_file(path) == PATCH_SHA256[name]


def apply_state(path):
    def run(*extra):
        return subprocess.run(
            ["git", "apply", "--whitespace=nowarn", *extra, str(path)],
            cwd=package_root,
            capture_output=True,
            text=True,
        )
    if run("--check").returncode == 0:
        return "pending"
    if run("--reverse", "--check").returncode == 0:
        return "applied"
    probe = run("--check")
    raise RuntimeError(f"{path.name} source drift:\n{probe.stdout}\n{probe.stderr}")


def apply_qwen4_mtp_topology():
    """Backport the Qwen4 draft fix using a stable semantic anchor.

    The frozen runtime is older than the upstream hunk used to author patch
    0044, so the neighboring Qwen3Next lines are not present.  The Qwen3-MoE
    draft branch immediately following the insertion point is stable in both
    revisions; use it as the anchor and make the transform idempotent.
    """
    source = package_root / "python/sglang/srt/configs/model_config.py"
    text = source.read_text()
    marker = 'self.hf_config.architectures[0] = "Qwen4ExpForCausalLMMTP"'
    if marker not in text:
        anchor = (
            '        if is_draft_model and self.hf_config.architectures[0] '
            '== "Qwen3MoeForCausalLM":\n'
        )
        assert text.count(anchor) == 1, "Qwen3-MoE draft anchor drifted"
        block = """        if (
            is_draft_model
            and self.hf_config.architectures[0]
            == "Qwen4ExpForConditionalGeneration"
        ):
            # The target and draft initially share the HF config. Isolate the
            # draft before reducing its topology to the one-layer MTP head.
            from copy import deepcopy

            self.hf_config = deepcopy(self.hf_config)
            self.hf_text_config = get_hf_text_config(self.hf_config)
            self.hf_config.architectures[0] = "Qwen4ExpForCausalLMMTP"
            text_config = self.hf_text_config
            text_config.num_nextn_predict_layers = 1
            text_config.num_hidden_layers = 1
            text_config.layer_types = ["full_attention"]
            text_config.full_attention_interval = 1

"""
        source.write_text(text.replace(anchor, block + anchor))
    patched = source.read_text()
    assert marker in patched
    assert 'text_config.num_hidden_layers = 1' in patched
    compile(patched, str(source), "exec")


for name in PATCH_ORDER:
    path = patch_root / name
    if name == "0044-qwen4-mtp-draft-topology.patch":
        apply_qwen4_mtp_topology()
        print("patch", name, "OK (semantic anchor)")
        continue
    state = apply_state(path)
    if state == "pending":
        result = subprocess.run(
            ["git", "apply", "--whitespace=nowarn", str(path)],
            cwd=package_root,
            capture_output=True,
            text=True,
        )
        assert result.returncode == 0, result.stdout + result.stderr
    assert apply_state(path) == "applied", name
    print("patch", name, "OK")

PYTHON = "/usr/bin/python3.12" if Path("/usr/bin/python3.12").is_file() else sys.executable
cuda_home = package_root / "cuda-13.1"
shim = scratch / "libcuda-link"
shim.mkdir(exist_ok=True)
drivers = [
    p for p in (
        Path("/usr/lib/x86_64-linux-gnu/libcuda.so.1"),
        Path("/usr/local/cuda/compat/libcuda.so.1"),
        Path("/usr/local/nvidia/lib64/libcuda.so.1"),
    ) if p.is_file()
]
assert drivers, "libcuda.so.1 not found"
if not (shim / "libcuda.so").exists():
    (shim / "libcuda.so").symlink_to(drivers[0])

ENV = {
    **os.environ,
    "CUDA_HOME": str(cuda_home),
    "CUDA_VISIBLE_DEVICES": "0,1,2,3",
    "PATH": f"{cuda_home}/bin:{package_root}/bin:{os.environ.get('PATH', '')}",
    "PYTHONPATH": f"{SGL_SRC}:{SITE}:{SITE}/nvidia_cutlass_dsl/dsl_packages",
    "LIBRARY_PATH": f"{shim}:{os.environ.get('LIBRARY_PATH', '')}",
    "FLASHINFER_CUDA_ARCH_LIST": "8.9",
    "TORCH_CUDA_ARCH_LIST": "8.9",
    "MAX_JOBS": "4",
    "FLASHINFER_NVCC_THREADS": "1",
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
    "DO_NOT_TRACK": "1",
    "MPLBACKEND": "Agg",
    "SGLANG_RUST_SERVER": "0",
    "SGLANG_RUST_BUILD_MODE": "never",
    "SGLANG_QWEN4_MNNVL_HC": "0",
    "PYTORCH_ALLOC_CONF": "expandable_segments:True",
    "SGLANG_QSA_DIRECT_DECODE": "1",
    "SGLANG_QSA_DIRECT_MIN_ROWS": "1",
    "ARC_QSA_FP4_FUSED": "1",
}
probe = subprocess.run(
    [
        PYTHON, "-c",
        "import torch,sglang,transformers;"
        "print(torch.__version__,sglang.__version__,torch.cuda.get_device_capability());"
        "print(torch.__file__);print(sglang.__file__);"
        "assert torch.cuda.is_available()",
    ],
    env=ENV,
    capture_output=True,
    text=True,
)
assert probe.returncode == 0, probe.stdout + probe.stderr
print(probe.stdout)

CLIENT_ENV = {**os.environ, "PYTHONPATH": str(CFG.scratch_path / "src")}
if subprocess.run(
    [sys.executable, "-c", "import httpx,numpy,transformers"],
    env=CLIENT_ENV,
    capture_output=True,
).returncode:
    CLIENT_ENV["PYTHONPATH"] = f"{CFG.scratch_path / 'src'}:{SITE}"

In [ ]:
"""Execute quantize, packed decode, and packed prefill on both DP GPU groups."""

probe_code = r"""
import json
import torch
from sglang.srt.layers.quantization.kvfp4_tensor import nvfp4_quantize_triton
from sglang.srt.layers.attention.qsa.sparse_attn import (
    qwen_sparse_gqa_decode_selected_fp4_triton,
    qwen_sparse_nvfp4_expand_triton,
)

torch.manual_seed(19)
levels = torch.tensor([0., .5, 1., 1.5, 2., 3., 4., 6.], device="cuda")


def dequant(packed, sf, global_scale):
    code = torch.stack((packed & 15, packed >> 4), dim=-1)
    code = code.reshape(*packed.shape[:-1], packed.shape[-1] * 2).long()
    sign = torch.where((code & 8) != 0, -1., 1.)
    mag = levels[code & 7] * sign
    return mag * sf.float().repeat_interleave(16, dim=-1) * global_scale


def reference(q, kd, vd, table, req, idx, scale):
    out = []
    for row in range(q.shape[0]):
        logical = idx[row][idx[row] >= 0].long()
        slots = table[req[row].long(), logical].long()
        scores = torch.einsum("hd,kd->hk", q[row].float(), kd[slots, 0].float()) * scale
        out.append(torch.einsum("hk,kd->hd", scores.softmax(-1), vd[slots, 0].float()))
    return torch.stack(out)


for rows in (1, 8):
    tokens, dim = rows * 73, 256
    q = torch.randn(rows, 12, dim, device="cuda", dtype=torch.bfloat16)
    k = torch.randn(tokens, 1, dim, device="cuda", dtype=torch.bfloat16)
    v = torch.randn_like(k)
    kg = torch.tensor([float(k.float().abs().amax()) / (6 * 448)], device="cuda")
    vg = torch.tensor([float(v.float().abs().amax()) / (6 * 448)], device="cuda")
    kp2, ksf2 = nvfp4_quantize_triton(k.reshape(-1, dim), kg)
    vp2, vsf2 = nvfp4_quantize_triton(v.reshape(-1, dim), vg)
    kp, vp = kp2.view(tokens, 1, dim // 2), vp2.view(tokens, 1, dim // 2)
    ksf, vsf = ksf2.view(tokens, 1, dim // 16), vsf2.view(tokens, 1, dim // 16)
    kd, vd = dequant(kp, ksf, float(kg)), dequant(vp, vsf, float(vg))

    table = torch.randperm(tokens, device="cuda", dtype=torch.int64).to(torch.int32)
    table = table.reshape(rows, 73)
    req = torch.arange(rows - 1, -1, -1, device="cuda", dtype=torch.int32)
    idx = torch.full((rows, 128), -1, device="cuda", dtype=torch.int32)
    lens = torch.full((rows,), 73, device="cuda", dtype=torch.int32)
    for row in range(rows):
        idx[row, :37 + row % 5] = torch.arange(37 + row % 5, device="cuda")

    scale = dim ** -0.5
    got = qwen_sparse_gqa_decode_selected_fp4_triton(
        q, kp, vp, ksf, vsf, table, req, idx, lens, scale, float(kg), float(vg)
    ).float()
    want = reference(q, kd, vd, table, req, idx, scale)
    torch.testing.assert_close(got, want, rtol=.025, atol=.025)

    lengths = torch.tensor([37 + row % 5 for row in range(rows)], device="cuda")
    cu = torch.cat((torch.zeros(1, device="cuda", dtype=torch.int64), lengths.cumsum(0)))
    ek, ev = qwen_sparse_nvfp4_expand_triton(
        kp, vp, ksf, vsf, table, req, cu, int(cu[-1]), float(kg), float(vg)
    )
    want_k, want_v = [], []
    for row in range(rows):
        active = int(lengths[row].item())
        slots = table[req[row].long(), :active].long()
        want_k.append(kd[slots])
        want_v.append(vd[slots])
    torch.testing.assert_close(ek.float(), torch.cat(want_k).float(), rtol=.005, atol=.005)
    torch.testing.assert_close(ev.float(), torch.cat(want_v).float(), rtol=.005, atol=.005)
    print(json.dumps({
        "rows": rows,
        "tokens": tokens,
        "decode_max_abs": float((got - want).abs().max()),
        "k_global": float(kg),
        "v_global": float(vg),
    }))
print("NVFP4_QUANTIZE_DECODE_PREFILL_PASS")
"""

for physical_gpu in ("0", "2"):
    gate_env = {**ENV, "CUDA_VISIBLE_DEVICES": physical_gpu}
    result = subprocess.run(
        [PYTHON, "-c", probe_code],
        env=gate_env,
        capture_output=True,
        text=True,
        timeout=600,
    )
    print("GPU", physical_gpu, result.stdout, result.stderr)
    assert result.returncode == 0, f"NVFP4 gate failed on physical GPU {physical_gpu}"
    assert "NVFP4_QUANTIZE_DECODE_PREFILL_PASS" in result.stdout
print("NVFP4_KERNEL_GATES_OK")

In [ ]:
"""Launch the radix-aware EP4/attention-DP2 MTP profile."""

import re
import urllib.request

LOG = CFG.scratch_path / "sglang_server.log"
args = [
    PYTHON, "-m", "sglang.launch_server",
    "--model-path", str(TARGET),
    "--tokenizer-path", str(TARGET),
    "--served-model-name", "tir",
    "--host", "127.0.0.1",
    "--port", str(CFG.port),
    "--tp", str(CFG.tp_size),
    "--dp-size", str(CFG.dp_size),
    "--enable-dp-attention",
    "--ep-size", str(CFG.ep_size),
    "--quantization", "modelopt_mixed",
    "--fp4-gemm-backend", "marlin",
    "--moe-runner-backend", "marlin",
    "--attention-backend", CFG.attention_backend,
    "--linear-attn-prefill-backend", "triton",
    "--linear-attn-decode-backend", "triton",
    "--ple-offload-embedding",
    "--language-only",
    "--json-model-override-args",
    json.dumps({"text_config": {
        "ple_embedding_dtype": "float8_e4m3fn",
        "num_nextn_predict_layers": 1,
    }}),
    "--dtype", "bfloat16",
    "--kv-cache-dtype", CFG.kv_cache_dtype,
    "--page-size", str(CFG.page_size),
    "--context-length", str(CFG.context_length),
    "--chunked-prefill-size", str(CFG.chunked_prefill),
    "--max-running-requests", str(CFG.max_running_global),
    "--max-mamba-cache-size", str(CFG.max_mamba_cache_global),
    "--mamba-ssm-dtype", CFG.mamba_ssm_dtype,
    "--mamba-radix-cache-strategy", "extra_buffer_lazy",
    "--mamba-max-states-per-path", "1",
    "--enable-cache-report",
    "--mem-fraction-static", str(CFG.mem_fraction),
    "--cuda-graph-max-bs-decode", str(CFG.cuda_graph_max_bs),
    "--disable-prefill-cuda-graph",
    "--reasoning-parser", "qwen3",
    "--tool-call-parser", "qwen3_coder",
    "--model-loader-extra-config",
    json.dumps({"enable_multithread_load": True, "num_threads": 2}),
]
if CFG.max_total_tokens is not None:
    args += ["--max-total-tokens", str(CFG.max_total_tokens)]
if not CFG.enable_sglang_custom_all_reduce:
    args.append("--disable-custom-all-reduce")
if CFG.enable_mtp:
    args += [
        "--speculative-algorithm", "NEXTN",
        "--speculative-num-steps", "3",
        "--speculative-eagle-topk", "1",
        "--speculative-num-draft-tokens", "4",
    ]
if CFG.enable_replayssm_spec:
    args.append("--enable-linear-replayssm-spec")
assert "--disable-radix-cache" not in args
assert "--enable-linear-replayssm" not in args
print(" ".join(args), flush=True)
server = subprocess.Popen(args, env=ENV, stdout=LOG.open("w"), stderr=subprocess.STDOUT)

info, started = None, time.time()
while time.time() - started < CFG.server_timeout_s:
    if server.poll() is not None:
        print(LOG.read_text(errors="replace")[-12000:])
        raise RuntimeError(f"server died during startup, rc={server.returncode}")
    try:
        with urllib.request.urlopen(
            f"http://127.0.0.1:{CFG.port}/get_server_info", timeout=5
        ) as fh:
            info = json.load(fh)
        break
    except Exception:
        time.sleep(10)
assert info is not None, "server startup timeout"
print(f"server ready in {time.time() - started:.0f}s")

assert info["tp_size"] == 4 and info["dp_size"] == 2 and info["ep_size"] == 4, info
assert info["enable_dp_attention"] is True
assert info["kv_cache_dtype"] == "nvfp4"
assert info.get("enable_linear_replayssm") is False
if CFG.enable_mtp:
    assert info["speculative_algorithm"] == "EAGLE"
    assert info["speculative_num_steps"] == 3
    assert info["speculative_eagle_topk"] == 1
    assert info["speculative_num_draft_tokens"] == 4
    assert info["enable_linear_replayssm_spec"] is CFG.enable_replayssm_spec
else:
    assert info.get("speculative_algorithm") in (None, "NONE"), info.get(
        "speculative_algorithm"
    )
    assert not info.get("enable_linear_replayssm_spec", False)
assert info["disable_custom_all_reduce"] is (
    not CFG.enable_sglang_custom_all_reduce
), info["disable_custom_all_reduce"]
states = info.get("internal_states") or []
assert len(states) == 2, states

workers = []
for index, state in enumerate(states):
    memory = state["memory_usage"]
    worker = {
        "index": index,
        "pool": int(memory["token_capacity"]),
        "running": int(state["effective_max_running_requests_per_dp"]),
        "mamba": int(state["max_mamba_cache_size"]),
        "memory": memory,
    }
    workers.append(worker)
    assert worker["pool"] >= CFG.min_pool_tokens_per_worker, worker
    assert worker["running"] >= CFG.live_per_worker, worker
    states_per_request = 5 if CFG.enable_mtp else 4
    assert worker["mamba"] >= states_per_request * CFG.live_per_worker, worker
    assert CFG.live_per_worker * CFG.pool_context <= worker["pool"] * CFG.pool_safety, worker
print("workers", json.dumps(workers, indent=2))

assert info["disable_radix_cache"] is False
assert info["mamba_radix_cache_strategy"] == "extra_buffer_lazy"
assert info["uses_mamba_radix_cache"] is True
log = LOG.read_text(errors="replace")
bad = [
    line for line in log.splitlines()
    if "nvfp4" in line.lower()
    and any(word in line.lower() for word in ("fallback", "not supported", "unavailable"))
]
assert not bad, "NVFP4 fallback detected:\n" + "\n".join(bad[-20:])
(Path("/kaggle/working") / "deployment_start.json").write_text(
    json.dumps({"server_info": info, "workers": workers, "args": args}, indent=2)
)

In [ ]:
"""Check TIR tools, effort, sticky DP routing, and real multi-turn radix hits."""

def chat(messages, max_tokens=512, tools=None, temperature=0.0, dp_rank=None, effort=None):
    body = {
        "model": "tir",
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "top_k": -1,
    }
    if tools:
        body["tools"] = tools
    if dp_rank is not None:
        body["routed_dp_rank"] = dp_rank
    if effort:
        body["reasoning_effort"] = effort
    req = urllib.request.Request(
        f"{CFG.base_url}/chat/completions",
        data=json.dumps(body).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=900) as fh:
        return json.load(fh)


sys.path.insert(0, str(CFG.scratch_path / "src"))
from arcforge.tir import protocol

tir_cfg = protocol.load_config(CFG.scratch_path / "src/arcforge/tir/tir_config.json")
task = next(iter(tasks.values()))
messages = [
    {"role": "system", "content": protocol.system_prompt(tir_cfg, CFG.max_turns)},
    {"role": "user", "content": protocol.render_user_prompt(
        task["train"], task["test"][0]["input"]
    )},
]
calls = []
for attempt in range(2):
    choice = chat(
        messages,
        max_tokens=32_768,
        tools=[tir_cfg["tool"]],
        temperature=CFG.temperature,
        effort=CFG.effort,
        dp_rank=0,
    )["choices"][0]
    calls = choice["message"].get("tool_calls") or []
    print("tool smoke", attempt + 1, choice.get("finish_reason"), len(calls))
    if calls:
        break
assert calls, "model finished twice without a valid TIR tool call"

p_hi = chat(
    [{"role": "user", "content": "hi"}], 8, effort="xhigh", dp_rank=0
)["usage"]["prompt_tokens"]
p_lo = chat(
    [{"role": "user", "content": "hi"}], 8, effort="low", dp_rank=0
)["usage"]["prompt_tokens"]
assert p_hi != p_lo, (p_hi, p_lo)


def cached(response):
    usage = response["usage"]
    details = usage.get("prompt_tokens_details") or {}
    return int(details.get("cached_tokens") or 0), int(usage["prompt_tokens"])


cache_rows = []
for rank in range(CFG.dp_size):
    tag = os.urandom(8).hex()
    filler = "\n".join(f"{tag} row {i}: {i * 7 % 97}" for i in range(500))
    turn1 = [{"role": "user", "content": f"Table:\n{filler}\nReply OK."}]
    first = chat(turn1, 8, dp_rank=rank)
    turn2 = turn1 + [
        {"role": "assistant", "content": first["choices"][0]["message"]["content"] or "OK"},
        {"role": "user", "content": "Keep the table in context and reply DONE."},
    ]
    second = chat(turn2, 8, dp_rank=rank)
    turn3 = turn2 + [
        {"role": "assistant", "content": second["choices"][0]["message"]["content"] or "DONE"},
        {"role": "user", "content": "Reply FINAL."},
    ]
    third = chat(turn3, 8, dp_rank=rank)
    c2, p2 = cached(second)
    c3, p3 = cached(third)
    row = {"rank": rank, "turn2": [c2, p2], "turn3": [c3, p3]}
    cache_rows.append(row)
    assert c2 > 0.5 * p2 and c3 > 0.5 * p3, row
print("RADIX_MULTI_TURN_PASS", cache_rows)
(Path("/kaggle/working") / "smoke.json").write_text(
    json.dumps({"tool_calls": len(calls), "effort_tokens": [p_hi, p_lo],
                "cache": cache_rows}, indent=2)
)

In [ ]:
"""Persist the independent MTP + ReplaySSM capacity result."""

assert CFG.enable_mtp is True
assert CFG.enable_replayssm_spec is True
result = {
    "profile": "mtp_replayssm_probe",
    "workers": workers,
    "aggregate_token_capacity": sum(worker["pool"] for worker in workers),
    "server_info": {
        "tp_size": info["tp_size"],
        "dp_size": info["dp_size"],
        "ep_size": info["ep_size"],
        "kv_cache_dtype": info["kv_cache_dtype"],
        "speculative_algorithm": info["speculative_algorithm"],
        "speculative_num_steps": info["speculative_num_steps"],
        "speculative_num_draft_tokens": info["speculative_num_draft_tokens"],
        "enable_linear_replayssm_spec": info["enable_linear_replayssm_spec"],
        "disable_radix_cache": info["disable_radix_cache"],
        "mamba_radix_cache_strategy": info["mamba_radix_cache_strategy"],
    },
    "radix_smoke": cache_rows,
}
result_path = Path("/kaggle/working/mtp_replayssm_capacity.json")
result_path.write_text(json.dumps(result, indent=2))
shutil.copy2(LOG, "/kaggle/working/sglang_server.log")
print("MTP_REPLAYSSM_CAPACITY", json.dumps({
    "per_dp": [worker["pool"] for worker in workers],
    "aggregate": result["aggregate_token_capacity"],
    "running_per_dp": [worker["running"] for worker in workers],
    "mamba_per_dp": [worker["mamba"] for worker in workers],
}, indent=2))
print("MTP_REPLAYSSM_PROBE_OK", result_path)

server.terminate()
try:
    server.wait(timeout=120)
except subprocess.TimeoutExpired:
    server.kill()